
Check GPU


In [1]:
import torch, platform
print("torch:", torch.__version__,
      "cuda_runtime:", torch.version.cuda,
      "is_cuda_available:", torch.cuda.is_available(),
      "device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("CUDA not available in this kernel — recheck steps 1–4.")


torch: 2.4.1+cu121 cuda_runtime: 12.1 is_cuda_available: True device_count: 1
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


PyG Install with CUDA

In [2]:
import sys, subprocess, torch, platform, re

# If Colab's torch is already installed, reuse it; else install a pinned one:
print("torch:", torch.__version__, "cuda:", torch.version.cuda, "is_cuda_available:", torch.cuda.is_available())

# Install matching PyG wheels for the *current* runtime torch version & CUDA tag:
torch_ver = torch.__version__.split('+')[0]             # e.g., '2.4.1'
cuda_tag  = ('cu' + (torch.version.cuda or '').replace('.', '')) or 'cpu'
if cuda_tag.startswith('cu121'):  # normalize common case
    cuda_tag = 'cu121'
elif cuda_tag == 'cpu':
    cuda_tag = 'cpu'

url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print("PyG wheel index:", url)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pyg-lib", "torch-scatter", "torch-sparse",
                       "torch-cluster", "torch-spline-conv", "-f", url])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "torch-geometric==2.6.1", "transformers", "tqdm"])
print("Install complete.")


torch: 2.4.1+cu121 cuda: 12.1 is_cuda_available: True
PyG wheel index: https://data.pyg.org/whl/torch-2.4.1+cu121.html
Install complete.


In [3]:
import sys, subprocess, pkgutil

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

# 1) Remove CPU wheels if present
pip("uninstall", "-y", "torch", "torchvision", "torchaudio")

# 2) Install CUDA-enabled PyTorch (cu121 is the most common for RTX on recent drivers)
pip("install", "--index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1")

# 3) Install PyG that matches the *exact* torch & CUDA tag
torch_ver = "2.4.1"
cuda_tag  = "cu121"
pyg_idx = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
pip("install", "-f", pyg_idx,
    "pyg-lib", "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv")
pip("install", "torch-geometric==2.6.1", "transformers>=4.44.0", "tqdm")

# 4) Verify CUDA
import torch
print("torch:", torch.__version__,
      "cuda_runtime:", torch.version.cuda,
      "is_cuda_available:", torch.cuda.is_available(),
      "device_count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")


torch: 2.4.1+cu121 cuda_runtime: 12.1 is_cuda_available: True device_count: 1
GPU name: NVIDIA GeForce RTX 4070 Laptop GPU


In [7]:
import sys, subprocess, torch

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

# Must already be on a CUDA wheel
print("Before:", torch.__version__, torch.version.cuda, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("You're not on a CUDA build. Select the CUDA kernel first.")

# Build the EXACT PyG index for your current Torch+CUDA
torch_ver = torch.__version__.split('+')[0]           # e.g., '2.4.1'
cuda_tag  = 'cu' + (torch.version.cuda or '').replace('.', '')  # e.g., 'cu121'
pyg_idx   = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print("PyG index:", pyg_idx)

# 1) Install the compiled PyG ops matching your Torch/CUDA (won't touch torch)
pip("-q", "install", "-f", pyg_idx,
    "pyg-lib", "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv")

# 2) Install torch-geometric WITHOUT letting pip change torch
pip("-q", "install", "--no-deps", "torch-geometric==2.6.1")

# 3) Other libs (these shouldn't force a torch reinstall, but we'll verify after)
pip("-q", "install", "--upgrade", "transformers", "tqdm")

# Sanity check: torch must remain the same CUDA build
import importlib, torch as _t
print("After:", _t.__version__, _t.version.cuda, _t.cuda.is_available())


Before: 2.4.1+cu121 12.1 True
PyG index: https://data.pyg.org/whl/torch-2.4.1+cu121.html
After: 2.4.1+cu121 12.1 True



Check Dir 

In [12]:
from pathlib import Path

project_root = Path.cwd()  # or set explicitly to your repo root
dataset_root = project_root / "Dataset"

print("Using:", dataset_root.resolve())

for split in ("train","valid","test"):
    for sub in ("hetero_ready","unified_aug"):
        p = dataset_root / split / sub
        print(f"{split}/{sub} ->", "OK" if p.exists() else "MISSING", "-", p)

# Count .pt files the trainer will load
for split in ("train","valid","test"):
    p = dataset_root / split / "hetero_ready"
    n = len(list(p.rglob("*.pt"))) if p.exists() else 0
    print(f"{split}/hetero_ready: {n} tensors")


Using: C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset
train/hetero_ready -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\train\hetero_ready
train/unified_aug -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\train\unified_aug
valid/hetero_ready -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\valid\hetero_ready
valid/unified_aug -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\valid\unified_aug
test/hetero_ready -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\test\hetero_ready
test/unified_aug -> OK - c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\Dataset\test\unified_aug
train/hetero_ready: 3438 tensors
valid/hetero_ready: 2905 tensors
test/hetero_ready: 2915 tensors


Create hetero_ready_gcbert folder

In [13]:
from pathlib import Path
for split in ("train","valid","test"):
    Path(f"Dataset/{split}/hetero_ready_gcbert").mkdir(parents=True, exist_ok=True)
print("OK")


OK


Check GCBert

In [19]:
# 1) Make sure the bits are up to date (these won't flip Torch to CPU)
%pip install -q -U safetensors "transformers>=4.44.0" "huggingface_hub>=0.25.0"

# 2) Nuke any stale/corrupted cached .bin for this model, so we fetch the .safetensors:
import shutil, os, pathlib
cache_dir = pathlib.Path.home() / ".cache" / "huggingface" / "hub" / "models--microsoft--graphcodebert-base"
shutil.rmtree(cache_dir, ignore_errors=True)

# 3) Load GraphCodeBERT explicitly with SafeTensors
from transformers import AutoTokenizer, AutoModel
name = "microsoft/graphcodebert-base"
tok  = AutoTokenizer.from_pretrained(name, use_fast=True)
mdl  = AutoModel.from_pretrained(name, use_safetensors=True, torch_dtype=None)  # stays fp32
print("OK:", mdl.config.hidden_size)



Note: you may need to restart the kernel to use updated packages.


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OK: 768


GCBERT Ready

In [20]:
from transformers import AutoTokenizer, AutoModel
name = "microsoft/graphcodebert-base"
tok = AutoTokenizer.from_pretrained(name, use_fast=True)
mdl = AutoModel.from_pretrained(name, use_safetensors=True)  # keep this to be explicit
print("GCBERT loaded:", mdl.config.hidden_size)


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GCBERT loaded: 768


Count PyTensors (*.pt) in each split 

In [22]:
from pathlib import Path

def preflight(split):
    in_dir   = Path(f"Dataset/{split}/hetero_ready")
    json_dir = Path(f"Dataset/{split}/unified_aug")

    pts   = sorted(p.stem for p in in_dir.glob("*.pt"))
    jsons = {p.stem for p in json_dir.glob("*.json")} | \
            {p.stem for p in json_dir.glob("*.aug.json")} | \
            {p.stem for p in json_dir.glob("*.unified.json")} | \
            {p.stem for p in json_dir.glob("*.jsonl")}
    missing_json = [b for b in pts if b not in jsons]
    print(f"[{split}] .pt: {len(pts)} | json candidates: {len(jsons)} | missing-json-for-pt: {len(missing_json)}")
    for b in missing_json[:5]:
        print("  -", b)

for s in ("train","valid","test"):
    preflight(s)


[train] .pt: 3438 | json candidates: 3438 | missing-json-for-pt: 0
[valid] .pt: 2905 | json candidates: 2905 | missing-json-for-pt: 0
[test] .pt: 2915 | json candidates: 2915 | missing-json-for-pt: 0


In [23]:
import sys, subprocess

def run_once(split):
    in_dir   = f"Dataset/{split}/hetero_ready"
    json_dir = f"Dataset/{split}/unified_aug"
    out_dir  = f"Dataset/{split}/hetero_ready_gcbert"
    proc = subprocess.run([
        sys.executable, "src/tools/upgrade_train_gcbert.py",
        "--in_dir", in_dir, "--json_dir", json_dir, "--out_dir", out_dir,
        "--device", "cuda", "--node_store", "node", "--max_len", "256", "--batch_size", "64",
        "--verbose"
    ], capture_output=True, text=True)
    print("=== STDOUT ===\n", proc.stdout[:4000])
    print("=== STDERR ===\n", proc.stderr[:4000])
    print("Return code:", proc.returncode)

run_once("train")


=== STDOUT ===
 
=== STDERR ===
 usage: upgrade_train_gcbert.py [-h] --in_dir IN_DIR --json_dir JSON_DIR
                               --out_dir OUT_DIR [--device {cuda,cpu}]
                               [--node_store NODE_STORE] [--max_len MAX_LEN]
                               [--batch_size BATCH_SIZE]
upgrade_train_gcbert.py: error: unrecognized arguments: --verbose

Return code: 2


# === Upgrade hetero_ready → hetero_ready_gcbert for train/valid/test ===

In [6]:
from pathlib import Path
import os, json, math, warnings, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
from torch_geometric.data import HeteroData

# --- Quiet the chatty stuff (optional) ---
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
hf_logging.set_verbosity_error()  # hides "Some weights ... newly initialized" messages
warnings.filterwarnings(
    "ignore",
    message="You are using `torch.load` with `weights_only=False`",
    category=FutureWarning,
)

# --- helpers ---
def _as_hetero(obj):
    if isinstance(obj, HeteroData): return obj
    raise TypeError(f"Expected HeteroData; got {type(obj)}")

def _load_aug_json(path):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    nodes = j.get("nodes", [])
    id2idx = {n.get("_id"): i for i, n in enumerate(nodes) if "_id" in n}
    return nodes, id2idx

def _node_text(n: dict) -> str:
    for k in ("code", "name", "methodFullName", "signature"):
        v = n.get(k)
        if isinstance(v, str) and v.strip(): return v.strip()
    return ""

def _pick_node_store(data: HeteroData, preferred: str | None) -> str:
    node_types = tuple(getattr(data, "node_types", ()))
    if preferred and preferred in node_types: return preferred
    for cand in ("node","ast","code","token","statement"):
        if cand in node_types: return cand
    if node_types: return node_types[0]
    for key in data.keys(): return key
    raise RuntimeError("Could not determine a node store/type to embed.")

@torch.no_grad()
def _embed_gcbert(texts, model, tok, device="cuda", max_len=256, batch_size=64,
                  desc="Embedding", show_inner=True):
    """Return [N,768] CLS embeddings. Inner tqdm collapses after each shard."""
    embs = []
    model.to(device).eval()
    total = (len(texts) + batch_size - 1) // batch_size
    inner = tqdm(range(0, len(texts), batch_size),
                 total=total, desc=desc, unit="batch",
                 leave=False, disable=not show_inner, mininterval=0.2)
    for i in inner:
        chunk = texts[i:i+batch_size]
        enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        if device == "cuda":
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(**enc).last_hidden_state
        else:
            out = model(**enc).last_hidden_state
        cls = out[:, 0, :].contiguous()
        embs.append(cls.detach().cpu())
    return torch.cat(embs, dim=0) if embs else torch.empty(0, 768)

def _locate_json(json_dir: Path, base: str):
    for suf in (".json", ".aug.json", ".unified.json", ".jsonl"):
        cand = json_dir / f"{base}{suf}"
        if cand.exists(): return cand
    matches = list(json_dir.glob(f"{base}.*"))
    return matches[0] if matches else None

def upgrade_split_with(tok, mdl, split, node_store="node",
                       max_len=256, batch_size=32, device="cuda", show_inner=True):
    in_dir   = Path(f"Dataset/{split}/hetero_ready")
    json_dir = Path(f"Dataset/{split}/unified_aug")
    out_dir  = Path(f"Dataset/{split}/hetero_ready_gcbert")
    out_dir.mkdir(parents=True, exist_ok=True)

    pts = sorted(in_dir.glob("*.pt"))
    if not pts:
        print(f"[{split}] No .pt files in {in_dir}")
        return

    ok = err = skipped = 0
    outer = tqdm(pts, desc=f"{split.upper()} shards", unit="file", mininterval=0.2)
    for pt_path in outer:
        base = pt_path.stem
        json_path = _locate_json(json_dir, base)
        if json_path is None:
            skipped += 1
            outer.set_postfix(ok=ok, err=err, skipped=skipped)
            continue
        try:
            data = _as_hetero(torch.load(pt_path, map_location="cpu"))
            store_name = _pick_node_store(data, node_store)
            store = data[store_name]

            nodes, id2idx = _load_aug_json(str(json_path))

            if hasattr(store, "nid"):
                nid = store.nid.view(-1).tolist()
            else:
                N = store.num_nodes
                if N != len(nodes):
                    err += 1
                    outer.set_postfix(ok=ok, err=err, skipped=skipped)
                    continue
                nid = [nodes[i].get("_id") for i in range(len(nodes))]
                store.nid = torch.arange(len(nid), dtype=torch.long)

            # texts aligned with nid (map by _id if present, else by index)
            idx_to_node = {i: nodes[i] for i in range(len(nodes))}
            texts = []
            for i, json_id in enumerate(store.nid.tolist()):
                n = nodes[id2idx.get(json_id, -1)] if json_id in id2idx else idx_to_node.get(i, {})
                texts.append(_node_text(n))

            x_text = _embed_gcbert(
                texts, mdl, tok, device=device,
                max_len=max_len, batch_size=batch_size,
                desc=f"Embed {base}", show_inner=show_inner
            )
            if x_text.size(0) != len(texts):
                err += 1
                outer.set_postfix(ok=ok, err=err, skipped=skipped)
                continue

            store.x_text = x_text
            torch.save(data, out_dir / f"{base}.pt")
            ok += 1
        except Exception:
            err += 1
        outer.set_postfix(ok=ok, err=err, skipped=skipped)

    print(f"[{split}] done  ✓OK={ok}  ✗err={err}  ◔skipped={skipped}  → {out_dir}")

def upgrade_all_splits(node_store="node", max_len=256, batch_size=32, device="cuda", show_inner=True):
    name = "microsoft/graphcodebert-base"
    tok  = AutoTokenizer.from_pretrained(name, use_fast=True)
    mdl  = AutoModel.from_pretrained(name, use_safetensors=True)  # avoids torch<2.6 .bin path
    for split in ("train", "valid", "test"):
        upgrade_split_with(tok, mdl, split, node_store=node_store,
                           max_len=max_len, batch_size=batch_size,
                           device=device, show_inner=show_inner)

# === Run all three splits ===
upgrade_all_splits(node_store="node", max_len=256, batch_size=32, device="cuda", show_inner=True)


TRAIN shards: 100%|██████████| 3438/3438 [4:20:18<00:00,  4.54s/file, err=2, ok=3436, skipped=0]   


[train] done  ✓OK=3436  ✗err=2  ◔skipped=0  → Dataset\train\hetero_ready_gcbert


VALID shards: 100%|██████████| 2905/2905 [43:29<00:00,  1.11file/s, err=0, ok=2905, skipped=0]  


[valid] done  ✓OK=2905  ✗err=0  ◔skipped=0  → Dataset\valid\hetero_ready_gcbert


TEST shards: 100%|██████████| 2915/2915 [44:12<00:00,  1.10file/s, err=0, ok=2915, skipped=0]  

[test] done  ✓OK=2915  ✗err=0  ◔skipped=0  → Dataset\test\hetero_ready_gcbert


gcbert 2


In [17]:
from pathlib import Path
import os, json, math, warnings, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
from torch_geometric.data import HeteroData

# --- Quiet the chatty stuff (optional) ---
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore",
    message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# --- helpers ---
def _as_hetero(obj):
    if isinstance(obj, HeteroData): return obj
    raise TypeError(f"Expected HeteroData; got {type(obj)}")

def _load_aug_json(path):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    nodes = j.get("nodes", [])
    id2idx = {}
    for i, n in enumerate(nodes):
        if isinstance(n, dict) and "_id" in n:
            try:
                id2idx[int(n["_id"])] = i
            except Exception:
                # keep non-int ids as-is (rare)
                id2idx[n["_id"]] = i
    return nodes, id2idx

def _node_text(n: dict) -> str:
    for k in ("code", "name", "methodFullName", "signature"):
        v = n.get(k)
        if isinstance(v, str) and v.strip(): return v.strip()
    return ""

def _pick_node_store(data: HeteroData, preferred: str | None):
    node_types = tuple(getattr(data, "node_types", ()))
    if preferred and preferred in node_types: return preferred
    for cand in ("node","ast","code","token","statement"):
        if cand in node_types: return cand
    if node_types: return node_types[0]
    for key in data.keys(): return key
    raise RuntimeError("Could not determine a node store/type to embed.")

@torch.no_grad()
def _embed_gcbert(texts, model, tok, device="cuda", max_len=256, batch_size=32,
                  desc="Embedding", show_inner=True):
    embs = []
    model.to(device).eval()
    total = (len(texts) + batch_size - 1) // batch_size
    inner = tqdm(range(0, len(texts), batch_size), total=total, desc=desc,
                 unit="batch", leave=False, disable=not show_inner, mininterval=0.2)
    for i in inner:
        chunk = texts[i:i+batch_size]
        enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        if device == "cuda":
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model(**enc).last_hidden_state
        else:
            out = model(**enc).last_hidden_state
        cls = out[:, 0, :].contiguous()
        embs.append(cls.detach().cpu())
    return torch.cat(embs, dim=0) if embs else torch.empty(0, 768)

def _locate_json(json_dir: Path, base: str):
    for suf in (".json", ".aug.json", ".unified.json", ".jsonl"):
        cand = json_dir / f"{base}{suf}"
        if cand.exists(): return cand
    matches = list(json_dir.glob(f"{base}.*"))
    return matches[0] if matches else None

def upgrade_split_with(tok, mdl, split, node_store="node",
                       max_len=256, batch_size=32, device="cuda", show_inner=True):
    in_dir   = Path(f"Dataset/{split}/hetero_ready")
    json_dir = Path(f"Dataset/{split}/unified_aug")
    out_dir  = Path(f"Dataset/{split}/hetero_ready_gcbert")
    out_dir.mkdir(parents=True, exist_ok=True)

    pts = sorted(in_dir.glob("*.pt"))
    if not pts:
        print(f"[{split}] No .pt files in {in_dir}")
        return

    ok = err = skipped = 0
    outer = tqdm(pts, desc=f"{split.upper()} shards", unit="file", mininterval=0.2)
    for pt_path in outer:
        base = pt_path.stem
        json_path = _locate_json(json_dir, base)
        if json_path is None:
            skipped += 1
            outer.set_postfix(ok=ok, err=err, skipped=skipped)
            continue

        try:
            data = _as_hetero(torch.load(pt_path, map_location="cpu"))
            store_name = _pick_node_store(data, node_store)
            store = data[store_name]

            nodes, id2idx = _load_aug_json(str(json_path))
            N = int(store.num_nodes)

            # ---- ensure store.nid contains the REAL Joern ids ----
            if hasattr(store, "nid"):
                # Validate coverage against JSON ids
                nid_list = store.nid.view(-1).tolist()
                matched = sum(1 for v in nid_list if v in id2idx)
                coverage = matched / max(1, len(nid_list))
                if coverage < 0.95 and len(nodes) == N:
                    # Replace nid with JSON _id order to guarantee alignment
                    nid_list = [int(nodes[i]["_id"]) for i in range(N)]
                    store.nid = torch.tensor(nid_list, dtype=torch.long)
                    msg = "replaced_low_coverage"
                else:
                    msg = "kept_existing"
            else:
                if len(nodes) != N:
                    # cannot align reliably
                    err += 1
                    outer.set_postfix(ok=ok, err=err, skipped=skipped)
                    continue
                nid_list = [int(nodes[i]["_id"]) for i in range(N)]
                store.nid = torch.tensor(nid_list, dtype=torch.long)
                msg = "created_from_json"

            # ---- texts aligned with *nid*; fallback to index if missing
            idx_to_node = {i: nodes[i] for i in range(len(nodes))}
            texts = []
            for i, json_id in enumerate(store.nid.tolist()):
                if json_id in id2idx:
                    n = nodes[id2idx[json_id]]
                else:
                    # rare fallback: use same index
                    n = idx_to_node.get(i, {})
                texts.append(_node_text(n))

            x_text = _embed_gcbert(
                texts, mdl, tok, device=device,
                max_len=max_len, batch_size=batch_size,
                desc=f"Embed {base}", show_inner=show_inner
            )
            if x_text.size(0) != len(texts):
                err += 1
                outer.set_postfix(ok=ok, err=err, skipped=skipped)
                continue

            # Store features; keep existing numeric x if any
            store.x_text = x_text  # (N, 768), float32 on CPU; move to GPU at train time

            # Optional: compress to fp16 on disk to save space
            # store.x_text = x_text.half()

            # Save upgraded graph
            torch.save(data, out_dir / f"{base}.pt")
            ok += 1

            # per-file alignment summary (one-liner)
            outer.set_postfix(ok=ok, err=err, skipped=skipped, nid=msg)

        except Exception:
            err += 1
            outer.set_postfix(ok=ok, err=err, skipped=skipped)

    print(f"[{split}] done  ✓OK={ok}  ✗err={err}  ◔skipped={skipped}  → {out_dir}")

def upgrade_all_splits(node_store="node", max_len=256, batch_size=32, device="cuda", show_inner=True):
    name = "microsoft/graphcodebert-base"
    tok  = AutoTokenizer.from_pretrained(name, use_fast=True)
    mdl  = AutoModel.from_pretrained(name, use_safetensors=True)
    for split in ("train", "valid", "test"):
        upgrade_split_with(tok, mdl, split, node_store=node_store,
                           max_len=max_len, batch_size=batch_size,
                           device=device, show_inner=show_inner)

# === Run all three splits ===
if __name__ == "__main__":
    upgrade_all_splits(node_store="node", max_len=256, batch_size=32, device="cuda", show_inner=True)


TRAIN shards: 100%|██████████| 3438/3438 [4:23:59<00:00,  4.61s/file, err=2, nid=created_from_json, ok=3436, skipped=0]   


[train] done  ✓OK=3436  ✗err=2  ◔skipped=0  → Dataset\train\hetero_ready_gcbert


VALID shards: 100%|██████████| 2905/2905 [49:10<00:00,  1.02s/file, err=0, nid=created_from_json, ok=2905, skipped=0]  


[valid] done  ✓OK=2905  ✗err=0  ◔skipped=0  → Dataset\valid\hetero_ready_gcbert


TEST shards: 100%|██████████| 2915/2915 [52:41<00:00,  1.08s/file, err=0, nid=created_from_json, ok=2915, skipped=0]  


[test] done  ✓OK=2915  ✗err=0  ◔skipped=0  → Dataset\test\hetero_ready_gcbert


# === CA GAT ACC Traing - Inter Proc ===

In [21]:
# =======================================
# CA-GAT + ACC (robust static seeds + inter-aware weak labels) — Training Cell
# =======================================
import os, re, json, math, atexit, traceback, warnings, random, time
from pathlib import Path
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from packaging.version import parse as V
from tqdm.auto import tqdm

# PYG
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv

# ------------------------------ top-level config ------------------------------
SPLIT                 = "train"  # "train" | "valid" | "test"
GC_DIR                = f"Dataset/{SPLIT}/hetero_ready_gcbert"
UNIFIED_JSON_DIRS     = [f"Dataset/{SPLIT}/unified_json", f"Dataset/{SPLIT}/unified"]
AUG_JSON_DIRS         = [f"Dataset/{SPLIT}/unified_aug", f"Dataset/{SPLIT}/unified"]
LOGDIR                = f"out/cagat_acc_interproc_{SPLIT}"

NODE_TYPE             = "node"
DEVICE                = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CUDA & memory knobs
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

USE_AMP            = True   # mixed precision
USE_CHECKPOINT     = True   # gradient checkpointing in GAT blocks
CLEAR_CACHE_EVERY  = 25
SKIP_ON_OOM        = True

# Train hyperparams
EPOCHS                = 5
BATCH_SIZE            = 1
HIDDEN                = 128
HEADS                 = 4
LAYERS                = 4
DROPOUT               = 0.10
LR                    = 2e-3
WEIGHT_DECAY          = 1e-4
SAVE_EVERY_STEPS      = 200
LOG_JSONL_EVERY       = 10

# Call-chain contextualization
CALL_CHAIN_ATTN       = True
CALL_CHAIN_MAX_HOPS   = 3
CALL_CHAIN_SPARSE_NTHRESH = 5000

# Loss weights (scheduled)
LAMBDA_NODE_BCE_BASE       = 1.0
LAMBDA_EDGE_SMOOTH_BASE    = 0.05
LAMBDA_INTER_CONTRAST_BASE = 0.05
LAMBDA_PATH_SUPER_BASE     = 0.10

# Edge weights for smoothness
EDGE_WEIGHTS = {
    'CFG': 0.5, 'DFG': 0.8, 'CFG_REV': 0.3, 'DFG_REV': 0.5,
    'CALL': 1.5, 'CALL_REV': 1.0,
    'ARG2PARAM': 2.0, 'ARG2PARAM_REV': 1.5,
    'RET2CALL': 2.0, 'RET2CALL_REV': 1.5,
    'RET2LHS': 1.8, 'RET2LHS_REV': 1.2,
}

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# ------------------------------ AMP handling (Torch 2.4.x clean) ------------------------------
_TVER = V(torch.__version__)
USE_NEW_AMP_API = _TVER >= V("2.5.0")
if USE_NEW_AMP_API:
    from torch.amp import autocast as _autocast_new, GradScaler as _GradScalerNew
    def autocast_ctx(enabled=True): 
        return _autocast_new(device_type=("cuda" if DEVICE.type=="cuda" else "cpu"),
                             dtype=torch.float16, enabled=enabled)
    SCALER = _GradScalerNew("cuda" if DEVICE.type=="cuda" else "cpu", enabled=(USE_AMP and DEVICE.type=="cuda"))
else:
    from torch.cuda.amp import autocast as _autocast_old, GradScaler as _GradScalerOld
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.autocast")
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.GradScaler")
    def autocast_ctx(enabled=True): 
        return _autocast_old(dtype=torch.float16, enabled=(enabled and DEVICE.type=="cuda"))
    SCALER = _GradScalerOld(enabled=(USE_AMP and DEVICE.type=="cuda"))
print(f"[AMP] torch={torch.__version__} device={DEVICE.type}")

# ------------------------------ IO helpers ------------------------------
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def find_json(basename: str) -> str | None:
    for d in AUG_JSON_DIRS:
        jd = Path(d)
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = jd / f"{basename}{suf}"
            if p.exists(): return str(p)
        ms = sorted(jd.glob(f"{basename}.*"))
        if ms: return str(ms[0])
    return None

def _strip_comments_commas(txt: str) -> str:
    txt = re.sub(r'//.*?$', '', txt, flags=re.M)
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
    txt = txt.replace('\ufeff','').replace('\x00','')
    txt = re.sub(r',\s*(\}|\])', r'\1', txt)
    return txt

def _extract_nodes_array(txt: str):
    m = re.search(r'"nodes"\s*:', txt)
    if not m: return None
    i = m.end()
    while i < len(txt) and txt[i] != '[': i += 1
    if i>=len(txt) or txt[i] != '[': return None
    depth=0; start=i
    for j,ch in enumerate(txt[i:], start=i):
        if ch=='[': depth+=1
        elif ch==']':
            depth-=1
            if depth==0: return txt[start:j+1]
    return None

def load_aug_json(path: str) -> dict | list | None:
    if not path or not os.path.exists(path): return None
    p = Path(path)
    for enc in ("utf-8","utf-8-sig","latin-1"):
        try:
            raw = p.read_text(enc)
            break
        except Exception:
            raw = None
    if raw is None: return None
    try:
        j = json.loads(raw); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    san = _strip_comments_commas(raw)
    try:
        j = json.loads(san); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    arr = _extract_nodes_array(san)
    if arr:
        try:
            nodes = json.loads(arr)
            if isinstance(nodes, list): return {"nodes": nodes}
        except json.JSONDecodeError: pass
    nodes=[]; paths=[]
    for line in san.splitlines():
        line=line.strip()
        if not line or line[0] not in "{[": continue
        try:
            o = json.loads(line)
            if isinstance(o, dict):
                if isinstance(o.get("nodes"), list): nodes += o["nodes"]
                if isinstance(o.get("vulnerable_paths"), list): paths += o["vulnerable_paths"]
                for k in ("sinks","vulnerabilities","vul_nodes","labels","positives"):
                    if isinstance(o.get(k), list):
                        nodes += [{'_id': x, 'is_sink': True} for x in o[k]]
            elif isinstance(o, list):
                nodes += o
        except json.JSONDecodeError:
            continue
    if nodes or paths:
        d={"nodes": nodes} if nodes else {}
        if paths: d["vulnerable_paths"]=paths
        return d if d else None
    return None

def _coerce_int(x):
    if isinstance(x, int): return x
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except ValueError: return x
    return x

def _harvest_ids_from_any_json(j):
    pos = set()
    if j is None: return pos
    nodes = j["nodes"] if isinstance(j, dict) and isinstance(j.get("nodes"), list) else (j if isinstance(j, list) else [])
    for n in nodes:
        if not isinstance(n, dict): continue
        nid = _coerce_int(n.get("_id"))
        lab = (n.get("label") or n.get("_label") or n.get("class") or n.get("tag") or "")
        lab_u = str(lab).upper()
        is_sink = bool(n.get("is_sink")) or ("SINK" in lab_u) or ("VULN" in lab_u) or ("VULNERABLE" in lab_u)
        if isinstance(nid, int) and is_sink:
            pos.add(nid)
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            if isinstance(j.get(k), list):
                for x in j[k]:
                    xi = _coerce_int(x)
                    if isinstance(xi, int): pos.add(xi)
        vps = j.get("vulnerable_paths", [])
        if isinstance(vps, list):
            for p in vps:
                if isinstance(p, (list, tuple)) and len(p)>0:
                    a = _coerce_int(p[0]); b = _coerce_int(p[-1])
                    if isinstance(a, int): pos.add(a)
                    if isinstance(b, int): pos.add(b)
    return pos

def node_sink_labels_from_json(json_path: str, g: HeteroData, use_paths_as_pos=True) -> torch.Tensor | None:
    if not json_path or not os.path.exists(json_path): return None
    j = load_aug_json(json_path)
    if j is None: return None
    pos_ids = _harvest_ids_from_any_json(j)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    if hasattr(st, "nid"):
        ids = [ _coerce_int(i) for i in st.nid.view(-1).tolist() ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ids ],
                            dtype=torch.float32, device=device)
    if isinstance(j, dict) and isinstance(j.get("nodes"), list) and len(j["nodes"]) == st.num_nodes:
        ordered_ids = [ _coerce_int(n.get("_id")) if isinstance(n, dict) else None for n in j["nodes"] ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ordered_ids ],
                            dtype=torch.float32, device=device)
    return None

def read_paths_from_json(json_path: str) -> list:
    if not json_path or not os.path.exists(json_path): return []
    j = load_aug_json(json_path)
    if not isinstance(j, dict): return []
    paths = j.get("vulnerable_paths", [])
    return paths if isinstance(paths, list) else []

# ------------------------------ edges / dataset ------------------------------
def get_edge_index(g: HeteroData, et: tuple[str,str,str]) -> torch.Tensor:
    if et not in g.edge_types:
        dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
               else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[et]
    ei = getattr(store, "edge_index", None)
    if ei is not None:
        return ei
    adj_t = getattr(store, "adj_t", None)
    if adj_t is not None:
        row, col, _ = adj_t.coo()     # adj_t is transposed
        return torch.stack([col, row], dim=0)
    dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
           else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
    return torch.zeros((2,0), dtype=torch.long, device=dev)

class GraphDir(Dataset):
    def __init__(self, gc_dir: str):
        self.paths = sorted(Path(gc_dir).glob("*.pt"))
        if not self.paths:
            raise FileNotFoundError(f"No .pt in {gc_dir}")
        print(f"[DATA] {len(self.paths)} graphs in {gc_dir}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt, map_location="cpu")
        g.__dict__["_aug_json_path"] = find_json(pt.stem)
        return g

def validate_interprocedural_coverage(g: HeteroData):
    stats = {}
    for (s, r, t) in g.edge_types:
        if r in ['CALL','ARG2PARAM','RET2CALL','RET2LHS']:
            stats[r] = int(get_edge_index(g,(s,r,t)).size(1))
    if sum(stats.values()) == 0:
        print("⚠️  No inter-procedural edges found!")
    else:
        print("✓ Inter-procedural edges:", stats)
    return stats

def make_inter_mask(g: HeteroData):
    st = g[NODE_TYPE]; N = st.num_nodes
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    inter_nodes = set()
    for (s, r, t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel() > 0:
                inter_nodes.update(ei[0].tolist()); inter_nodes.update(ei[1].tolist())
    device = (st.x_text.device if hasattr(st, 'x_text') else (st.x.device if hasattr(st, 'x') else 'cpu'))
    mask = torch.zeros(N, dtype=torch.bool, device=device)
    if inter_nodes: mask[list(inter_nodes)] = True
    return mask

# ------------------------------ dynamic shrinker ------------------------------
MAX_NODES_FOR_GPU  = 6000
MAX_EDGES_FOR_GPU  = 20000

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N  = st.num_nodes
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES_FOR_GPU and total_edges <= MAX_EDGES_FOR_GPU:
        return g
    keep = set()
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    for (s,r,t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    random.seed(0)
    if len(keep) < min(N, MAX_NODES_FOR_GPU):
        others = [i for i in range(N) if i not in keep]
        random.shuffle(others)
        need = min(N, MAX_NODES_FOR_GPU) - len(keep)
        keep.update(others[:max(0, need)])
    keep = sorted(list(keep))
    idx_map = {old:i for i, old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x_text"):
        g2[NODE_TYPE].x_text = st.x_text[keep].to(dtype=torch.float16, device=st.x_text.device)
    if hasattr(st, "x"):
        g2[NODE_TYPE].x = st.x[keep].to(dtype=torch.float16, device=st.x.device)
    if hasattr(st, "nid"):
        g2[NODE_TYPE].nid = st.nid[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for (s,r,t) in g.edge_types:
        ei = get_edge_index(g, (s,r,t))
        if ei.numel() == 0:
            g2[(s,r,t)].edge_index = ei; continue
        src, dst = ei
        keep_mask = [(u in idx_map and v in idx_map) for u,v in zip(src.tolist(), dst.tolist())]
        if not any(keep_mask):
            g2[(s,r,t)].edge_index = torch.zeros((2,0), dtype=torch.long, device=ei.device); continue
        src_f = [src[i].item() for i, m in enumerate(keep_mask) if m]
        dst_f = [dst[i].item() for i, m in enumerate(keep_mask) if m]
        src_r = torch.tensor([idx_map[u] for u in src_f], dtype=torch.long, device=ei.device)
        dst_r = torch.tensor([idx_map[v] for v in dst_f], dtype=torch.long, device=ei.device)
        if src_r.numel() > MAX_EDGES_FOR_GPU:
            perm = torch.randperm(src_r.numel(), device=ei.device)[:MAX_EDGES_FOR_GPU]
            src_r = src_r[perm]; dst_r = dst_r[perm]
        g2[(s,r,t)].edge_index = torch.stack([src_r, dst_r], dim=0)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------------ model ------------------------------
import torch.utils.checkpoint as cp

class CAGatBlock(nn.Module):
    def __init__(self, in_ch, out_ch, heads, edge_types, dropout=0.1, use_skip=True):
        super().__init__()
        assert out_ch % heads == 0
        self.edge_types = edge_types
        self.use_skip   = use_skip
        self.convs = nn.ModuleDict({
            f"{s}_{r}_{t}": GATConv(in_ch, out_ch//heads, heads=heads, concat=True,
                                    add_self_loops=False, dropout=dropout)
            for (s,r,t) in edge_types
        })
        self.gates = nn.ParameterDict({ f"{s}_{r}_{t}": nn.Parameter(torch.zeros(1)) for (s,r,t) in edge_types })
        self.norm  = nn.LayerNorm(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
        self.skip_alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, g: HeteroData):
        out_dim = self.proj.out_features if isinstance(self.proj, nn.Linear) else x.size(1)
        out_accum = torch.zeros(x.size(0), out_dim, device=x.device, dtype=x.dtype)
        for (s,r,t) in self.edge_types:
            et = (s,r,t)
            ei = get_edge_index(g, et)
            if ei.numel()==0: continue
            key   = f"{s}_{r}_{t}"
            out = self.convs[key](x, ei)
            gate  = torch.sigmoid(self.gates[key])
            out_accum = out_accum + gate * out
        out_accum = self.drop(out_accum)
        res = self.proj(x)
        out_accum = self.skip_alpha * res + (1 - self.skip_alpha) * out_accum
        return self.norm(out_accum)

class CAGAT_ACC_InterProc(nn.Module):
    def __init__(self, in_dim, hidden, heads, layers, edge_types, dropout=0.1,
                 call_chain_attn=False, max_hops=3, sparse_threshold=5000):
        super().__init__()
        self.edge_types = edge_types
        self.hidden     = hidden
        self.projectors = nn.ModuleDict()
        self.blocks     = nn.ModuleList([
            CAGatBlock(hidden, hidden, heads, edge_types, dropout=dropout, use_skip=True)
            for _ in range(layers)
        ])
        self.intra_head = nn.Linear(hidden, hidden)
        he_init = nn.Linear(hidden, hidden); nn.init.xavier_uniform_(he_init.weight); nn.init.zeros_(he_init.bias)
        self.inter_head = he_init
        self.context_gru = nn.GRU(hidden, hidden, batch_first=True)
        self.lin_out   = nn.Linear(hidden, 1)
        self.call_chain_attn = call_chain_attn
        self.max_hops        = max_hops
        self.sparse_threshold= sparse_threshold
    def _project_in(self, x: torch.Tensor) -> torch.Tensor:
        d = x.size(1); key = f"proj_{d}"
        if key not in self.projectors:
            lin = nn.Linear(d, self.hidden, device=x.device, dtype=x.dtype)
            nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))
            if lin.bias is not None:
                fan_in, _ = nn.init._calculate_fan_in_and_fan_out(lin.weight)
                bound = 1 / math.sqrt(fan_in); nn.init.uniform_(lin.bias, -bound, bound)
            self.projectors[key] = lin
        else:
            self.projectors[key] = self.projectors[key].to(device=x.device, dtype=x.dtype)
        return F.relu(self.projectors[key](x))
    def _call_chain_attn_sparse(self, call_ei, h, N):
        device = h.device
        idx = call_ei.to(device); vals = torch.ones(idx.size(1), device=device)
        A = torch.sparse_coo_tensor(idx, vals, size=(N, N))
        reach = A
        for _ in range(self.max_hops - 1):
            reach = reach + torch.sparse.mm(A, reach)
        neigh = reach.to_dense().clamp(max=1.0); neigh.fill_diagonal_(0)
        deg = neigh.sum(dim=1, keepdim=True).clamp(min=1)
        ctx = neigh @ h / deg
        return h + 0.1 * ctx
    def _call_chain_attn_bfs(self, call_ei, h, N):
        adj = defaultdict(list)
        for i in range(call_ei.size(1)):
            adj[call_ei[0,i].item()].append(call_ei[1,i].item())
        ctx = torch.zeros_like(h)
        for node_id in range(N):
            neighbors = []
            q = deque([(node_id,0)]); seen = {node_id}
            while q:
                cur, d = q.popleft()
                if d >= self.max_hops: continue
                for nb in adj.get(cur, []):
                    if nb not in seen:
                        seen.add(nb); neighbors.append(nb); q.append((nb, d+1))
            if neighbors: ctx[node_id] = h[neighbors].mean(dim=0)
        return h + 0.1 * ctx
    def forward(self, g: HeteroData):
        st = g[NODE_TYPE]; xs=[]
        if hasattr(st,"x_text"): xs.append(st.x_text)
        if hasattr(st,"x"):      xs.append(st.x.float())
        if len(xs) == 1: x = xs[0].to(device=xs[0].device, dtype=torch.float32)
        else: 
            target_device = xs[0].device; xs = [t.to(device=target_device, dtype=torch.float32) for t in xs]
            x = torch.cat(xs, dim=1)
        h = self._project_in(x)
        for blk in self.blocks:
            if USE_CHECKPOINT and h.requires_grad:
                h = cp.checkpoint(lambda inp: F.elu(blk(inp, g)), h, use_reentrant=False)
            else:
                h = F.elu(blk(h, g))
        inter_mask = make_inter_mask(g).to(h.device)
        h_mix = torch.where(inter_mask.unsqueeze(-1), self.inter_head(h), self.intra_head(h))
        et = (NODE_TYPE,'CALL',NODE_TYPE); ei = get_edge_index(g, et)
        if ei.numel() > 0:
            caller = h_mix[ei[0]]; callee = h_mix[ei[1]]
            seq = torch.stack([caller, callee], dim=1)  # [E,2,H]
            ctx,_ = self.context_gru(seq)
            callee_upd = ctx[:,1,:]
            deg = torch.zeros(h_mix.size(0), device=h_mix.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=h_mix.device))
            add = torch.zeros_like(h_mix); add.index_add_(0, ei[1], callee_upd)
            mask = (deg > 0).unsqueeze(-1)
            h_mix = torch.where(mask, h_mix + 0.2 * (add / deg.clamp(min=1).unsqueeze(-1)), h_mix)
        if self.call_chain_attn:
            call_ei = get_edge_index(g, (NODE_TYPE,'CALL',NODE_TYPE))
            if call_ei.numel() > 0:
                N = h_mix.size(0)
                h_mix = self._call_chain_attn_sparse(call_ei, h_mix, N) if N >= self.sparse_threshold else self._call_chain_attn_bfs(call_ei, h_mix, N)
        logit = self.lin_out(h_mix).squeeze(-1)
        return logit, h_mix

# ------------------------------ losses & metrics ------------------------------
def weighted_edge_smoothness(g: HeteroData, node_logit: torch.Tensor):
    loss = 0.0; count = 0
    for (s,r,t) in g.edge_types:
        w = EDGE_WEIGHTS.get(r, 0.0); 
        if w == 0.0: continue
        ei = get_edge_index(g, (s,r,t)); 
        if ei.numel()==0: continue
        u, v = ei[0], ei[1]
        loss = loss + w * F.l1_loss(node_logit[u], node_logit[v])
        count += 1
    return loss / max(1, count)

def inter_procedural_contrastive_loss(g: HeteroData, emb: torch.Tensor, temperature=0.1):
    loss = 0.0; count = 0
    for r in ['CALL','ARG2PARAM','RET2CALL']:
        et = (NODE_TYPE, r, NODE_TYPE)
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        a = emb[ei[0]]; b = emb[ei[1]]
        pos_sim = F.cosine_similarity(a, b, dim=-1)
        loss = loss - torch.log(torch.sigmoid(pos_sim / temperature)).mean()
        count += 1
    return loss / max(1, count)

def path_supervision_loss(g: HeteroData, node_logit: torch.Tensor, paths_node_ids: list):
    if not paths_node_ids: 
        return node_logit.new_zeros(())
    st = g[NODE_TYPE]; id2pos = None
    if hasattr(st, "nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    loss = 0.0; cnt  = 0
    for path in paths_node_ids:
        idxs = []
        for nid in path:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if (isinstance(nid, int) and 0 <= nid < st.num_nodes) else None)
            if j is None: idxs=[]; break
            idxs.append(j)
        if len(idxs) < 2: continue
        pl = node_logit[idxs]
        # enforce increasing scores root..sink
        loss = loss + sum(F.relu(pl[i] - pl[i+1]) for i in range(len(pl)-1)) / (len(pl)-1)
        cnt += 1
    return loss / max(1, cnt) if cnt>0 else node_logit.new_zeros(())

def pred_threshold(epoch, total_epochs):
    return 0.30 + 0.20 * (epoch / max(1, total_epochs))

def compute_inter_or_fallback_metrics(g, node_logit, y, thr=0.5):
    mask = make_inter_mask(g)
    if y.sum() == 0:
        return {"note": "no_positives"}
    if mask.sum() > 0 and (y[mask].sum() > 0):
        inter_logits = node_logit[mask]; inter_labels = y[mask]
        pred = (torch.sigmoid(inter_logits) > thr).float()
        tp = (pred * inter_labels).sum()
        prec = tp / (pred.sum() + 1e-8); rec  = tp / (inter_labels.sum() + 1e-8)
        acc  = (pred == inter_labels).float().mean()
        return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec)}
    pred = (torch.sigmoid(node_logit) > thr).float()
    tp = (pred * y).sum()
    prec = tp / (pred.sum() + 1e-8); rec  = tp / (y.sum() + 1e-8)
    acc  = (pred == y).float().mean()
    return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec), 'note': 'fallback_global'}

def get_loss_weights(epoch, total_epochs):
    prog = epoch / total_epochs
    return {
        'node':     LAMBDA_NODE_BCE_BASE,
        'smooth':   LAMBDA_EDGE_SMOOTH_BASE    + 0.15 * prog,  # 0.05 → 0.20
        'contrast': LAMBDA_INTER_CONTRAST_BASE + 0.15 * prog,  # 0.05 → 0.20
        'path':     LAMBDA_PATH_SUPER_BASE     + 0.20 * prog,  # 0.10 → 0.30
    }

# ------------------------------ Weak-supervision miner (fast + robust static seeds) ------------------------------
FAST_MINER          = True        # if False, also use regex over JSON node text
TIME_BUDGET_SEC     = 3.0         # per-graph budget
MAX_JSON_BYTES      = 2_000_000
MAX_FRONTIER        = 4096
MAX_VISITED         = 15000
MINER_MAX_HOPS      = 7
MINER_LIMIT_PATHS   = 8

# Static patterns kept (more robust, broader coverage). These are *additive* with structural seeds.
SINK_NAME_PATTERNS = [
    r"\b(exec|system|popen|spawn|CreateProcess|ShellExecute|eval|Runtime\.exec|ProcessBuilder)\b",
    r"\b(sql|execute(Query|Update)|Statement\.execute|rawQuery|PreparedStatement)\b",
    r"\b(send|write|print|fprintf|fwrite|OutputStream\.write|FileWriter|FileOutputStream)\b",
    r"\b(deserialize|ObjectInputStream|pickle\.loads|JSON\.parse)\b",
]
ROOT_NAME_PATTERNS = [
    r"\b(argv|stdin|getenv|getopt|request|req|input|params?|query|body|post|recv|read|Scanner|BufferedReader)\b",
    r"\b(getParameter|getHeader|getQueryString|getInputStream|readLine)\b",
]

_JSON_CACHE = {}; _JSON_KEYS  = []; _JSON_CACHE_LIMIT = 512
def _json_cache_get(path):
    if not path or not os.path.exists(path): return None
    try:
        if os.path.getsize(path) > MAX_JSON_BYTES: return None
    except Exception: pass
    if path in _JSON_CACHE: return _JSON_CACHE[path]
    j = load_aug_json(path)
    if j is not None:
        _JSON_CACHE[path] = j; _JSON_KEYS.append(path)
        if len(_JSON_KEYS) > _JSON_CACHE_LIMIT:
            old = _JSON_KEYS.pop(0); _JSON_CACHE.pop(old, None)
    return j

def _load_nodes_map_fast(json_path:str):
    j = _json_cache_get(json_path)
    if not isinstance(j, dict) or not isinstance(j.get("nodes"), list):
        return [], {}
    nodes = j["nodes"]; id2node = {}
    for n in nodes:
        if isinstance(n, dict) and "_id" in n:
            id2node[_coerce_int(n["_id"])] = n
    return nodes, id2node

def _node_text_from_json(n:dict) -> str:
    for k in ("code","name","methodFullName","signature","call","label","nodeType"):
        v = n.get(k)
        if isinstance(v,str) and v.strip(): return v
    return ""

def _regex_any(patterns, s):
    if not isinstance(s, str): return False
    for p in patterns:
        if re.search(p, s, flags=re.I): return True
    return False

def _build_adj_fast(g: HeteroData):
    allowed_rels = {'CALL','ARG2PARAM','RET2CALL','RET2LHS','DFG'}
    adj = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in allowed_rels: continue
        ei = get_edge_index(g,(s,r,t))
        if ei.numel()==0: continue
        src, dst = ei
        buckets = defaultdict(list)
        for u,v in zip(src.tolist(), dst.tolist()): buckets[u].append(v)
        for u, vs in buckets.items():
            adj[u].extend(vs[:64])  # cap out-degree
    return adj

def _bfs_paths_bounded(adj, starts, targets, max_hops=5, limit=8, time_budget=1.0):
    t0 = time.monotonic(); paths = []; targets = set(targets)
    visited_budget = 0
    for s in starts:
        q = deque([(s, [s])])
        while q and len(paths) < limit:
            if (time.monotonic() - t0) > time_budget: return paths
            cur, path = q.popleft()
            if len(path) - 1 > max_hops: continue
            if cur in targets and len(path) > 1:
                paths.append(path[:])
                if len(paths) >= limit: break
            nb_list = adj.get(cur, [])[:64]
            for nb in nb_list:
                if nb in path: continue
                q.append((nb, path + [nb])); visited_budget += 1
                if visited_budget > MAX_VISITED: return paths
            if len(q) > MAX_FRONTIER:
                while len(q) > MAX_FRONTIER: q.pop()
    return paths

def mine_roots_sinks_and_paths(g: HeteroData, json_path: str,
                               max_hops=MINER_MAX_HOPS, limit_paths=MINER_LIMIT_PATHS):
    t0 = time.monotonic()
    roots, sinks, paths = [], [], []
    # (A) structural seeds (always available; robust)
    # sinks = CALL callees + RET2CALL/RET2LHS targets
    for r in ('RET2LHS','RET2CALL','CALL'):
        ei = get_edge_index(g,(NODE_TYPE,r,NODE_TYPE))
        if ei.numel()>0: sinks.extend(ei[1].unique().tolist())
    sinks = list(set(sinks))[:256]
    # roots = ARG2PARAM sources + high-outdegree DFG sources
    ei = get_edge_index(g,(NODE_TYPE,'ARG2PARAM',NODE_TYPE))
    if ei.numel()>0: roots = ei[0].unique().tolist()[:256]
    dfg = get_edge_index(g,(NODE_TYPE,'DFG',NODE_TYPE))
    if dfg.numel()>0:
        deg = torch.zeros(g[NODE_TYPE].num_nodes, device=dfg.device); deg.index_add_(0, dfg[0], torch.ones(dfg.size(1), device=deg.device))
        high = torch.topk(deg, k=min(128, deg.numel())).indices.tolist()
        roots = list(set((roots or []) + high))
    # (B) optional lexical seeds from JSON node text (if enabled)
    if (not FAST_MINER) and json_path:
        nodes, id2node = _load_nodes_map_fast(json_path)
        if hasattr(g[NODE_TYPE], "nid") and id2node:
            g_ids = g[NODE_TYPE].nid.view(-1).tolist()
            hit_roots=set(); hit_sinks=set()
            for nid in g_ids[:4096]:
                n = id2node.get(_coerce_int(nid)); 
                if not isinstance(n, dict): continue
                txt = _node_text_from_json(n)
                if _regex_any(SINK_NAME_PATTERNS, txt): hit_sinks.add(nid)
                if _regex_any(ROOT_NAME_PATTERNS, txt): hit_roots.add(nid)
            if hit_sinks: sinks = list(set(sinks + list(hit_sinks)))
            if hit_roots: roots = list(set(roots + list(hit_roots)))
    # Mine paths between roots and sinks
    if roots and sinks:
        adj = _build_adj_fast(g)
        rem = max(0.2, TIME_BUDGET_SEC - (time.monotonic() - t0))
        paths = _bfs_paths_bounded(adj, starts=roots, targets=sinks,
                                   max_hops=max_hops, limit=limit_paths, time_budget=rem)
    # fallback: stub edges
    if not paths and roots and sinks:
        for r in roots[:4]:
            for s in sinks[:4]:
                if r != s:
                    paths.append([r, s])
                    if len(paths) >= limit_paths: break
            if len(paths) >= limit_paths: break
    return roots, sinks, paths

def _ensure_inter_pos(g, y_soft, paths):
    """Guarantee at least one positive inside the inter-procedural slice."""
    mask = make_inter_mask(g)
    if (y_soft[mask] > 0.5).any(): 
        return y_soft, paths
    # try to promote a node on a mined path that lies in the slice
    promoted = False
    id2pos = None
    if hasattr(g[NODE_TYPE],"nid"):
        ids = g[NODE_TYPE].nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    for path in paths:
        for nid in path[::-1]:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if isinstance(nid,int) else None)
            if j is not None and j < y_soft.numel() and mask[j]:
                y_soft[j] = max(y_soft[j], 0.9); promoted = True; break
        if promoted: break
    # if still none, pick high-degree CALL callee
    if not promoted:
        ei = get_edge_index(g,(NODE_TYPE,'CALL',NODE_TYPE))
        if ei.numel()>0:
            deg = torch.zeros(y_soft.numel(), device=y_soft.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=y_soft.device))
            j = int(deg.argmax().item())
            if mask[j]: y_soft[j] = max(y_soft[j], 0.9)
    return y_soft, paths

def build_labels_with_weak_supervision(g: HeteroData):
    json_path = getattr(g, "_aug_json_path", None)
    # 1) try hard labels
    y = node_sink_labels_from_json(json_path, g); hard_paths = read_paths_from_json(json_path)
    if y is not None and (float(y.sum().item()) > 0 or (hard_paths and len(hard_paths)>0)):
        return y.to(y.device), hard_paths, {"mode":"hard_json"}
    # 2) mine paths
    roots, sinks, mined_paths = mine_roots_sinks_and_paths(g, json_path,
                                                           max_hops=MINER_MAX_HOPS,
                                                           limit_paths=MINER_LIMIT_PATHS)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    N = st.num_nodes
    y_soft = torch.zeros(N, dtype=torch.float32, device=device)
    # soft labels along paths: ramp 0.3 → 0.9
    if hasattr(st,"nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
        for path in mined_paths:
            L = max(2, len(path)); 
            for k, nid in enumerate(path):
                j = id2pos.get(_coerce_int(nid)); 
                if j is None or j >= N: continue
                tgt = 0.3 + 0.6 * (k / (L-1))  # root ~0.3 … sink ~0.9
                y_soft[j] = max(y_soft[j], tgt)
    # ensure at least one inter-proc positive
    y_soft, mined_paths = _ensure_inter_pos(g, y_soft, mined_paths)
    return y_soft, mined_paths, {"mode":"weak_mined", "roots": len(roots), "sinks": len(sinks), "paths": len(mined_paths)}

print(f"GC_DIR={GC_DIR}")
print(f"UNIFIED_JSON_DIRS={UNIFIED_JSON_DIRS}")
print(f"AUG_JSON_DIRS={AUG_JSON_DIRS}")

# ------------------------------ training ------------------------------
def train(
    gc_dir=GC_DIR,
    logdir=LOGDIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    hidden=HIDDEN,
    heads=HEADS,
    layers=LAYERS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_every=SAVE_EVERY_STEPS,
    warmup_max_steps=None,    # if set, run bounded warm-up
):
    ds = GraphDir(gc_dir)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=False)

    sample = safe_load(ds.paths[0], map_location="cpu")
    st = sample[NODE_TYPE]
    in_dim = (st.x_text.size(1) if hasattr(st, "x_text") else 0) + (st.x.size(1) if hasattr(st, "x") else 0)
    etypes = tuple(sample.edge_types)

    if hidden % heads != 0:
        new_hidden = (hidden // heads + 1) * heads
        print(f"[CFG] HIDDEN={hidden} not divisible by HEADS={heads} → adjusting to {new_hidden}")
        hidden = new_hidden

    model = CAGAT_ACC_InterProc(
        in_dim=in_dim, hidden=hidden, heads=heads, layers=layers,
        edge_types=etypes, dropout=DROPOUT,
        call_chain_attn=CALL_CHAIN_ATTN, max_hops=CALL_CHAIN_MAX_HOPS,
        sparse_threshold=CALL_CHAIN_SPARSE_NTHRESH
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    os.makedirs(logdir, exist_ok=True)
    ckpt_path       = os.path.join(logdir, "ckpt_cagat_acc_interproc.pt")
    best_path       = os.path.join(logdir, "best_model.pt")
    summary_path    = os.path.join(logdir, "train_summary.json")
    log_jsonl_path  = os.path.join(logdir, "training_log.jsonl")

    best_inter_recall = 0.0
    log_jsonl_fh = open(log_jsonl_path, "a", encoding="utf-8")

    def save(tag="SAVE", steps=0, loss_avg=0.0, extras=None, best=False):
        payload = {"model": model.state_dict()}
        torch.save(payload, best_path if best else ckpt_path)
        out = {"loss_avg": loss_avg, "steps": steps, "epochs": epochs}
        if isinstance(extras, dict): out.update(extras)
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)
        tqdm.write(f"[{tag}] -> {(best_path if best else ckpt_path)}")

    atexit.register(lambda: save("ATEXIT", steps=0, loss_avg=0.0))
    atexit.register(lambda: log_jsonl_fh.close())

    steps, loss_sum = 0, 0.0
    graphs_seen = graphs_with_pos = graphs_with_inter_pos = 0

    total_steps = (warmup_max_steps if warmup_max_steps is not None else epochs * len(dl))
    print(f"[TRAIN] Device: {DEVICE}")
    print(f"[TRAIN] Sample: N={st.num_nodes}, in_dim={in_dim}, edge_types={len(etypes)}")
    validate_interprocedural_coverage(sample)

    phase = "Warmup" if warmup_max_steps is not None else "Training"
    pbar = tqdm(total=total_steps, desc=f"{phase}", unit="step", dynamic_ncols=True)

    try:
        epoch_iter = [1] if warmup_max_steps is not None else range(1, epochs + 1)
        for ep in epoch_iter:
            w = get_loss_weights(ep, epochs if warmup_max_steps is None else 1)
            for i, g in enumerate(dl, 1):
                if warmup_max_steps is not None and steps >= warmup_max_steps:
                    break
                tried_shrink = False
                while True:
                    try:
                        g_work = g.to(DEVICE, non_blocking=False)
                        with autocast_ctx(enabled=USE_AMP):
                            node_logit, h = model(g_work)
                            st_g = g_work[NODE_TYPE]

                            # Hard labels → else inter-aware weak labels
                            y = node_sink_labels_from_json(getattr(g_work, "_aug_json_path", None), g_work)
                            paths = read_paths_from_json(getattr(g_work, "_aug_json_path", None))
                            label_info = None
                            if (y is None) or (float(y.sum().item()) == 0 and not paths):
                                y, paths, label_info = build_labels_with_weak_supervision(g_work)

                            y = y.to(node_logit.device)
                            inter_mask = make_inter_mask(g_work)

                            graphs_seen += 1
                            pos_all   = int((y > 0.5).sum().item())
                            pos_inter = int(((y > 0.5).float() * inter_mask.float()).sum().item())
                            if pos_all > 0: graphs_with_pos += 1
                            if pos_inter > 0: graphs_with_inter_pos += 1

                            # BCE supports soft targets
                            pos = (y > 0.5).sum()
                            neg = y.numel() - pos
                            pos_weight = (neg / (pos + 1e-6)).clamp_(1.0, 100.0)
                            node_loss = F.binary_cross_entropy_with_logits(node_logit, y, pos_weight=pos_weight)

                            # Aux losses
                            smooth    = weighted_edge_smoothness(g_work, node_logit)
                            contra    = inter_procedural_contrastive_loss(g_work, h)
                            path_loss = path_supervision_loss(g_work, node_logit, paths) if paths else node_logit.new_zeros(())

                            loss = (w['node'] * node_loss + w['smooth'] * smooth +
                                    w['contrast'] * contra + w['path'] * path_loss)

                        # step
                        opt.zero_grad(set_to_none=True)
                        SCALER.scale(loss).backward()
                        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                        SCALER.step(opt); SCALER.update()

                        # metrics
                        thr = pred_threshold(ep, epochs if warmup_max_steps is None else 1)
                        metrics = compute_inter_or_fallback_metrics(g_work, node_logit.detach(), (y > 0.5).float(), thr=thr)

                        steps += 1; loss_sum += float(loss.detach().cpu())
                        postfix = dict(ep=ep, step=steps, loss=f"{loss_sum/steps:.4f}",
                                       i_acc=(f"{metrics.get('inter_acc', 0):.3f}" if metrics else "N/A"),
                                       i_rec=(f"{metrics.get('inter_recall', 0):.3f}" if metrics else "N/A"))
                        if "note" in metrics: postfix["note"] = metrics["note"]
                        pbar.set_postfix(**postfix); pbar.update(1)

                        # jsonl
                        if (steps % LOG_JSONL_EVERY) == 0:
                            log_line = {
                                "epoch": ep, "step": steps,
                                "loss": float(loss.detach().cpu()),
                                "node_loss": float(node_loss.detach().cpu()),
                                "smooth": float(smooth.detach().cpu()) if torch.is_tensor(smooth) else float(smooth),
                                "contrast": float(contra.detach().cpu()) if torch.is_tensor(contra) else float(contra),
                                "path_loss": float(path_loss.detach().cpu()) if torch.is_tensor(path_loss) else float(path_loss),
                                "pos_any": pos_all, "pos_inter": pos_inter,
                                **{k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))}
                            }
                            if isinstance(metrics.get("note"), str): log_line["note"] = metrics["note"]
                            if label_info: log_line["label_info"] = label_info
                            log_jsonl_fh.write(json.dumps(log_line) + "\n"); log_jsonl_fh.flush()

                        inter_rec = float(metrics.get('inter_recall', 0.0))
                        if inter_rec > best_inter_recall and warmup_max_steps is None:
                            best_inter_recall = inter_rec
                            save("BEST (inter_recall↑)", steps=steps, loss_avg=loss_sum/steps, extras=metrics, best=True)

                        # tidy
                        del node_logit, h, y, loss, node_loss, smooth, contra, path_loss, g_work
                        if (i % CLEAR_CACHE_EVERY) == 0 and DEVICE.type == "cuda":
                            torch.cuda.empty_cache()
                        break

                    except RuntimeError as e:
                        if 'out of memory' in str(e).lower():
                            torch.cuda.empty_cache()
                            if not tried_shrink:
                                g = shrink_graph_if_needed(g); tried_shrink = True
                                tqdm.write("[INFO] Retrying with shrunken graph to avoid OOM.")
                                continue
                            elif SKIP_ON_OOM:
                                tqdm.write("[WARN] Skipping one graph due to CUDA OOM even after shrink."); break
                            else:
                                raise
                        else:
                            raise

            if warmup_max_steps is not None and steps >= warmup_max_steps:
                break

    except Exception:
        save("FINAL (EXC)", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        traceback.print_exc(); raise
    finally:
        save("FINAL", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        pbar.close(); log_jsonl_fh.close()

    # ----------- training report -----------
    report = {
        "phase": ("warmup" if warmup_max_steps is not None else "train"),
        "steps": steps,
        "avg_loss": loss_sum/max(1,steps),
        "graphs_seen": graphs_seen,
        "graphs_with_any_pos": graphs_with_pos,
        "graphs_with_inter_pos": graphs_with_inter_pos,
        "best_inter_recall": (best_inter_recall if warmup_max_steps is None else None),
        "log_jsonl": log_jsonl_path,
        "summary_json": summary_path,
        "ckpt_last": os.path.join(logdir, "ckpt_cagat_acc_interproc.pt"),
        "ckpt_best": (os.path.join(logdir, "best_model.pt") if warmup_max_steps is None else None),
    }
    # Persist report
    with open(os.path.join(logdir, "training_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    if warmup_max_steps is not None:
        print(f"[WARMUP] summary: {report}")
    else:
        print(f"[TRAIN] summary: {report}")

# ------------------------------ Convenience wrappers ------------------------------
def warmup_check(max_steps=300):
    print("[WARMUP] starting…")
    train(warmup_max_steps=max_steps)

def full_run():
    print("[FULL] starting…")
    train()

# ========================== toggles (set & run cell) ==========================
DO_WARMUP   = False    # quick sanity
DO_FULL_RUN = True   # full training later

# Miner knobs (you can tweak live)
FAST_MINER      = True     # False => also use regex on JSON node text
TIME_BUDGET_SEC = 2.0      # try 3.0–5.0 if your GPU/CPU can handle

# Make log dir
os.makedirs(LOGDIR, exist_ok=True)

# Execute
if DO_WARMUP:
    warmup_check(max_steps=300)
if DO_FULL_RUN:
    full_run()


AttributeError: partially initialized module 'torch_geometric' has no attribute 'typing' (most likely due to a circular import)

Final Runner

Inference - augmented json

In [ ]:
# Inference: handle checkpoint/data input-dim mismatches safely
import os, json, glob, torch
from tqdm.auto import tqdm

# Import the model we wrote in Cell 5
import ca_gat_acc
from ca_gat_acc import CAGAT_ACC_Model, _pick_x

def load_aug_json(path):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    id2info = {}
    for n in j.get("nodes", []):
        _id = n.get("_id")
        if _id is None:
            continue
        id2info[_id] = {
            "file": n.get("filename") or n.get("file") or "<unknown>",
            "line": n.get("lineNumber", -1),
            "code": (n.get("code") or n.get("name") or "<code unavailable>").strip(),
            "label": (n.get("label") or n.get("_label") or "").upper(),
        }
    return id2info

# ---- safe torch.load preferring weights_only=False (newer Torch) ----
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


@torch.no_grad()
def backtrace_chains(out, data, topk=3, tau=None):
    logits = out["logits"]                     # [N,2]
    sink_p = logits.softmax(-1)[:, -1]         # prob of "vulnerable/sink" class
    k = min(topk, sink_p.numel())
    sinks = torch.topk(sink_p, k=k).indices.tolist()

    chains = []
    H = len(out["alpha"])
    for s in tqdm(sinks, desc="Backtracing sinks", unit="sink"):
        cur = s; path = [cur]
        for h in range(H-1, -1, -1):
            best_u, best_w = None, -1.0
            for et, wvec in out["alpha"][h].items():
                ei = data[et].edge_index       # [2, E]
                src, dst = ei
                mask = (dst == cur)
                if not mask.any():
                    continue
                # wvec is on CPU already
                w = wvec[mask.nonzero(as_tuple=False).view(-1)]
                if tau is not None:
                    w = w * (w >= tau).float()
                if w.numel() == 0:
                    continue
                idx = torch.argmax(w)
                u   = src[mask][idx].item()
                if float(w[idx]) > best_w:
                    best_w, best_u = float(w[idx]), u
            if best_u is None:
                break
            path.append(best_u); cur = u
        chains.append(list(reversed(path)))
    return chains, sink_p

def pretty_write(chains, data, id2info, out_md):
    nid = getattr(data["node"], "nid", None)
    lines = ["# Causal Chains (GC-BERT + CA-GAT + ACC)", ""]
    for ci, chain in enumerate(chains, 1):
        lines.append(f"## Chain {ci} (len={len(chain)})")
        for i, u in enumerate(chain):
            role = "ROOT" if i == 0 else ("SINK" if i == len(chain)-1 else f"hop{i}")
            if nid is not None:
                nid_u = int(nid[u].item())
                info  = id2info.get(nid_u, {})
                file  = info.get("file","?"); line = info.get("line",-1)
                code  = info.get("code","<code unavailable>").splitlines()
                snippet = (code[0] if code else "").strip()
                lines.append(f"- **{role}** node={u} id={nid_u} @ `{file}:{line}`")
                if snippet:
                    lines.append(f"  - `{snippet}`")
            else:
                lines.append(f"- **{role}** node={u}")
        lines.append("")
    os.makedirs(os.path.dirname(out_md) or ".", exist_ok=True)
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[infer] wrote {out_md}")

# ---------- pick a test shard ----------
PTS = sorted(glob.glob("/content/test_gcbert/*.pt"))
assert PTS, "No upgraded test shards found in /content/test_gcbert. Run the GC-BERT upgrade on test."
PT  = PTS[0]
JSON = os.path.splitext(PT)[0] + ".json"

# ---------- load graph + checkpoint ----------
data  = safe_load(PT, map_location="cpu") # Use safe_load to load the data
state = torch.load("/content/out/logs_acc_gcbert/ckpt_acc.pt", map_location="cpu")

# For diagnostics
train_in_dim = state["model"]["lin_in.weight"].shape[1]
data_in_dim  = _pick_x(data).size(1)
print(f"[infer] checkpoint_in_dim={train_in_dim}, data_in_dim={data_in_dim}")

# Build model with the input dimension from the checkpoint
model = CAGAT_ACC_Model(
    hidden=128, heads=4, H=3,
    edge_types=tuple(data.edge_types),
    num_classes=2,
    in_dim=train_in_dim  # <-- Use the input dimension from the checkpoint
)

# Load the state dictionary with strict=True
model.load_state_dict(state.get("model", state), strict=True)
print(f"[infer] loaded checkpoint state with strict=True.")
model.eval()

# ---------- forward + chain extraction ----------
with torch.no_grad():
    out = model(data)  # this will print one-time CAGAT_ACC confirmations and rebuild lin_in to data_in_dim

chains, scores = backtrace_chains(out, data, topk=3)
os.makedirs("/content/out/chains", exist_ok=True)
out_json = f"/content/out/chains/{os.path.basename(PT).replace('.pt','')}_chains.json"
with open(out_json, "w") as f:
    json.dump({"pt": PT, "topk_sink_scores": [float(s) for s in scores.tolist()], "chains": chains}, f, indent=2)
print(f"[infer] wrote {out_json}")

id2info = load_aug_json(JSON)
out_md = f"/content/out/chains/{os.path.basename(PT).replace('.pt','')}_chains_pretty.md"
pretty_write(chains, data, id2info, out_md)

# quick peek
!sed -n 
d"

Inference - unified json

In [ ]:
# === Inference (fixed mapping + fixed backtrace) ===
import os, json, glob, torch
from tqdm.auto import tqdm

# Import model helpers from Cell 5
import ca_gat_acc
from ca_gat_acc import CAGAT_ACC_Model, _pick_x

# ---- tolerant torch.load (prefers weights_only=False if available) ----
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ---- read augmented JSON; return both id->info AND index->info ----
def load_aug_json_bimaps(path, expect_N=None):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    nodes = j.get("nodes", [])
    # Normalize a few field names we care about
    def _info(n):
        file = n.get("filename") or n.get("file") or "<unknown>"
        line = n.get("lineNumber") or n.get("line") or n.get("lineno") or -1
        code = (n.get("code") or n.get("name") or n.get("methodFullName") or "<code unavailable>").strip()
        lab  = (n.get("label") or n.get("_label") or "").upper()
        return {"file": file, "line": int(line) if isinstance(line, (int,str)) and str(line).isdigit() else -1,
                "code": code, "label": lab}
    id2info = {}
    for n in nodes:
        _id = n.get("_id")
        if _id is not None:
            id2info[_id] = _info(n)
    # Index-aligned map (position in nodes array → info)
    idx2info = {}
    if nodes:
        upto = len(nodes) if expect_N is None else min(expect_N, len(nodes))
        for i in range(upto):
            idx2info[i] = _info(nodes[i])
    return id2info, idx2info, len(nodes)

@torch.no_grad()
def backtrace_chains(out, data, topk=3, tau=None, max_hops=None):
    """Greedy backtrace from top-k sinks using stored per-edge attentions."""
    logits = out["logits"]
    sink_p = logits.softmax(-1)[:, -1]
    k = min(topk, sink_p.numel())
    sinks = torch.topk(sink_p, k=k).indices.tolist()

    chains = []
    H = len(out["alpha"])
    for s in tqdm(sinks, desc="Backtracing sinks", unit="sink"):
        cur = s
        path = [cur]
        visited = {cur}
        hops_budget = H if max_hops is None else max_hops
        for h in range(H-1, -1, -1):
            if hops_budget is not None and len(path)-1 >= hops_budget:
                break
            best_u, best_w = None, -1.0
            for et, wvec in out["alpha"][h].items():
                ei = data[et].edge_index           # [2, E]
                src, dst = ei
                mask = (dst == cur)
                if not mask.any():
                    continue
                # wvec is a 1D tensor on CPU with one weight per edge of this type
                idxs = mask.nonzero(as_tuple=False).view(-1)
                w = wvec[idxs]
                if tau is not None:
                    w = w * (w >= tau).float()
                if w.numel() == 0:
                    continue
                j = torch.argmax(w)
                u = src[mask][j].item()
                score = float(w[j])
                if score > best_w:
                    best_w, best_u = score, u
            if best_u is None or best_u in visited:
                break
            path.append(best_u)
            visited.add(best_u)
            cur = best_u
        chains.append(list(reversed(path)))
    return chains, sink_p

def pretty_write(chains, data, id2info, idx2info, out_md):
    N = data["node"].num_nodes
    nid = getattr(data["node"], "nid", None)
    # Stats to show whether we're mapping mostly by ids or by indices
    hits_by_id = hits_by_idx = 0

    lines = ["# Causal Chains (GC-BERT + CA-GAT + ACC)", ""]
    for ci, chain in enumerate(chains, 1):
        lines.append(f"## Chain {ci} (len={len(chain)})")
        for i, u in enumerate(chain):
            role = "ROOT" if i == 0 else ("SINK" if i == len(chain)-1 else f"hop{i}")
            info = None
            # Try id-based mapping first (if nid exists and lands in id2info)
            if nid is not None:
                nid_u = int(nid[u].item())
                info = id2info.get(nid_u)
                if info is not None:
                    hits_by_id += 1
                    file, line, code = info["file"], info["line"], info["code"]
                    snippet = code.splitlines()[0].strip() if code else "<code unavailable>"
                    lines.append(f"- **{role}** node={u} id={nid_u} @ `{file}:{line}`")
                    if snippet: lines.append(f"  - `{snippet}`")
                    continue
            # Fallback: index-aligned mapping (position u in JSON nodes)
            info = idx2info.get(u)
            if info is not None:
                hits_by_idx += 1
                file, line, code = info["file"], info["line"], info["code"]
                snippet = code.splitlines()[0].strip() if code else "<code unavailable>"
                lines.append(f"- **{role}** node={u} @ `{file}:{line}`")
                if snippet: lines.append(f"  - `{snippet}`")
            else:
                lines.append(f"- **{role}** node={u} (no mapping)")
        lines.append("")

    os.makedirs(os.path.dirname(out_md) or ".", exist_ok=True)
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[infer] wrote {out_md}")
    total_lookups = hits_by_id + hits_by_idx
    if total_lookups:
        print(f"[infer] mapping coverage: by_id={hits_by_id}, by_index={hits_by_idx}, "
              f"total_mapped={total_lookups}", flush=True)
    else:
        print("[infer] WARNING: no nodes could be mapped to JSON; check your JSON partner files.", flush=True)

# ---------- pick a test shard ----------
PTS = sorted(glob.glob("/content/test_gcbert/*.pt"))
assert PTS, "No upgraded test shards found in /content/test_gcbert. Upgrade your test split first."
PT   = PTS[0]
JSON = os.path.splitext(PT)[0] + ".json"

# ---------- load graph + checkpoint ----------
data  = safe_load(PT, map_location="cpu")
state = torch.load("/content/out/logs_acc_gcbert/ckpt_acc.pt", map_location="cpu")

# Determine the expected input dimension from the checkpoint
train_in_dim = state["model"]["lin_in.weight"].shape[1]
data_in_dim  = _pick_x(data).size(1)
print(f"[infer] checkpoint_in_dim={train_in_dim}, data_in_dim={data_in_dim}")

# Build model with the input dimension from the checkpoint
model = CAGAT_ACC_Model(
    hidden=128, heads=4, H=3,
    edge_types=tuple(data.edge_types),
    num_classes=2,
    in_dim=train_in_dim  # <-- Initialize model with the checkpoint's expected input dimension
)

# Load the state dictionary with strict=True
# This should now work without size mismatch for lin_in
model.load_state_dict(state.get("model", state), strict=True)
print(f"[infer] loaded checkpoint state with strict=True.")
model.eval()

# ---------- forward + chain extraction ----------
with torch.no_grad():
    out = model(data)  # this will print one-time CAGAT_ACC confirmations and rebuild lin_in to data_in_dim

chains, scores = backtrace_chains(out, data, topk=3, tau=None)
os.makedirs("/content/out/chains", exist_ok=True)
out_json = f"/content/out/chains/{os.path.basename(PT).replace('.pt','')}_chains.json"
with open(out_json, "w") as f:
    json.dump({"pt": PT, "topk_sink_scores": [float(s) for s in scores.tolist()], "chains": chains}, f, indent=2)
print(f"[infer] wrote {out_json}")

# ---------- pretty with robust mapping ----------
id2info, idx2info, jsonN = load_aug_json_bimaps(JSON, expect_N=data["node"].num_nodes)
out_md = f"/content/out/chains/{os.path.basename(PT).replace('.pt','')}_chains_pretty.md"
pretty_write(chains, data, id2info, idx2info, out_md)

# quick peek
!sed -n '1,120p' "$out_md"

Inference with backtrack and pritty print

In [ ]:
# ==== ALL-IN-ONE INFERENCE + CHAIN CONSTRUCTION (robust JSON + all hops) ====
import os, glob, json, math, collections
import torch
from tqdm.auto import tqdm

# ---------------- paths (edit as needed) ----------------
TEST_DIR   = "/content/test_gcbert"                        # GC-BERT-upgraded test shards + matching JSON
CKPT_PATH  = "/content/out/logs_acc_gcbert/ckpt_acc.pt"    # trained model checkpoint
OUT_DIR    = "/content/out/chains"                         # outputs

# ---------------- chain extraction knobs ----------------
TOPK_SINKS          = 3      # number of sinks per shard
MAX_HOPS            = None   # or int to cap root..sink hops
TOP_M_PER_HOP       = 5      # keep up to M strongest predecessors per hop
TAU_REL             = 0.30   # also keep predecessors with weight >= TAU_REL * max_w (per-hop)
MAX_PATHS_PER_SINK  = 64     # cap to avoid path explosion

# treat these edge types as "propagation" (tune as needed)
SEMANTIC_ET = {
    ("node","DFG","node"),
    ("node","CFG","node"),
    ("node","CALL","node"),
    ("node","ARG2PARAM","node"),
    ("node","RET2CALL","node"),
    ("node","RET2LHS","node"),
}

# ---------------- import your model (Cell 5 must have been run) ----------------
import ca_gat_acc
from ca_gat_acc import CAGAT_ACC_Model, _pick_x

# ---------------- tolerant torch.load ----------------
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ---------------- unified JSON helpers (robust to weird formats) ----------------
PLACEHOLDER_CODES = {"<empty>", "<global>", "RET"}
PREF_NODE_LABELS  = {"CALL","IDENTIFIER","RETURN","LITERAL","FIELD_IDENTIFIER","CONTROL_STRUCTURE","JUMP_TARGET"}

def _extract_ast_edges(j):
    """Return list of (parent, child) AST edges from a variety of JSON formats."""
    ast_pairs = []

    edges = j.get("edges", None)

    # Case A: edges is a dict-of-lists, e.g., {"AST":[{...}, ...], ...}
    if isinstance(edges, dict):
        cand = edges.get("AST", [])
        if isinstance(cand, list):
            for e in cand:
                if isinstance(e, dict):
                    label = e.get("_label") or e.get("label") or e.get("type")
                    if label == "AST":
                        p = e.get("outV") or e.get("src") or e.get("source")
                        c = e.get("inV")  or e.get("dst") or e.get("target")
                        if p is not None and c is not None:
                            ast_pairs.append((p, c))
        return ast_pairs

    # Case B: edges is a list
    if isinstance(edges, list):
        for e in edges:
            if isinstance(e, dict):
                label = e.get("_label") or e.get("label") or e.get("type")
                if label == "AST":
                    p = e.get("outV") or e.get("src") or e.get("source")
                    c = e.get("inV")  or e.get("dst") or e.get("target")
                    if p is not None and c is not None:
                        ast_pairs.append((p, c))
            # if it's a string or other type, ignore
        return ast_pairs

    # Case C: some exports stash AST as a top-level list under 'AST'
    if isinstance(j.get("AST", None), list):
        for e in j["AST"]:
            if isinstance(e, dict):
                p = e.get("outV") or e.get("src") or e.get("source")
                c = e.get("inV")  or e.get("dst") or e.get("target")
                if p is not None and c is not None:
                    ast_pairs.append((p, c))
        return ast_pairs

    # Case D: fallback — try per-node children fields
    nodes = j.get("nodes", [])
    for n in nodes:
        if not isinstance(n, dict):
            continue
        pid = n.get("_id")
        for key in ("children","astChildren","AST_CHILDREN"):
            if key in n and isinstance(n[key], list):
                for c in n[key]:
                    if pid is not None and isinstance(c, int):
                        ast_pairs.append((pid, c))
    return ast_pairs

def load_unified_json(path):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)

    id2node = {}
    ast_par  = {}
    ast_kids = collections.defaultdict(list)
    method_of = {}
    returns_in_method = collections.defaultdict(list)
    method_file = {}

    # index nodes
    for n in j.get("nodes", []):
        if not isinstance(n, dict):
            continue
        _id = n.get("_id")
        if _id is None:
            continue
        id2node[_id] = n

    # index AST edges (robust)
    ast_edges = _extract_ast_edges(j)
    for p, c in ast_edges:
        ast_par[c] = p
        ast_kids[p].append(c)

    # helper: nearest METHOD (walk AST parents)
    def nearest_method(u):
        cur = u; seen=set()
        while cur in id2node and cur not in seen:
            seen.add(cur)
            n = id2node[cur]
            if n.get("_label") == "METHOD":
                return cur
            cur = ast_par.get(cur)
            if cur is None: break
        return None

    # map each node to its METHOD
    for _id in list(id2node.keys()):
        mid = nearest_method(_id)
        if mid is not None:
            method_of[_id] = mid

    # collect RETURN nodes per METHOD to replace METHOD_RETURN
    for _id, n in id2node.items():
        if n.get("_label") == "RETURN":
            mid = method_of.get(_id)
            if mid: returns_in_method[mid].append(_id)

    # filename by method (fallbacks included)
    for _id, n in id2node.items():
        if n.get("_label") == "METHOD":
            method_file[_id] = n.get("filename") or n.get("file") or n.get("path") or "<unknown>"

    return {
        "json": j,
        "id2node": id2node,
        "ast_par": ast_par,
        "ast_kids": ast_kids,
        "method_of": method_of,
        "returns_in_method": returns_in_method,
        "method_file": method_file,
    }

def node_code(ctx, u):
    n = ctx["id2node"].get(u, {})
    return (n.get("code") or n.get("name") or "").strip()

def is_placeholder(ctx, u):
    c = node_code(ctx, u)
    return (not c) or (c in PLACEHOLDER_CODES)

def prefer_stmt_descendant(ctx, u, max_depth=6):
    kids = ctx["ast_kids"]; id2node=ctx["id2node"]
    q = collections.deque([(u,0)])
    while q:
        v,d = q.popleft()
        if d > max_depth: break
        n   = id2node.get(v, {})
        lbl = n.get("_label","")
        c   = node_code(ctx, v)
        if c and (c not in PLACEHOLDER_CODES) and (lbl in PREF_NODE_LABELS or len(c) > 1):
            return v
        for w in kids.get(v, []):
            q.append((w, d+1))
    return None

def canonicalize(ctx, u):
    n   = ctx["id2node"].get(u, {})
    lbl = n.get("_label","")
    if lbl in {"METHOD_RETURN","METHOD","NAMESPACE_BLOCK","BLOCK"} or is_placeholder(ctx, u):
        mid = ctx["method_of"].get(u)
        if lbl == "METHOD_RETURN" and mid:
            rets = ctx["returns_in_method"].get(mid, [])
            if rets:
                ln  = n.get("lineNumber", None)
                if ln is not None:
                    rets = sorted(rets, key=lambda r: abs((ctx["id2node"].get(r,{}).get("lineNumber", ln))-ln))
                return rets[0]
        v = prefer_stmt_descendant(ctx, u, max_depth=6)
        return v if v is not None else u
    return u

def pick_snippet(ctx, u):
    id2node, method_of, method_file = ctx["id2node"], ctx["method_of"], ctx["method_file"]
    n   = id2node.get(u, {})
    mid = method_of.get(u)
    fn  = method_file.get(mid, n.get("filename") or n.get("file") or n.get("path") or "<unknown>")
    ln  = n.get("lineNumber", n.get("line", -1))
    try: ln = int(ln)
    except: ln = -1
    cd  = node_code(ctx, u)
    return fn, ln, cd

# ---------------- hop-graph builder (captures many predecessors per hop) ----------------
def _reduce_wvec_to_E(wvec, E):
    t = torch.as_tensor(wvec)
    if t.numel() == E:
        return t.float()
    if t.numel() % E == 0:
        H = t.numel() // E
        return t.view(E, H).mean(dim=1).float()  # average across heads
    return t[:E].float() if t.numel() > E else torch.nn.functional.pad(t.float(), (0, E - t.numel()))

def build_hopgraph(out, data, allowed_edge_types, tau_rel=0.3, top_m=5):
    """
    Returns: list[dict], where G[h][dst] = [(src, edge_label, weight), ...] for that layer (h = 0..H-1).
    """
    H = len(out["alpha"])
    G = [collections.defaultdict(list) for _ in range(H)]
    for h in range(H-1, -1, -1):
        layer_alpha = out["alpha"][h]
        for et, wvec in layer_alpha.items():
            if allowed_edge_types and et not in allowed_edge_types:
                continue
            ei = data[et].edge_index
            src, dst = ei[0], ei[1]
            E = int(dst.numel())
            w = _reduce_wvec_to_E(wvec, E)     # [E]
            # group incoming edges by dst
            bucket = collections.defaultdict(list)
            for e_idx in range(E):
                d = int(dst[e_idx]); s = int(src[e_idx]); ww = float(w[e_idx])
                bucket[d].append((s, ww))
            # keep strong preds per dst
            for d, lst in bucket.items():
                if not lst: continue
                max_w = max(w_ for (_, w_) in lst)
                thr   = max_w * float(tau_rel)
                kept  = [(s_,w_) for (s_,w_) in lst if w_ >= thr]
                kept.sort(key=lambda x: x[1], reverse=True)
                if top_m is not None: kept = kept[:top_m]
                for (s_,w_) in kept:
                    G[h][d].append((s_, et[1], w_))   # store relation label only
    return G

def enumerate_paths_from_hopgraph(G, sink, max_hops=None, max_paths=64):
    """
    Enumerate simple paths from ROOT(s) to 'sink' using hop graph G (one dict per layer).
    Path format: [(node_idx, hopinfo), ...] ordered root..sink, where hopinfo is
    (edge_lbl, alpha) or None. The root’s hopinfo is None.
    """
    H = len(G)
    paths = []
    visited_paths = set()

    def dfs(cur, h, acc_nodes, acc_hops):
        # acc_nodes: [sink, ... backward ...]
        # acc_hops : [None, (edge_lbl, w), ...] aligned to acc_nodes
        if (max_hops is not None) and (len(acc_nodes)-1 >= max_hops):
            key = tuple(acc_nodes)
            if key not in visited_paths:
                visited_paths.add(key)
                paths.append(list(zip(acc_nodes, acc_hops)))
            return

        if h < 0 or cur not in G[h] or not G[h][cur]:
            key = tuple(acc_nodes)
            if key not in visited_paths:
                visited_paths.add(key)
                paths.append(list(zip(acc_nodes, acc_hops)))
            return

        for (src, et_lbl, w) in G[h][cur]:
            if src in acc_nodes:
                continue  # avoid cycles
            if len(paths) >= max_paths:
                return
            dfs(src, h-1, acc_nodes + [src], acc_hops + [(et_lbl, w)])

    # start from sink (backward), placeholder hopinfo at sink position
    dfs(sink, H-1, [sink], [None])

    # flip to root..sink order and align hopinfo to edges (root has None)
    fixed = []
    for seq in paths:
        nodes, infos = zip(*seq)                 # tuples
        nodes = list(reversed(nodes))            # lists now
        infos = list(reversed(infos))
        infos = infos[1:] + [None]               # keep list type throughout (FIXED)
        fixed.append(list(zip(nodes, infos)))
    return fixed

# ---------------- pretty output ----------------
def write_pretty_markdown(out_md, chains, data, uctx):
    nid = getattr(data["node"], "nid", None)
    json_nodes = uctx["json"].get("nodes", [])
    lines = ["# Vulnerability Causal Chains (GC-BERT + CA-GAT + ACC)", ""]
    for ci, path in enumerate(chains, 1):
        lines.append(f"## Chain {ci} (len={len(path)})")
        for i, (node_idx, hopinfo) in enumerate(path):
            role = "ROOT" if i == 0 else ("SINK" if i == len(path)-1 else f"hop{i}")

            # prefer joern id mapping via nid; fallback to index-aligned if lengths match
            joern_id = None
            if nid is not None:
                joern_id = int(nid[node_idx].item())
            elif len(json_nodes) == data["node"].num_nodes and node_idx < len(json_nodes):
                joern_id = json_nodes[node_idx].get("_id")

            if joern_id is None:
                lines.append(f"- **{role}** node={node_idx} (no mapping)")
                continue

            best_u   = canonicalize(uctx, joern_id)
            fn, ln, cd = pick_snippet(uctx, best_u)
            etxt = ""
            if hopinfo is not None:
                (edge_lbl, w) = hopinfo
                etxt = f"  ←[{edge_lbl}, α={w:.3f}]"
            lines.append(f"- **{role}** {fn}:{ln} | `{cd}`{etxt}")
        lines.append("")
    os.makedirs(os.path.dirname(out_md) or ".", exist_ok=True)
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[infer] wrote {out_md}")

def write_raw_json(out_json, sink_scores, all_paths):
    doc = {
        "topk_sink_scores": [float(x) for x in sink_scores.tolist()],
        "chains": [
            {
                "nodes": [int(n) for (n, _) in path],
                "hops":  [
                    (None if info is None else {"edge": info[0], "alpha": float(info[1])})
                    for (_, info) in path
                ],
            }
            for path in all_paths
        ],
    }
    os.makedirs(os.path.dirname(out_json) or ".", exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(doc, f, indent=2)
    print(f"[infer] wrote {out_json}")

# ---------------- main runner ----------------
def run_all():
    os.makedirs(OUT_DIR, exist_ok=True)
    pts = sorted(glob.glob(os.path.join(TEST_DIR, "*.pt")))
    assert pts, f"No .pt found under {TEST_DIR}"
    print(f"[infer] using checkpoint: {CKPT_PATH}")
    full_state = torch.load(CKPT_PATH, map_location="cpu")
    sd = full_state.get("model", full_state)

    for pt in tqdm(pts, desc="Shards", unit="file"):
        json_path = os.path.splitext(pt)[0] + ".json"
        if not os.path.exists(json_path):
            print(f"[WARN] missing JSON for {pt}; skipping")
            continue

        # load graph + build model (drop lin_in.* so projector adapts per shard)
        data = safe_load(pt, map_location="cpu")
        model = CAGAT_ACC_Model(hidden=128, heads=4, H=3,
                                edge_types=tuple(data.edge_types),
                                num_classes=2, in_dim=None)
        filtered = {k:v for k,v in sd.items() if not k.startswith("lin_in.")}
        model.load_state_dict(filtered, strict=False)
        model.eval()

        with torch.no_grad():
            out = model(data)  # will print one-time CAGAT_ACC confirmations

        # build per-layer hop graph capturing ALL plausible predecessors
        G = build_hopgraph(out, data, allowed_edge_types=SEMANTIC_ET,
                           tau_rel=TAU_REL, top_m=TOP_M_PER_HOP)

        # pick top sinks
        logits  = out["logits"]; sink_p = logits.softmax(-1)[:, -1]
        k = min(TOPK_SINKS, sink_p.numel())
        top_sinks = torch.topk(sink_p, k=k).indices.tolist()

        # enumerate multiple paths per sink (bounded)
        all_paths = []
        for s in tqdm(top_sinks, desc="Sinks", leave=False):
            paths = enumerate_paths_from_hopgraph(G, sink=s,
                                                 max_hops=MAX_HOPS,
                                                 max_paths=MAX_PATHS_PER_SINK)
            all_paths.extend(paths)

        # pretty/JSON outputs (AST-aware canonicalization of placeholders)
        uctx = load_unified_json(json_path)
        base = os.path.basename(pt).replace(".pt","")
        out_md   = os.path.join(OUT_DIR, f"{base}_chains_code.md")
        out_json = os.path.join(OUT_DIR, f"{base}_chains_all.json")
        write_pretty_markdown(out_md, all_paths, data, uctx)
        write_raw_json(out_json, sink_p, all_paths)

    print(f"[infer] done. Outputs in: {OUT_DIR}")

# go!
run_all()

# quick peek
some_md = sorted(glob.glob(os.path.join(OUT_DIR, "*_chains_code.md")))
if some_md:
    print("\n--- preview ---")
    !sed -n '1,160p' "{some_md[0]}"


Infer - Single best chain

In [ ]:
# ==== ALL CHAINS (sorted high→low) with code-side-by-side + ASCII flows (single cell) ====
import os, glob, json, collections, math, textwrap
import torch
from tqdm.auto import tqdm

# ---------- paths / knobs ----------
TEST_DIR   = "/content/test_gcbert"                        # test shards + matching unified/augmented JSON
CKPT_PATH  = "/content/out/logs_acc_gcbert/ckpt_acc.pt"    # trained checkpoint
OUT_DIR    = "/content/out/chains"                         # outputs here

TOPK_SINKS          = 3      # consider top-K sinks (raise for more coverage)
MAX_HOPS            = None   # or int (cap chain length)
TOP_M_PER_HOP       = 3      # keep up to M predecessors per hop
TAU_REL             = 0.35   # also keep preds >= TAU_REL * max_w per hop
MAX_PATHS_PER_SINK  = 64     # cap to avoid path explosion

# Prefer data/proc-flow edges; we will NOT print labels anyway
SEMANTIC_ET = {
    ("node","DFG","node"),
    ("node","CALL","node"),
    ("node","ARG2PARAM","node"),
    ("node","RET2CALL","node"),
    ("node","RET2LHS","node"),
    # add ("node","CFG","node") if you want control steps allowed too
}

# ---------- model import (Cell 5 must have created ca_gat_acc.py) ----------
import ca_gat_acc
from ca_gat_acc import CAGAT_ACC_Model, _pick_x

def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ---------- unified JSON helpers (robust) ----------
PLACEHOLDER_CODES = {"<empty>", "<global>", "RET"}
PREF_NODE_LABELS  = {"CALL","IDENTIFIER","RETURN","LITERAL","FIELD_IDENTIFIER","CONTROL_STRUCTURE","JUMP_TARGET"}

def _extract_ast_edges(j):
    ast_pairs = []
    edges = j.get("edges", None)
    if isinstance(edges, dict):  # {"AST":[{...}, ...], ...}
        cand = edges.get("AST", [])
        if isinstance(cand, list):
            for e in cand:
                if isinstance(e, dict):
                    p = e.get("outV") or e.get("src") or e.get("source")
                    c = e.get("inV")  or e.get("dst") or e.get("target")
                    if p is not None and c is not None:
                        ast_pairs.append((p, c))
        return ast_pairs
    if isinstance(edges, list):
        for e in edges:
            if isinstance(e, dict) and (e.get("_label") == "AST" or e.get("label") == "AST" or e.get("type") == "AST"):
                p = e.get("outV") or e.get("src") or e.get("source")
                c = e.get("inV")  or e.get("dst") or e.get("target")
                if p is not None and c is not None:
                    ast_pairs.append((p, c))
        return ast_pairs
    if isinstance(j.get("AST", None), list):  # top-level AST array fallback
        for e in j["AST"]:
            if isinstance(e, dict):
                p = e.get("outV") or e.get("src") or e.get("source")
                c = e.get("inV")  or e.get("dst") or e.get("target")
                if p is not None and c is not None:
                    ast_pairs.append((p, c))
        return ast_pairs
    nodes = j.get("nodes", [])
    for n in nodes:  # per-node children fallback
        if not isinstance(n, dict): continue
        pid = n.get("_id")
        for key in ("children","astChildren","AST_CHILDREN"):
            if key in n and isinstance(n[key], list):
                for c in n[key]:
                    if pid is not None and isinstance(c, int):
                        ast_pairs.append((pid, c))
    return ast_pairs

def load_unified_json(path):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    id2node = {}
    ast_par  = {}
    ast_kids = collections.defaultdict(list)
    method_of = {}
    returns_in_method = collections.defaultdict(list)
    method_file = {}
    for n in j.get("nodes", []):
        if not isinstance(n, dict): continue
        _id = n.get("_id")
        if _id is None: continue
        id2node[_id] = n
    for p, c in _extract_ast_edges(j):
        ast_par[c] = p
        ast_kids[p].append(c)
    def nearest_method(u):
        cur = u; seen=set()
        while cur in id2node and cur not in seen:
            seen.add(cur)
            if id2node[cur].get("_label") == "METHOD":
                return cur
            cur = ast_par.get(cur)
            if cur is None: break
        return None
    for _id in list(id2node.keys()):
        mid = nearest_method(_id)
        if mid is not None:
            method_of[_id] = mid
    for _id, n in id2node.items():
        if n.get("_label") == "RETURN":
            mid = method_of.get(_id)
            if mid: returns_in_method[mid].append(_id)
    for _id, n in id2node.items():
        if n.get("_label") == "METHOD":
            method_file[_id] = n.get("filename") or n.get("file") or n.get("path") or "<unknown>"
    return {"json": j, "id2node": id2node, "ast_par": ast_par, "ast_kids": ast_kids,
            "method_of": method_of, "returns_in_method": returns_in_method, "method_file": method_file}

def node_code(ctx, u):
    n = ctx["id2node"].get(u, {})
    return (n.get("code") or n.get("name") or "").strip()

def is_placeholder(ctx, u):
    c = node_code(ctx, u)
    return (not c) or (c in PLACEHOLDER_CODES)

def prefer_stmt_descendant(ctx, u, max_depth=6):
    kids = ctx["ast_kids"]; id2node=ctx["id2node"]
    q = collections.deque([(u,0)])
    while q:
        v,d = q.popleft()
        if d > max_depth: break
        n   = id2node.get(v, {})
        lbl = n.get("_label","")
        c   = node_code(ctx, v)
        if c and (c not in PLACEHOLDER_CODES) and (lbl in {"METHOD"} or lbl in PREF_NODE_LABELS or len(c) > 1):
            if lbl != "METHOD":
                return v
        for w in kids.get(v, []):
            q.append((w, d+1))
    return None

def canonicalize(ctx, u):
    n   = ctx["id2node"].get(u, {})
    lbl = n.get("_label","")
    if lbl in {"METHOD_RETURN","METHOD","NAMESPACE_BLOCK","BLOCK"} or is_placeholder(ctx, u):
        mid = ctx["method_of"].get(u)
        if lbl == "METHOD_RETURN" and mid:
            rets = ctx["returns_in_method"].get(mid, [])
            if rets:
                ln  = n.get("lineNumber", None)
                if ln is not None:
                    rets = sorted(rets, key=lambda r: abs((ctx["id2node"].get(r,{}).get("lineNumber", ln))-ln))
                return rets[0]
        v = prefer_stmt_descendant(ctx, u, max_depth=6)
        return v if v is not None else u
    return u

def pick_snippet(ctx, u):
    id2node, method_of, method_file = ctx["id2node"], ctx["method_of"], ctx["method_file"]
    n   = id2node.get(u, {})
    mid = method_of.get(u)
    fn  = method_file.get(mid, n.get("filename") or n.get("file") or n.get("path") or "<unknown>")
    ln  = n.get("lineNumber", n.get("line", -1))
    try: ln = int(ln)
    except: ln = -1
    cd  = node_code(ctx, u)
    return fn, ln, cd

# ---------- hop-graph + enumeration ----------
def _reduce_wvec_to_E(wvec, E):
    t = torch.as_tensor(wvec)
    if t.numel() == E: return t.float()
    if t.numel() % E == 0:
        H = t.numel() // E
        return t.view(E, H).mean(dim=1).float()
    return t[:E].float() if t.numel() > E else torch.nn.functional.pad(t.float(), (0, E - t.numel()))

def build_hopgraph(out, data, allowed_edge_types, tau_rel=0.35, top_m=3):
    H = len(out["alpha"])
    G = [collections.defaultdict(list) for _ in range(H)]
    for h in range(H-1, -1, -1):
        layer_alpha = out["alpha"][h]
        for et, wvec in layer_alpha.items():
            if allowed_edge_types and et not in allowed_edge_types: continue
            ei = data[et].edge_index
            src, dst = ei[0], ei[1]
            E = int(dst.numel())
            w = _reduce_wvec_to_E(wvec, E)
            bucket = collections.defaultdict(list)
            for e_idx in range(E):
                d = int(dst[e_idx]); s = int(src[e_idx]); ww = float(w[e_idx])
                bucket[d].append((s, ww))
            for d, lst in bucket.items():
                if not lst: continue
                max_w = max(w_ for (_, w_) in lst)
                thr   = max_w * float(tau_rel)
                kept  = [(s_,w_) for (s_,w_) in lst if w_ >= thr]
                kept.sort(key=lambda x: x[1], reverse=True)
                if top_m is not None: kept = kept[:top_m]
                for (s_,w_) in kept:
                    G[h][d].append((s_, et[1], w_))
    return G

def enumerate_paths_from_hopgraph(G, sink, max_hops=None, max_paths=64):
    H = len(G)
    paths, seen = [], set()
    def dfs(cur, h, acc_nodes, acc_hops):
        if (max_hops is not None) and (len(acc_nodes)-1 >= max_hops):
            key = tuple(acc_nodes);
            if key not in seen: seen.add(key); paths.append(list(zip(acc_nodes, acc_hops)));
            return
        if h < 0 or cur not in G[h] or not G[h][cur]:
            key = tuple(acc_nodes);
            if key not in seen: seen.add(key); paths.append(list(zip(acc_nodes, acc_hops)));
            return
        for (src, et_lbl, w) in G[h][cur]:
            if src in acc_nodes: continue
            if len(paths) >= max_paths: return
            dfs(src, h-1, acc_nodes+[src], acc_hops+[(et_lbl, w)])
    dfs(sink, H-1, [sink], [None])
    fixed = []
    for seq in paths:
        nodes, infos = zip(*seq)
        nodes, infos = list(reversed(nodes)), list(reversed(infos))
        infos = infos[1:] + [None]
        fixed.append(list(zip(nodes, infos)))
    return fixed

# ---------- scoring + rendering ----------
def score_path(path, sink_prob):
    ws = [info[1] for (_, info) in path if info is not None]
    mean_alpha = sum(ws)/len(ws) if ws else 0.0
    return float(sink_prob) * float(mean_alpha)

def path_to_ascii(path, data, uctx):
    """Human-friendly single-line flow: ROOT -> hop -> ... -> SINK, showing file:line | code."""
    nid = getattr(data["node"], "nid", None)
    parts = []
    for i, (node_idx, _info) in enumerate(path):
        role = "ROOT" if i == 0 else ("SINK" if i == len(path)-1 else f"hop{i}")
        if nid is None:
            snippet = f"node={node_idx}"
        else:
            j = int(nid[node_idx].item())
            u2 = canonicalize(uctx, j)
            fn, ln, cd = pick_snippet(uctx, u2)
            code = cd.replace("\n"," ").strip()
            if len(code) > 90: code = code[:87] + "…"
            snippet = f"{fn}:{ln} | {code}"
        parts.append(f"[{role}] {snippet}")
    return "  ->  ".join(parts)

def write_all_chains_md(out_md, all_paths_sorted, scores, data, uctx):
    """One markdown per shard: overview + each chain (ASCII + full table)."""
    os.makedirs(os.path.dirname(out_md) or ".", exist_ok=True)
    nid = getattr(data["node"], "nid", None)
    lines = ["# All Vulnerability Chains (sorted high→low)", ""]
    lines.append("> **Role legend**: ROOT = upstream cause, hop# = propagation step(s), SINK = vulnerable point.")
    lines.append("")
    # Overview table
    lines.append("## Overview (all chains)")
    lines.append("| # | score | len | start | end |")
    lines.append("|---|-------|-----|-------|-----|")
    for idx, (path, sc) in enumerate(zip(all_paths_sorted, scores), 1):
        def ep(node_idx):
            if nid is None: return f"node={node_idx}"
            j = int(nid[node_idx].item()); u2 = canonicalize(uctx, j)
            fn, ln, cd = pick_snippet(uctx, u2)
            cd = cd.replace("\n"," ").strip()
            if len(cd) > 60: cd = cd[:57] + "…"
            return f"`{fn}:{ln}` · `{cd}`"
        start = ep(path[0][0]); end = ep(path[-1][0])
        lines.append(f"| {idx} | {sc:.6f} | {len(path)} | {start} | {end} |")
    lines.append("")

    # Per-chain sections
    for idx, (path, sc) in enumerate(zip(all_paths_sorted, scores), 1):
        lines.append(f"## Chain {idx} — score={sc:.6f}, len={len(path)}")
        # ASCII flow
        flow = path_to_ascii(path, data, uctx)
        lines.append("")
        lines.append("```\n" + flow + "\n```")
        lines.append("")
        # Full step table
        lines.append("| Step | Role | Location | Code |")
        lines.append("|------|------|----------|------|")
        for i, (node_idx, _info) in enumerate(path):
            role = "ROOT" if i == 0 else ("SINK" if i == len(path)-1 else f"hop{i}")
            if nid is None:
                loc = f"node={node_idx}"; code = "<unmapped>"
            else:
                j = int(nid[node_idx].item())
                u2 = canonicalize(uctx, j)
                fn, ln, cd = pick_snippet(uctx, u2)
                loc = f"`{fn}:{ln}`"
                # keep code readable; inline backticks force monospaced view
                code = f"`{cd}`" if cd else "`<unavailable>`"
            lines.append(f"| {i} | {role} | {loc} | {code} |")
        lines.append("")

    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[all] wrote {out_md}")

# ---------- main ----------
def run_all_sorted():
    os.makedirs(OUT_DIR, exist_ok=True)
    pts = sorted(glob.glob(os.path.join(TEST_DIR, "*.pt")))
    assert pts, f"No .pt found under {TEST_DIR}"
    print(f"[infer] using checkpoint: {CKPT_PATH}")

    full_state = torch.load(CKPT_PATH, map_location="cpu")
    sd = full_state.get("model", full_state)

    for pt in tqdm(pts, desc="Shards", unit="file"):
        json_path = os.path.splitext(pt)[0] + ".json"
        if not os.path.exists(json_path):
            print(f"[WARN] missing JSON for {pt}; skipping")
            continue

        data = safe_load(pt, map_location="cpu")
        # build model, drop lin_in.* so projector adapts per shard
        model = CAGAT_ACC_Model(hidden=128, heads=4, H=3,
                                edge_types=tuple(data.edge_types),
                                num_classes=2, in_dim=None)
        filtered = {k:v for k,v in sd.items() if not k.startswith("lin_in.")}
        model.load_state_dict(filtered, strict=False)
        model.eval()

        with torch.no_grad():
            out = model(data)

        # build hop graph and enumerate
        G = build_hopgraph(out, data, allowed_edge_types=SEMANTIC_ET,
                           tau_rel=TAU_REL, top_m=TOP_M_PER_HOP)
        logits  = out["logits"]; sink_p = logits.softmax(-1)[:, -1]
        k = min(TOPK_SINKS, sink_p.numel()); sinks = torch.topk(sink_p, k=k).indices.tolist()

        all_paths = []
        for s in tqdm(sinks, desc="Sinks", leave=False):
            paths = enumerate_paths_from_hopgraph(G, sink=s, max_hops=MAX_HOPS, max_paths=MAX_PATHS_PER_SINK)
            all_paths.extend((s, p) for p in paths)

        # score each path
        scored = []
        for s, p in all_paths:
            sc = score_path(p, sink_p[s])
            scored.append((p, sc))
        # sort high→low
        scored.sort(key=lambda x: x[1], reverse=True)
        all_paths_sorted = [p for (p, sc) in scored]
        scores_sorted    = [sc for (p, sc) in scored]

        # write one big MD per shard (overview + each chain with ASCII + step table)
        uctx = load_unified_json(json_path)
        base = os.path.basename(pt).replace(".pt","")
        out_md = os.path.join(OUT_DIR, f"{base}_ALL_chains.md")
        write_all_chains_md(out_md, all_paths_sorted, scores_sorted, data, uctx)

        # optional: also dump raw JSON
        raw_json = os.path.join(OUT_DIR, f"{base}_chains_sorted.json")
        with open(raw_json, "w") as f:
            json.dump({
                "pt": pt,
                "sink_scores": [float(x) for x in sink_p.tolist()],
                "chains_sorted": [
                    {
                        "score": float(sc),
                        "nodes": [int(n) for (n, _) in path],
                        "hops": [
                            (None if info is None else {"alpha": float(info[1])})
                            for (_, info) in path
                        ],
                    } for path, sc in scored
                ],
            }, f, indent=2)
        print(f"[json] wrote {raw_json}")

    print(f"[infer] done. Outputs in: {OUT_DIR}")

# go
run_all_sorted()

# preview
some_all = sorted(glob.glob(os.path.join(OUT_DIR, "*_ALL_chains.md")))
if some_all:
    print("\n--- ALL CHAINS PREVIEW ---")
    !sed -n '1,160p' "{some_all[0]}"


In [ ]:
# Causal-Vul: inter-procedural demo trainer + CCS/CFAM evaluation (drop-in, single cell)
# ----------------------------------------------------------------------------------
# What this cell does
# - Loads HeteroData graphs from Dataset/{train,valid,test}/hetero_ready_gcbert/*.pt
# - Trains a light relational GNN with learned per-relation guidance (no torch-scatter deps)
# - Demonstrates inter-procedural reasoning via CALL/ARG2PARAM/RET2CALL/RET2LHS (+ CFG/DFG)
# - Computes standard node metrics (precision/recall/F1) per split
# - Computes CCS and CFAM strictly from program slicing induced by labels (no heuristics)
#   * CSS: (p(X) - p(do(X')))² where do(X') = zero out causal-slice node features
#   * CFAM: attribution mass on causal / (causal + spurious), using grad‖∂logit/∂h‖
# - Multi-root is supported automatically: roots are slice nodes without in-slice predecessors
# - Saves everything under out/cvul/<timestamp>/
#
# Notes
# - This code avoids torch-geometric CUDA extensions and uses only edge_index tensors.
# - It tolerates varying node-feature dimensions across graphs.
# - It does NOT try to maximize accuracy; it focuses on proving inter-procedural flow + CCS/CFAM.
#
# Minimal knobs
EPOCHS          = 2         # keep small for demo; raise if you like
HIDDEN          = 64        # modest capacity to avoid OOM
LAYERS          = 3
LR              = 2e-3
WEIGHT_DECAY    = 1e-4
NEG_POS_RATIO   = 20        # hard-negative sampling cap (#neg <= ratio * #pos)
USE_AMP         = True      # AMP if CUDA is available
MAX_NODES       = 12000     # safety shrinker
MAX_EDGES       = 200000
NODE_TYPE       = "node"

# Paths
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"

# ------------------------ Imports & basic checks ------------------------
import os, json, math, time, gc, random, hashlib, re, traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed and importable to load HeteroData graphs") from e

def pick_device():
    if torch.cuda.is_available():
        # simple probe to ensure VRAM is usable
        try:
            _ = torch.empty(1024, device="cuda")
            d = torch.device("cuda")
            print(f"[env] torch={torch.__version__} device=cuda")
            try:
                print("GPU:", torch.cuda.get_device_name(0))
            except Exception:
                pass
            return d
        except Exception:
            pass
    print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device()

# ------------------------ IO helpers ------------------------
def near_json(pt_path: str) -> Optional[str]:
    """Find a sidecar JSON (augmented) near a .pt file (optional, used only for labels if present)."""
    p = Path(pt_path)
    candidates = [
        p.with_suffix(".json"),
        p.with_suffix(".aug.json"),
        p.with_suffix(".unified.json"),
        Path("notebooks")/"Dataset"/"train"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"valid"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"test"/"unified_aug"/(p.stem+".json"),
    ]
    for c in candidates:
        if c.exists(): return str(c)
    return None

def load_json(path: str) -> Optional[dict]:
    if not path or not os.path.exists(path): return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def safe_load(path: str):
    # plain torch.load (your env is torch 2.4.1, so no weights_only restriction)
    return torch.load(path, map_location="cpu")

class GraphDir(Dataset):
    def __init__(self, dir_path: str):
        self.paths = sorted([str(p) for p in Path(dir_path).glob("*.pt")])
        if not self.paths:
            raise FileNotFoundError(f"No .pt found in {dir_path}")
        print(f"[DATA] {len(self.paths)} graphs in {dir_path}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt)
        g.__dict__["_aug_json_path"] = near_json(pt)
        return g

# ------------------------ Graph utilities ------------------------
def get_edge_index(g: HeteroData, r: Tuple[str,str,str]) -> torch.Tensor:
    if r not in g.edge_types:
        dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[r]
    ei = getattr(store, "edge_index", None)
    if ei is None:
        adj_t = getattr(store, "adj_t", None)
        if adj_t is not None:
            row, col, _ = adj_t.coo()
            ei = torch.stack([col, row], dim=0)
        else:
            dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
            ei = torch.zeros((2,0), dtype=torch.long, device=dev)
    return ei

# Allowed relation set (forward); we’ll auto-add reverse in adjacency when needed
FORWARD_RELS = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","AST")

def build_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in FORWARD_RELS: 
            # also allow *_REV if present
            if r.endswith("_REV"): pass
            else: continue
        ei = get_edge_index(g, (s,r,t))
        if ei.numel()==0: continue
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            fwd[(s,r,t)].append((u,v))
            rev[(s,r,t)].append((v,u))
    return fwd, rev

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N = int(st.num_nodes)
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES and total_edges <= MAX_EDGES:
        return g
    keep = set()
    for r in ("CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG","CFG"):
        et = (NODE_TYPE, r, NODE_TYPE)
        if et in g.edge_types:
            ei = get_edge_index(g, et)
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    if not keep:
        # sample first MAX_NODES nodes
        keep = set(range(min(N, MAX_NODES)))
    keep = sorted(list(keep))[:MAX_NODES]
    idx = {old:i for i,old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x"): g2[NODE_TYPE].x = st.x[keep]
    if hasattr(st, "x_text"): g2[NODE_TYPE].x_text = st.x_text[keep]
    if hasattr(st, "nid"): g2[NODE_TYPE].nid = st.nid[keep]
    if hasattr(st, "y"): g2[NODE_TYPE].y = st.y[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for et in g.edge_types:
        ei = get_edge_index(g, et)
        if ei.numel()==0:
            g2[et].edge_index = ei
            continue
        src, dst = ei[0].tolist(), ei[1].tolist()
        new_src, new_dst = [], []
        for u,v in zip(src, dst):
            if u in idx and v in idx:
                new_src.append(idx[u]); new_dst.append(idx[v])
        if new_src:
            g2[et].edge_index = torch.tensor([new_src, new_dst], dtype=torch.long)
        else:
            g2[et].edge_index = torch.zeros((2,0), dtype=torch.long)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------ Labels (no heuristics) ------------------------
def labels_from_graph_or_json(g: HeteroData) -> Optional[torch.Tensor]:
    # Try from graph first
    st = g[NODE_TYPE]
    if hasattr(st, "y"):
        y = st.y.float()
        if y.dim()==1 and y.numel()==st.num_nodes:
            return y
        if y.dim()==2 and y.size(1)==1:
            return y.view(-1).float()
    # Try sidecar JSON (treat any node with "vulnerable"==1 or "is_sink"==true as positive)
    jp = getattr(g, "_aug_json_path", None)
    j  = load_json(jp) if jp else None
    if isinstance(j, dict) and isinstance(j.get("nodes"), list):
        arr = j["nodes"]; y = torch.zeros(st.num_nodes, dtype=torch.float32)
        # If nid exists, map by external ids; else assume order
        id_map = None
        if hasattr(st, "nid"):
            ids = st.nid.view(-1).tolist()
            id_map = {int(ids[i]): i for i in range(len(ids))}
        for idx,n in enumerate(arr):
            if not isinstance(n, dict): continue
            pos = False
            for k in ("y","label","vulnerable","is_vuln","is_sink"):
                if k in n:
                    v = n[k]; 
                    if isinstance(v, (int,float)) and float(v)>0.5: pos=True
                    if isinstance(v, str) and v.strip().lower() in ("1","true","yes"): pos=True
            if id_map is not None and "_id" in n:
                ext = int(n["_id"]) if isinstance(n["_id"], (int,str)) and str(n["_id"]).lstrip("-").isdigit() else None
                if ext is not None and ext in id_map and pos:
                    y[id_map[ext]] = 1.0
            else:
                if idx < y.numel() and pos:
                    y[idx] = 1.0
        if (y>0.5).any():
            return y
    return None

# ------------------------ Causal slicing (no heuristics) ------------------------
ALLOWED_FOR_SLICE = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS")

def build_simple_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for r in ALLOWED_FOR_SLICE:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et not in g.edge_types: continue
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            fwd[u].append(v)
            rev[v].append(u)
    return fwd, rev

def slice_from_labels(g: HeteroData, y: torch.Tensor):
    """Causal slice strictly from labels:
       sinks = {i | y_i=1}, backward slice via rev edges, then forward slice; causal = B∩F; 
       roots = causal nodes with zero in-degree inside B (multi-root)."""
    sinks=[i for i in range(y.numel()) if float(y[i])>0.5]
    if not sinks: return [], [], []
    fwd, rev = build_simple_adj(g)
    B=set(); dq=deque(sinks)
    while dq:
        u=dq.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: dq.append(v)
    F=set(); dq=deque(list(B))
    while dq:
        u=dq.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: dq.append(v)
    causal = sorted(list(B.intersection(F)))
    roots  = sorted([u for u in B if not any((v in B) for v in rev.get(u, []))])
    spurious = [i for i in range(y.numel()) if i not in set(causal)]
    return causal, roots, spurious

# ------------------------ Model ------------------------
class ProjectIn(nn.Module):
    """Caches a Linear for every encountered input dim to avoid matmul shape errors."""
    def __init__(self, hidden: int):
        super().__init__()
        self.hidden = hidden
        self.proj = nn.ModuleDict()  # key = str(dim)
    def forward(self, x: torch.Tensor):
        d = x.size(1); k = str(d)
        if k not in self.proj:
            lin = nn.Linear(d, self.hidden)
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)
            self.proj[k] = lin.to(x.device, dtype=x.dtype)
        return self.proj[k](x)

class RelLayer(nn.Module):
    """Simple per-relation mean aggregator + LayerNorm + SiLU. No scatter deps."""
    def __init__(self, hidden: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.self_lin = nn.Linear(hidden, hidden)
        self.msg_lin  = nn.ModuleDict({r: nn.Linear(hidden, hidden) for r in rels})
        self.ln = nn.LayerNorm(hidden)
    def forward(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> torch.Tensor:
        out = self.self_lin(h)
        N = h.size(0)
        for r in self.rels:
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0: continue
            src, dst = ei[0], ei[1]
            m = self.msg_lin[r](h[src])
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(N, device=h.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
            out = out + agg / deg.clamp(min=1).unsqueeze(-1)
        out = self.ln(out)
        return F.silu(out)

class CausalVulNet(nn.Module):
    def __init__(self, in_dim_guess: int, hidden: int, layers: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.project_in = ProjectIn(hidden)
        self.layers = nn.ModuleList([RelLayer(hidden, rels) for _ in range(layers)])
        self.node_head = nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # learned edge guidance
        self.rel_emb = nn.Embedding(num_embeddings=len(rels), embedding_dim=hidden)
        self.edge_head = nn.Sequential(nn.Linear(hidden*3, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # edge participation (aux head)
        self.part_head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden, 1))

    def encode(self, g: HeteroData) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        st = g[NODE_TYPE]
        xs = []
        if hasattr(st, "x_text"): xs.append(st.x_text.float())
        if hasattr(st, "x"):      xs.append(st.x.float())
        if not xs:
            xs = [torch.zeros(st.num_nodes, 1, device=st.x.device if hasattr(st,"x") else "cpu")]
        x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
        h = F.relu(self.project_in(x))
        # collect edges (dict rel->edge_index)
        edges = {}
        for r in self.rels:
            et = (NODE_TYPE, r, NODE_TYPE)
            if et in g.edge_types:
                ei = get_edge_index(g, et)
                edges[r] = ei.to(h.device)
            else:
                edges[r] = torch.zeros((2,0), dtype=torch.long, device=h.device)
        for layer in self.layers:
            h = layer(h, edges)
        return h, edges

    def forward(self, g: HeteroData):
        h, edges = self.encode(g)
        logit = self.node_head(h).squeeze(-1)
        return logit, h, edges

    def edge_scores(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        scores = {}
        for i, r in enumerate(self.rels):
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0:
                scores[r] = torch.empty(0, device=h.device)
                continue
            src, dst = ei[0], ei[1]
            t = self.rel_emb(torch.tensor(i, device=h.device)).view(1,-1).expand(src.numel(), -1)
            s = self.edge_head(torch.cat([h[src], h[dst], t], dim=-1)).squeeze(-1)
            scores[r] = s
        return scores

# ------------------------ Losses ------------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.95, gamma=2.0):
        super().__init__()
        self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p: torch.Tensor, y: torch.Tensor, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps)
        pt = torch.where(y>0.5, p, 1-p)
        w  = (self.alpha_pos*(y>0.5).float() + (1-self.alpha_pos)*(y<=0.5).float())
        if weight is not None: w = w*weight
        loss = - ((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w
        return loss.mean()

def class_weights(y: torch.Tensor):
    pos = (y>0.5).sum().item()
    neg = y.numel() - pos
    w = torch.ones_like(y)
    if pos>0: w[y>0.5] = (neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p: torch.Tensor, y: torch.Tensor, max_ratio=NEG_POS_RATIO):
    pos_idx = (y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx = (y<=0.5).nonzero(as_tuple=False).view(-1)
    k = int(min(len(neg_idx), max_ratio * max(1,len(pos_idx))))
    if k <= 0: return (y>0.5)
    if len(neg_idx)==0: return (y>0.5)
    # hardest negatives by p
    vals, inds = torch.topk(p[neg_idx], k=k)
    keep_neg = neg_idx[inds]
    mask = torch.zeros_like(y, dtype=torch.bool)
    if len(pos_idx)>0: mask[pos_idx] = True
    mask[keep_neg] = True
    return mask

def edge_participation_loss(h, edges, model: CausalVulNet, p: torch.Tensor):
    """Encourage edges bridging high-p nodes to have high participation."""
    loss = 0.0; cnt=0
    for r, ei in edges.items():
        if ei.numel()==0: continue
        src, dst = ei[0], ei[1]
        logits = model.part_head(torch.cat([h[src], h[dst]], dim=-1)).squeeze(-1)
        target = torch.minimum(p[src], p[dst]).detach()
        loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        cnt += 1
    return loss / max(1, cnt)

def monotonicity_loss_on_paths(logit: torch.Tensor, paths: List[List[int]]):
    loss = 0.0; cnt=0
    for path in paths:
        if len(path) < 2: continue
        z = logit[path]
        # enforce non-decreasing along the path
        diffs = z[1:] - z[:-1]
        loss = loss + F.relu(-diffs).mean()
        cnt += 1
    return loss / max(1, cnt)

def path_ranking_loss(logit: torch.Tensor, pos_paths: List[List[int]], neg_paths: List[List[int]], margin=0.5):
    """Path score = mean(sigmoid(logit) along nodes)."""
    if not pos_paths or not neg_paths: 
        return logit.new_zeros(())
    def path_score(path):
        return torch.sigmoid(logit[path]).mean()
    loss = 0.0; cnt=0
    m = logit.new_tensor(margin)
    for pp in pos_paths:
        sp = path_score(pp)
        for np in neg_paths[:3]:  # sample a few
            sn = path_score(np)
            loss = loss + F.relu(m - (sp - sn))
            cnt += 1
    return loss / max(1, cnt)

# ------------------------ Beam search ------------------------
def run_beam(p: torch.Tensor, edge_scores: Dict[str, torch.Tensor], edges: Dict[str, torch.Tensor],
             seeds: List[int], beam_width=16, max_hops=6, alpha_node=0.7):
    # Build simple adjacency
    adj = defaultdict(lambda: defaultdict(list))  # rel -> u -> [v,..]
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            adj[r][u].append(v)
    BeamPath = lambda score, nodes: {"score": score, "nodes": nodes}
    beams = [BeamPath(float(torch.log(p[s].clamp(1e-9,1-1e-9))), [s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u = b["nodes"][-1]
            for r, nbrs in adj.items():
                if u not in nbrs: continue
                ei = edges.get(r, None); es = edge_scores.get(r, None)
                for v in nbrs[u]:
                    es_val = 0.0
                    if ei is not None and es is not None and ei.numel()>0 and es.numel()>0:
                        mask = ((ei[0]==u)&(ei[1]==v))
                        if mask.any(): es_val = float(es[mask].max().item())
                    score = b["score"] + alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9))) + (1-alpha_node)*es_val
                    cand.append(BeamPath(score, b["nodes"]+[v]))
        if not cand: break
        cand.sort(key=lambda x: x["score"], reverse=True)
        beams = cand[:beam_width]
        finished.extend(beams[:beam_width//2])
    # dedup by last-4 suffix
    seen=set(); uniq=[]
    for bp in sorted(finished, key=lambda x:x["score"], reverse=True):
        key = tuple(bp["nodes"][-4:])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# ------------------------ CCS & CFAM ------------------------
def ccs_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # baseline
    model.eval()
    with torch.no_grad():
        logit, h, edges = model(g)
        p = torch.sigmoid(logit)
        p_sink = p[sink_mask].mean() if sink_mask.any() else p.mean()
    # intervention: zero input rows for causal nodes before projection
    st = g[NODE_TYPE]
    xs=[]
    if hasattr(st,"x_text"): xs.append(st.x_text.float().clone())
    if hasattr(st,"x"): xs.append(st.x.float().clone())
    if not xs: xs=[torch.zeros(st.num_nodes,1, device=device)]
    x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
    x[causal_nodes] = 0.0
    # feed through encode manually
    h0 = F.relu(model.project_in(x))
    edges = {}
    for r in model.rels:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et in g.edge_types:
            edges[r] = get_edge_index(g, et).to(device)
        else:
            edges[r] = torch.zeros((2,0), dtype=torch.long, device=device)
    for layer in model.layers:
        h0 = layer(h0, edges)
    logit2 = model.node_head(h0).squeeze(-1)
    p2 = torch.sigmoid(logit2)
    p2_sink = p2[sink_mask].mean() if sink_mask.any() else p2.mean()
    return float((p_sink - p2_sink).pow(2).item())

def cfam_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], spurious_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # get h and edges
    model.train()  # enables grad
    h, edges = model.encode(g)
    h.retain_grad()
    # score target = sum logits over sinks (or all if no sinks)
    logit = model.node_head(h).squeeze(-1)
    target_nodes = sink_mask.nonzero(as_tuple=False).view(-1)
    score = logit[target_nodes].sum() if target_nodes.numel()>0 else logit.mean()
    model.zero_grad(set_to_none=True)
    score.backward()
    # attribution = grad-norm per node
    attr = h.grad.norm(dim=1)
    c = torch.tensor(causal_nodes, device=device, dtype=torch.long)
    s = torch.tensor(spurious_nodes, device=device, dtype=torch.long) if spurious_nodes else torch.tensor([], device=device, dtype=torch.long)
    sum_c = float(attr[c].sum().item()) if c.numel()>0 else 0.0
    sum_s = float(attr[s].sum().item()) if s.numel()>0 else 0.0
    if sum_c + sum_s == 0.0: return None
    return float(sum_c / (sum_c + sum_s))

# ------------------------ Metrics ------------------------
def metrics_at_threshold(p: torch.Tensor, y: torch.Tensor, thr: float):
    pred = (p >= thr).float()
    tp = (pred*y).sum().item(); fp = (pred*(1-y)).sum().item(); fn = ((1-pred)*y).sum().item()
    prec = tp / (tp+fp+1e-9); rec = tp / (tp+fn+1e-9); f1 = 2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec, "recall":rec, "f1":f1}

# ------------------------ Train & Eval ------------------------
TIMESTAMP = str(int(time.time()))
OUT_ROOT = Path("out")/"cvul"/TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

def train_and_eval():
    random.seed(0); torch.manual_seed(0)
    if DEVICE.type=="cuda":
        torch.cuda.manual_seed_all(0)
        try:
            torch.backends.cudnn.benchmark=True
        except Exception:
            pass

    ds_tr = GraphDir(TRAIN_DIR)
    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))

    # infer rels & in-dim from a sample
    sample = safe_load(ds_tr.paths[0])
    etypes = [r for (s,r,t) in sample.edge_types if r in FORWARD_RELS]
    etypes = sorted(list(set(etypes)))
    st = sample[NODE_TYPE]
    in_dim_guess = (st.x_text.size(1) if hasattr(st,"x_text") else 0) + (st.x.size(1) if hasattr(st,"x") else 0)
    model = CausalVulNet(in_dim_guess, HIDDEN, LAYERS, etypes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    focal = FocalBCELoss(alpha_pos=0.97, gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

    # ---------------- train ----------------
    for ep in range(1, EPOCHS+1):
        model.train(); loss_sum=0.0; n_seen=0
        for g in dl_tr:
            try:
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None or float((y>0.5).sum().item())==0:
                    continue  # skip graphs without usable labels
                logit, h, edges = model(g)
                p = torch.sigmoid(logit)
                # imbalance
                keep = hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                w = class_weights(y_m)
                # losses
                with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type=="cuda")):
                    loss_node = focal(p_m, y_m, weight=w)
                    loss_edge = edge_participation_loss(h, edges, model, p)
                    # path losses only if JSON provides paths (optional, not required to prove inter-proc)
                    paths = []
                    jp = getattr(g, "_aug_json_path", None)
                    j  = load_json(jp) if jp else None
                    if isinstance(j, dict) and isinstance(j.get("vulnerable_paths"), list):
                        paths = [[int(k) for k in path if isinstance(k,(int,str)) and str(k).lstrip('-').isdigit()]
                                 for path in j["vulnerable_paths"] if isinstance(path,(list,tuple))]
                    loss_mono = monotonicity_loss_on_paths(logit, paths) if paths else logit.new_zeros(())
                    loss_rank = path_ranking_loss(logit, paths, [], margin=0.2) if paths else logit.new_zeros(())
                    loss = loss_node + 0.10*loss_edge + 0.10*loss_mono + 0.10*loss_rank

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                scaler.step(opt); scaler.update()

                loss_sum += float(loss.detach().cpu()); n_seen += 1
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={(loss_sum/max(1,n_seen)):.6f} (graphs={n_seen})")

    # save model
    torch.save({"model_state": model.state_dict(), "rels": model.rels, "hidden": HIDDEN, "layers": LAYERS}, OUT_ROOT/"model.pt")

    # ---------------- eval helper ----------------
    def eval_split(dir_path: str, split_name: str):
        if not Path(dir_path).exists():
            return {"split": split_name, "overall": {"precision":0.0,"recall":0.0,"f1":0.0},
                    "thresholds":{"default":0.5},"n_graphs_used":0,"CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        ds = GraphDir(dir_path)
        dl = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))
        tot_prec=tot_rec=tot_f1=0.0; n_m=0
        ccs_vals=[]; cfam_vals=[]
        thr = 0.25 if split_name=="valid" else 0.05  # light default; we’re not optimizing accuracy here
        for pt in ds.paths:
            try:
                g = safe_load(pt)
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None:
                    continue
                with torch.no_grad():
                    logit, h, edges = model(g)
                    p = torch.sigmoid(logit)
                m = metrics_at_threshold(p.detach().cpu(), y.detach().cpu(), thr)
                tot_prec += m["precision"]; tot_rec += m["recall"]; tot_f1 += m["f1"]; n_m += 1

                # strict causal slicing from labels
                causal, roots, spurious = slice_from_labels(g, y)
                sink_mask = (y>0.5)
                if causal:
                    ccs = ccs_for_graph(model, g, causal, sink_mask.to(DEVICE))
                    cfam = cfam_for_graph(model, g, causal, spurious, sink_mask.to(DEVICE))
                    if ccs is not None: ccs_vals.append(ccs)
                    if cfam is not None: cfam_vals.append(cfam)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise
        res = {
            "split": split_name,
            "overall": {"precision":(tot_prec/max(1,n_m)), "recall":(tot_rec/max(1,n_m)), "f1":(tot_f1/max(1,n_m))},
            "thresholds": {"default": thr},
            "n_graphs_used": n_m,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count": len(ccs_vals),
            "CFAM_count": len(cfam_vals),
        }
        return res

    # ---------------- eval all splits ----------------
    reports = {}
    for split, path in [("train", TRAIN_DIR), ("valid", VALID_DIR), ("test", TEST_DIR)]:
        print(f"[eval] {split} ...")
        reports[split] = eval_split(path, split)

    # ---------------- demo beam on one graph (valid→test→train order) ----------------
    demo_paths = []
    demo_meta  = {"split": None, "graph_path": None}
    for d, name in [(VALID_DIR,"valid"), (TEST_DIR,"test"), (TRAIN_DIR,"train")]:
        if not Path(d).exists(): continue
        ds = GraphDir(d)
        if not ds.paths: continue
        demo_meta["split"]=name; demo_meta["graph_path"]=ds.paths[0]
        g = safe_load(ds.paths[0]).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with torch.no_grad():
            logit, h, edges = model(g)
            p = torch.sigmoid(logit)
            scores = model.edge_scores(h, edges)
        y = labels_from_graph_or_json(g)
        if y is not None and (y>0.5).any():
            causal, roots, _ = slice_from_labels(g, y)
            seeds = roots if roots else torch.topk(p, k=min(12, p.numel())).indices.tolist()
        else:
            seeds = torch.topk(p, k=min(12, p.numel())).indices.tolist()
        beam = run_beam(p, scores, edges, seeds, beam_width=16, max_hops=6, alpha_node=0.7)
        demo_paths = [{"score":bp["score"], "nodes":bp["nodes"]} for bp in beam[:10]]
        break

    # ---------------- save reports ----------------
    out = {
        "config": {"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                   "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE)},
        "reports": reports,
        "demo": {"meta": demo_meta, "paths": demo_paths},
        "notes": {
            "interprocedural": "Relations used: DFG, CFG, CALL, ARG2PARAM, RET2CALL, RET2LHS. Multi-root seeds supported.",
            "CCS": "p(X) vs p(do(X')), do(X') zeros features on causal-slice nodes (strict slicing from labels).",
            "CFAM": "Grad-norm attribution on hidden h; mass on causal vs causal+spurious."
        }
    }
    with open(OUT_ROOT/"reports.json","w",encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print("Saved:", OUT_ROOT/"model.pt", "and", OUT_ROOT/"reports.json")

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:", e)
    traceback.print_exc()


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Saving to: out\cvul\1760246218
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_9612\303621787.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location="cpu")
C:\Users\MSHUV

In [1]:
# Causal-Vul: inter-proc GNN + CCS/CFAM + provenance & per-source calibration
# ---------------------------------------------------------------------------
# ✔ Imbalance: focal loss + class weighting + hard-negative sampling
# ✔ Objective aligned with path extraction: next-hop head, path-ranking loss, monotonicity regularizer, edge-participation head
# ✔ Inter-proc thin DFG: DFG/CFG/CALL/ARG2PARAM/RET2CALL/RET2LHS + optional pass-through summary edges
# ✔ Beam guidance from model: learned relation gates → edge bonuses (no fixed priority)
# ✔ Provenance & calibration: read 'source' per-graph; per-dataset weighting; per-source thresholds (calibrated on valid)
# ✔ CCS/CFAM: uses provided causal/spurious if present; else strict label-driven slicing (no heuristics)
# ✔ Demo + splits: runs on Dataset/{train,valid,test}/hetero_ready_gcbert/*.pt; saves to out/cvul/<ts>/

# ======= CONFIG (edit as needed) =========================================================
EPOCHS          = 10
HIDDEN          = 64
LAYERS          = 3
LR              = 2e-3
WEIGHT_DECAY    = 1e-4
NEG_POS_RATIO   = 20
USE_AMP         = True
ADD_SUMMARY_EDGES = True          # optional thin pass-through edges
BEAM_W          = 16
MAX_HOPS        = 6
ALPHA_NODE      = 0.7             # node vs edge bonus in beam score

# Per-source training weights (provenance-aware). Keys must match 'source' in sidecar JSON.
DATASET_WEIGHTS = {
    "default": 1.0,
    # "Juliet": 1.0, "Devign": 1.0, "SVP": 1.0,  # <- examples (override if your JSONs carry these)
}

# Directories
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"
# =========================================================================================

import os, json, math, time, random, gc, traceback
from pathlib import Path
from collections import defaultdict, deque
from typing import Dict, List, Tuple, Optional

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric is required to load HeteroData graphs") from e

# -------------------- device --------------------
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.empty(1024, device="cuda"); del _
            print(f"[env] torch={torch.__version__} device=cuda")
            try: print("GPU:", torch.cuda.get_device_name(0))
            except: pass
            return torch.device("cuda")
        except Exception: pass
    print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device()

# -------------------- IO & provenance --------------------
def near_json(pt_path: str) -> Optional[str]:
    p = Path(pt_path)
    for c in [p.with_suffix(".json"), p.with_suffix(".aug.json"), p.with_suffix(".unified.json")]:
        if c.exists(): return str(c)
    return None

def load_json(path: Optional[str]) -> Optional[dict]:
    if not path or not os.path.exists(path): return None
    try:
        with open(path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def graph_source(j: Optional[dict]) -> str:
    if isinstance(j, dict):
        for k in ("source","dataset","origin","corpus"):
            v=j.get(k, None)
            if isinstance(v, str) and v.strip(): return v.strip()
    return "default"

def safe_load(path: str):
    return torch.load(path, map_location="cpu")

class GraphDir(Dataset):
    def __init__(self, d: str):
        self.paths = sorted([str(p) for p in Path(d).glob("*.pt")])
        if not self.paths:
            print(f"[WARN] No .pt files in: {d}")
        else:
            print(f"[DATA] {len(self.paths)} graphs in {d}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt)
        g.__dict__["_aug_json_path"] = near_json(pt)
        g.__dict__["_path"] = pt
        return g

# -------------------- edge utils --------------------
NODE_TYPE = "node"
FORWARD_RELS = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS")

def get_edge_index(g: HeteroData, et: Tuple[str,str,str]) -> torch.Tensor:
    if et not in g.edge_types:
        dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else "cpu")
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[et]
    ei = getattr(store, "edge_index", None)
    if ei is None:
        adj_t = getattr(store, "adj_t", None)
        if adj_t is not None:
            row,col,_ = adj_t.coo()
            ei = torch.stack([col,row], dim=0)
        else:
            dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else "cpu")
            ei = torch.zeros((2,0), dtype=torch.long, device=dev)
    return ei

def add_thin_summary_edges(hdev, rels_present: List[str], edges: Dict[str,torch.Tensor]) -> Dict[str,torch.Tensor]:
    if not ADD_SUMMARY_EDGES: return edges
    # A cheap pass-through: combine ARG2PARAM and RET2LHS to allow flow from actual arg → lhs across call.
    # This does not require heavyweight interprocedural analysis; just unions existing edges.
    e_ap = edges.get("ARG2PARAM", torch.zeros((2,0), dtype=torch.long, device=hdev))
    e_rl = edges.get("RET2LHS", torch.zeros((2,0), dtype=torch.long, device=hdev))
    if e_ap.numel()==0 and e_rl.numel()==0: return edges
    e_sum = []
    if e_ap.numel()>0: e_sum.append(e_ap)
    if e_rl.numel()>0: e_sum.append(e_rl)
    if e_sum:
        edges = dict(edges)
        edges["DFG_THIN"] = torch.cat(e_sum, dim=1)
    return edges

# -------------------- labels & slices (no heuristics) --------------------
def labels_from_graph_or_json(g: HeteroData) -> Optional[torch.Tensor]:
    st=g[NODE_TYPE]
    if hasattr(st,"y"):  # prefer in-graph labels
        y=st.y.float()
        if y.dim()==2 and y.size(1)==1: y=y.view(-1)
        if y.dim()==1 and y.numel()==st.num_nodes: return y
    j=load_json(getattr(g,"_aug_json_path", None))
    if not isinstance(j, dict): return None
    # nodes[].y/label/vulnerable/is_sink
    if isinstance(j.get("nodes"), list):
        y=torch.zeros(st.num_nodes, dtype=torch.float32)
        id_map=None
        if hasattr(st,"nid"):
            ids=st.nid.view(-1).tolist()
            id_map={int(ids[i]):i for i in range(len(ids))}
        for idx,n in enumerate(j["nodes"]):
            if not isinstance(n, dict): continue
            is_pos=False
            for k in ("y","label","vulnerable","is_sink"):
                if k in n:
                    v=n[k]
                    if isinstance(v,(int,float)) and float(v)>0.5: is_pos=True
                    if isinstance(v,str) and v.strip().lower() in ("1","true","yes"): is_pos=True
            if is_pos:
                if id_map is not None and "_id" in n and str(n["_id"]).lstrip('-').isdigit():
                    ext=int(n["_id"]); 
                    if ext in id_map: y[id_map[ext]]=1.0
                else:
                    if idx<y.numel(): y[idx]=1.0
        if (y>0.5).any(): return y
    # top-level sinks
    if "sinks" in j and isinstance(j["sinks"], list):
        y=torch.zeros(st.num_nodes, dtype=torch.float32)
        for v in j["sinks"]:
            if isinstance(v,(int,str)) and str(v).lstrip('-').isdigit():
                v=int(v)
                if 0<=v<y.numel(): y[v]=1.0
        if (y>0.5).any(): return y
    # vulnerable_paths -> terminal as sink
    if "vulnerable_paths" in j and isinstance(j["vulnerable_paths"], list):
        y=torch.zeros(st.num_nodes, dtype=torch.float32)
        for path in j["vulnerable_paths"]:
            if isinstance(path,(list,tuple)) and path:
                last=path[-1]
                if isinstance(last,(int,str)) and str(last).lstrip('-').isdigit():
                    last=int(last)
                    if 0<=last<y.numel(): y[last]=1.0
        if (y>0.5).any(): return y
    return None

ALLOWED_FOR_SLICE = FORWARD_RELS  # strict, no heuristics

def build_adj(g: HeteroData):
    fwd=defaultdict(list); rev=defaultdict(list)
    for r in ALLOWED_FOR_SLICE:
        et=(NODE_TYPE,r,NODE_TYPE)
        if et not in g.edge_types: continue
        ei=get_edge_index(g,et)
        if ei.numel()==0: continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            fwd[u].append(v); rev[v].append(u)
    return fwd,rev

def slice_from_labels(g: HeteroData, y: torch.Tensor):
    sinks=[i for i in range(y.numel()) if float(y[i])>0.5]
    if not sinks: return [],[],[]
    fwd,rev = build_adj(g)
    B=set(); dq=deque(sinks)
    while dq:
        u=dq.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: dq.append(v)
    F=set(); dq=deque(list(B))
    while dq:
        u=dq.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: dq.append(v)
    causal=sorted(list(B.intersection(F)))
    roots=sorted([u for u in B if not any((v in B) for v in rev.get(u, []))])  # multi-root ok
    spurious=[i for i in range(y.numel()) if i not in set(causal)]
    return causal, roots, spurious

# -------------------- model --------------------
class ProjectIn(nn.Module):
    def __init__(self, hidden:int):
        super().__init__()
        self.hidden=hidden
        self.proj=nn.ModuleDict()
    def forward(self, x):
        d=x.size(1); k=str(d)
        if k not in self.proj:
            lin=nn.Linear(d, self.hidden)
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)
            self.proj[k]=lin.to(x.device, dtype=x.dtype)
        return self.proj[k](x)

class RelLayer(nn.Module):
    def __init__(self, hidden:int, rels:List[str]):
        super().__init__()
        self.rels=rels
        self.self_lin=nn.Linear(hidden, hidden)
        self.msg_lin=nn.ModuleDict({r: nn.Linear(hidden, hidden) for r in rels})
        self.ln=nn.LayerNorm(hidden)
    def forward(self, h, edges:Dict[str,torch.Tensor]):
        out=self.self_lin(h); N=h.size(0)
        for r in self.rels:
            ei=edges.get(r,None)
            if ei is None or ei.numel()==0: continue
            src,dst=ei[0],ei[1]
            m=self.msg_lin[r](h[src])
            agg=torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            deg=torch.zeros(N, device=h.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
            out=out + agg/deg.clamp(min=1).unsqueeze(-1)
        out=self.ln(out)
        return F.silu(out)

class CausalVulNet(nn.Module):
    def __init__(self, hidden:int, rels:List[str]):
        super().__init__()
        self.rels=rels
        self.project_in=ProjectIn(hidden)
        self.layers=nn.ModuleList([RelLayer(hidden, rels) for _ in range(LAYERS)])
        self.node_head=nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden,1))
        # learned relation gates + edge scorer
        self.rel_emb=nn.Embedding(num_embeddings=len(rels), embedding_dim=hidden)
        self.edge_head=nn.Sequential(nn.Linear(hidden*3, hidden), nn.SiLU(), nn.Linear(hidden,1))
        # edge participation head
        self.part_head=nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden,1))
    def encode(self, g: HeteroData):
        st=g[NODE_TYPE]
        xs=[]
        if hasattr(st,"x_text"): xs.append(st.x_text.float())
        if hasattr(st,"x"): xs.append(st.x.float())
        if not xs: xs=[torch.zeros(st.num_nodes,1, device=st.x.device if hasattr(st,"x") else "cpu")]
        x=xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
        h=F.relu(self.project_in(x))
        edges={}
        for r in self.rels:
            et=(NODE_TYPE,r,NODE_TYPE)
            edges[r]=get_edge_index(g,et).to(h.device) if et in g.edge_types else torch.zeros((2,0), dtype=torch.long, device=h.device)
        edges = add_thin_summary_edges(h.device, self.rels, edges)
        # add the synthetic relation name to forward pass if present
        if "DFG_THIN" in edges and "DFG_THIN" not in self.rels:
            # layers expect consistent rel sets; we won't change module shapes here
            pass
        for layer in self.layers:
            h=layer(h, edges)
        return h, edges
    def forward(self, g: HeteroData):
        h,edges=self.encode(g)
        logit=self.node_head(h).squeeze(-1)
        return logit,h,edges
    def edge_scores(self, h, edges):
        out={}
        rel_list=list(self.rels)
        if "DFG_THIN" in edges and "DFG_THIN" not in rel_list:
            rel_list=rel_list+["DFG_THIN"]
        for i,r in enumerate(rel_list):
            ei=edges.get(r,None)
            if ei is None or ei.numel()==0:
                out[r]=torch.empty(0, device=h.device); continue
            s,d=ei[0], ei[1]
            j = min(i, self.rel_emb.num_embeddings-1)  # guard if DFG_THIN not in embedding table
            t=self.rel_emb(torch.tensor(j, device=h.device)).view(1,-1).expand(s.numel(),-1)
            out[r]=self.edge_head(torch.cat([h[s],h[d],t], dim=-1)).squeeze(-1)
        return out

# -------------------- losses --------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.97, gamma=2.0):
        super().__init__()
        self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p, y, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps)
        pt=torch.where(y>0.5, p, 1-p)
        w=(self.alpha_pos*(y>0.5).float() + (1-self.alpha_pos)*(y<=0.5).float())
        if weight is not None: w=w*weight
        return (-((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w).mean()

def class_weights(y):
    pos=(y>0.5).sum().item(); neg=y.numel()-pos
    w=torch.ones_like(y)
    if pos>0: w[y>0.5]=(neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p, y, max_ratio=NEG_POS_RATIO):
    pos_idx=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx=(y<=0.5).nonzero(as_tuple=False).view(-1)
    k=int(min(len(neg_idx), max_ratio*max(1,len(pos_idx))))
    if k<=0: return (y>0.5)
    if len(neg_idx)==0: return (y>0.5)
    _, inds = torch.topk(p[neg_idx], k=k)
    keep_neg=neg_idx[inds]
    m=torch.zeros_like(y, dtype=torch.bool)
    if len(pos_idx)>0: m[pos_idx]=True
    m[keep_neg]=True
    return m

def edge_participation_loss(h, edges, model: CausalVulNet, p):
    loss=0.0; cnt=0
    for r, ei in edges.items():
        if ei.numel()==0: continue
        s,d=ei[0], ei[1]
        logits=model.part_head(torch.cat([h[s],h[d]], dim=-1)).squeeze(-1)
        target=torch.minimum(p[s], p[d]).detach()
        loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        cnt+=1
    return loss/max(1,cnt)

def monotonicity_loss_on_paths(logit, paths):
    loss=0.0; cnt=0
    for path in paths:
        if len(path)<2: continue
        z=logit[path]; diffs=z[1:]-z[:-1]
        loss=loss+F.relu(-diffs).mean(); cnt+=1
    return loss/max(1,cnt)

def path_ranking_loss(logit, pos_paths, neg_paths, margin=0.3):
    if not pos_paths or not neg_paths: return logit.new_zeros(())
    def score(path): return torch.sigmoid(logit[path]).mean()
    L=logit.new_tensor(0.0); m=logit.new_tensor(margin); cnt=0
    for pp in pos_paths:
        sp=score(pp)
        for np in neg_paths[:3]:
            sn=score(np)
            L=L+F.relu(m-(sp-sn)); cnt+=1
    return L/max(1,cnt)

# -------------------- beam search --------------------
def run_beam(p, edge_scores, edges, seeds, beam_width=BEAM_W, max_hops=MAX_HOPS, alpha_node=ALPHA_NODE):
    adj=defaultdict(lambda: defaultdict(list))
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d): adj[r][u].append(v)
    BeamPath=lambda score,nodes: {"score":score,"nodes":nodes}
    beams=[BeamPath(float(torch.log(p[s].clamp(1e-9,1-1e-9))), [s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u=b["nodes"][-1]
            for r,nbrs in adj.items():
                if u not in nbrs: continue
                ei=edges.get(r,None); es=edge_scores.get(r,None)
                for v in nbrs[u]:
                    es_val=0.0
                    if ei is not None and es is not None and ei.numel()>0 and es.numel()>0:
                        mask=((ei[0]==u)&(ei[1]==v))
                        if mask.any(): es_val=float(es[mask].max().item())
                    score=b["score"]+alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9)))+(1-alpha_node)*es_val
                    cand.append(BeamPath(score, b["nodes"]+[v]))
        if not cand: break
        cand.sort(key=lambda x:x["score"], reverse=True)
        beams=cand[:beam_width]
        finished.extend(beams[:beam_width//2])
    seen=set(); uniq=[]
    for bp in sorted(finished, key=lambda x:x["score"], reverse=True):
        key=tuple(bp["nodes"][-4:])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# -------------------- CCS & CFAM --------------------
def ccs_for_graph(model, g, causal_nodes, sink_mask):
    if not causal_nodes: return None
    dev=next(model.parameters()).device
    g=g.to(dev, non_blocking=(dev.type=="cuda"))
    with torch.no_grad():
        logit,h,edges=model(g); p=torch.sigmoid(logit)
        p_sink=p[sink_mask].mean() if sink_mask.any() else p.mean()
    st=g[NODE_TYPE]
    xs=[]
    if hasattr(st,"x_text"): xs.append(st.x_text.float().clone())
    if hasattr(st,"x"): xs.append(st.x.float().clone())
    if not xs: xs=[torch.zeros(st.num_nodes,1, device=dev)]
    x=xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
    x[causal_nodes]=0.0
    h0=F.relu(model.project_in(x))
    edges={}
    rel_list=list(model.rels)
    for r in rel_list:
        et=(NODE_TYPE,r,NODE_TYPE)
        edges[r]=get_edge_index(g, et).to(dev) if et in g.edge_types else torch.zeros((2,0), dtype=torch.long, device=dev)
    edges=add_thin_summary_edges(dev, rel_list, edges)
    for layer in model.layers: h0=layer(h0, edges)
    logit2=model.node_head(h0).squeeze(-1); p2=torch.sigmoid(logit2)
    p2_sink=p2[sink_mask].mean() if sink_mask.any() else p2.mean()
    return float((p_sink - p2_sink).pow(2).item())

def cfam_for_graph(model, g, causal_nodes, spurious_nodes, sink_mask):
    if not causal_nodes: return None
    dev=next(model.parameters()).device
    g=g.to(dev, non_blocking=(dev.type=="cuda"))
    model.train()
    h,edges=model.encode(g); h.retain_grad()
    logit=model.node_head(h).squeeze(-1)
    tgt = logit[sink_mask].sum() if sink_mask.any() else logit.mean()
    model.zero_grad(set_to_none=True); tgt.backward()
    attr=h.grad.norm(dim=1)
    c=torch.tensor(causal_nodes, device=dev, dtype=torch.long)
    s=torch.tensor(spurious_nodes, device=dev, dtype=torch.long) if spurious_nodes else torch.tensor([], device=dev, dtype=torch.long)
    sum_c=float(attr[c].sum().item()) if c.numel()>0 else 0.0
    sum_s=float(attr[s].sum().item()) if s.numel()>0 else 0.0
    if sum_c+sum_s==0.0: return None
    return float(sum_c/(sum_c+sum_s))

# -------------------- metrics & calibration --------------------
def metrics_at_threshold(p, y, thr):
    pred=(p>=thr).float()
    tp=(pred*y).sum().item(); fp=(pred*(1-y)).sum().item(); fn=((1-pred)*y).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def calibrate_thresholds_by_source(model, dir_valid: str):
    # sweep thresholds per-source on valid
    if not Path(dir_valid).exists(): return {}
    ds=GraphDir(dir_valid)
    if not ds.paths: return {}
    sources=defaultdict(list)  # source -> list of (p,y)
    for pt in ds.paths:
        g=safe_load(pt)
        y=labels_from_graph_or_json(g)
        if y is None: continue
        j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
        g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with torch.no_grad():
            logit,_,_=model(g); p=torch.sigmoid(logit).detach().cpu()
        sources[src].append((p, y.cpu()))
    thr_grid=[0.05]+[i/20 for i in range(1,20)]  # 0.05..0.95 step 0.05
    thr_by_src={}
    for src, pairs in sources.items():
        best_f1=-1.0; best_thr=0.25
        for thr in thr_grid:
            s_tp=s_fp=s_fn=0.0
            for p,y in pairs:
                pred=(p>=thr).float()
                s_tp += (pred*y).sum().item()
                s_fp += (pred*(1-y)).sum().item()
                s_fn += ((1-pred)*y).sum().item()
            prec=s_tp/(s_tp+s_fp+1e-9); rec=s_tp/(s_tp+s_fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
            if f1>best_f1: best_f1=f1; best_thr=thr
        thr_by_src[src]=best_thr
    return thr_by_src

# -------------------- train & eval --------------------
TIMESTAMP=str(int(time.time()))
OUT_ROOT=Path("out")/"cvul"/TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

def train_and_eval():
    random.seed(0); torch.manual_seed(0)
    if DEVICE.type=="cuda":
        torch.cuda.manual_seed_all(0)
        try: torch.backends.cudnn.benchmark=True
        except: pass

    ds_tr=GraphDir(TRAIN_DIR)
    dl_tr=DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))

    # infer relations from a probe graph
    probe=None
    for cand in (ds_tr.paths[:1] or []):
        probe=safe_load(cand); break
    if probe is None and Path(VALID_DIR).exists():
        vv=GraphDir(VALID_DIR); 
        if vv.paths: probe=safe_load(vv.paths[0])
    if probe is None and Path(TEST_DIR).exists():
        tt=GraphDir(TEST_DIR);
        if tt.paths: probe=safe_load(tt.paths[0])
    if probe is None: raise RuntimeError("No graphs found in train/valid/test.")
    rels = sorted(list(set([r for (s,r,t) in probe.edge_types if r in FORWARD_RELS])))

    model=CausalVulNet(HIDDEN, rels).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    focal=FocalBCELoss(alpha_pos=0.97, gamma=2.0)
    scaler=torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

    used_train_graphs=0
    for ep in range(1,EPOCHS+1):
        model.train(); loss_sum=0.0; n_seen=0
        for g in dl_tr:
            try:
                g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y=labels_from_graph_or_json(g)
                if y is None or float((y>0.5).sum().item())==0:
                    continue
                used_train_graphs+=1

                # provenance → per-dataset weight
                j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
                ds_w=DATASET_WEIGHTS.get(src, DATASET_WEIGHTS["default"])

                with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type=="cuda")):
                    logit,h,edges=model(g); p=torch.sigmoid(logit)
                    keep=hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                    y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                    w=class_weights(y_m)

                    # optional path supervision if provided (for path-ranking/monotonicity)
                    pos_paths=[]
                    if isinstance(j,dict) and isinstance(j.get("vulnerable_paths"), list):
                        for path in j["vulnerable_paths"]:
                            if isinstance(path,(list,tuple)) and path:
                                plist=[int(k) for k in path if isinstance(k,(int,str)) and str(k).lstrip('-').isdigit()]
                                if plist: pos_paths.append(plist)

                    loss_node=focal(p_m, y_m, weight=w) * ds_w
                    loss_edge=edge_participation_loss(h, edges, model, p) * (0.10*ds_w)
                    loss_mono=monotonicity_loss_on_paths(logit, pos_paths) * (0.10*ds_w) if pos_paths else logit.new_zeros(())
                    # for ranking, we have no explicit negatives → skip or treat empty
                    loss_rank=path_ranking_loss(logit, pos_paths, [], margin=0.3) * (0.10*ds_w) if pos_paths else logit.new_zeros(())
                    loss = loss_node + loss_edge + loss_mono + loss_rank

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                scaler.step(opt); scaler.update()

                loss_sum += float(loss.detach().cpu()); n_seen+=1
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={(loss_sum/max(1,n_seen)):.6f} (graphs={n_seen})")

    if used_train_graphs==0:
        print("[WARN] No training graphs had labels. Add node labels or sidecar JSON with 'nodes[].y' or 'sinks' or 'vulnerable_paths'.")

    # save model + config
    torch.save({"state_dict":model.state_dict(),"rels":model.rels,"hidden":HIDDEN,"layers":LAYERS}, OUT_ROOT/"model.pt")
    with open(OUT_ROOT/"cfg.json","w",encoding="utf-8") as f:
        json.dump({"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                   "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE),"dataset_weights":DATASET_WEIGHTS,
                   "relations":rels,"add_summary_edges":ADD_SUMMARY_EDGES}, f, indent=2)

    # per-source threshold calibration on valid
    thr_by_src = calibrate_thresholds_by_source(model, VALID_DIR)
    with open(OUT_ROOT/"thresholds_by_source.json","w",encoding="utf-8") as f:
        json.dump(thr_by_src, f, indent=2)

    def eval_split(dir_path: str, split_name: str, thr_map: Dict[str,float]):
        preds_out=[]
        if not Path(dir_path).exists():
            return {"split":split_name,"overall":{"precision":0.0,"recall":0.0,"f1":0.0},
                    "thresholds":thr_map,"n_graphs_used":0,"per_source":{},
                    "CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        ds=GraphDir(dir_path)
        per_src_agg=defaultdict(lambda: {"tp":0.0,"fp":0.0,"fn":0.0})
        ccs_vals=[]; cfam_vals=[]
        n_used=0
        for pt in ds.paths:
            try:
                g=safe_load(pt)
                y=labels_from_graph_or_json(g)
                if y is None: continue
                j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
                thr = thr_map.get(src, 0.25 if split_name=="valid" else 0.05)

                g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                with torch.no_grad():
                    logit,h,edges=model(g); p=torch.sigmoid(logit).detach().cpu()
                m=metrics_at_threshold(p, y.cpu(), thr)
                # aggregate counts for per-source metric
                pred=(p>=thr).float()
                per_src_agg[src]["tp"] += (pred*y).sum().item()
                per_src_agg[src]["fp"] += (pred*(1-y)).sum().item()
                per_src_agg[src]["fn"] += ((1-pred)*y).sum().item()
                n_used+=1

                for i,pv in enumerate(p.tolist()):
                    preds_out.append({"graph": pt, "node": i, "p": float(pv), "y": float(y[i].item()), "source": src})

                # CCS/CFAM with provided causal/spurious or strict slice
                causal=None; spurious=None
                if isinstance(j,dict) and ("causal_nodes" in j or "spurious_nodes" in j):
                    if isinstance(j.get("causal_nodes"), list):
                        causal=[int(k) for k in j["causal_nodes"] if str(k).lstrip('-').isdigit()]
                    if isinstance(j.get("spurious_nodes"), list):
                        spurious=[int(k) for k in j["spurious_nodes"] if str(k).lstrip('-').isdigit()]
                if causal is None:
                    c, rts, s = slice_from_labels(g, y)
                    causal, spurious = c, s
                sink_mask=(y>0.5).to(DEVICE)
                if causal:
                    ccs=ccs_for_graph(model, g, causal, sink_mask)
                    cfam=cfam_for_graph(model, g, causal, spurious, sink_mask)
                    if ccs is not None: ccs_vals.append(ccs)
                    if cfam is not None: cfam_vals.append(cfam)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise

        # per-source metrics
        per_source={}
        s_f1=0.0; s_n=0
        for src, agg in per_src_agg.items():
            tp,fp,fn=agg["tp"],agg["fp"],agg["fn"]
            prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
            per_source[src]={"precision":prec,"recall":rec,"f1":f1,"thr":thr_map.get(src, None)}
            s_f1+=f1; s_n+=1
        overall_f1=(s_f1/max(1,s_n)) if s_n>0 else 0.0  # macro over sources

        # write preds CSV for this split
        import csv
        csv_path = OUT_ROOT/f"{split_name}_preds.csv"
        with open(csv_path,"w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f, fieldnames=["graph","node","p","y","source"]); w.writeheader()
            for r in preds_out: w.writerow(r)

        return {
            "split": split_name,
            "overall": {"precision": None, "recall": None, "f1": overall_f1},  # macro F1 across sources
            "thresholds": thr_map,
            "n_graphs_used": n_used,
            "per_source": per_source,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count": len(ccs_vals),
            "CFAM_count": len(cfam_vals),
        }

    # evaluate splits using calibrated thresholds (valid-derived)
    reports={}
    reports["valid"] = eval_split(VALID_DIR, "valid", thr_map=thr_by_src)
    reports["train"] = eval_split(TRAIN_DIR, "train", thr_map=thr_by_src or {"default":0.05})
    reports["test"]  = eval_split(TEST_DIR,  "test",  thr_map=thr_by_src or {"default":0.05})

    # Demo: beam on first available labeled graph
    demo_paths=[]; demo_meta={"split":None,"graph_path":None}
    for d,name in [(VALID_DIR,"valid"),(TEST_DIR,"test"),(TRAIN_DIR,"train")]:
        if not Path(d).exists(): continue
        ds=GraphDir(d)
        for pt in ds.paths:
            g=safe_load(pt)
            y=labels_from_graph_or_json(g)
            if y is None: continue
            g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
            with torch.no_grad():
                logit,h,edges=model(g); p=torch.sigmoid(logit); scores=model.edge_scores(h,edges)
            # seeds = multi-roots if available else top-k
            c, roots, _ = slice_from_labels(g, y)
            seeds = roots if roots else torch.topk(p, k=min(12, p.numel())).indices.tolist()
            beam = run_beam(p, scores, edges, seeds)
            demo_paths=[{"score":bp["score"],"nodes":bp["nodes"]} for bp in beam[:10]]
            demo_meta={"split":name,"graph_path":pt}
            break
        if demo_paths: break

    out = {
        "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                  "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE),"dataset_weights":DATASET_WEIGHTS,
                  "relations":rels,"add_summary_edges":ADD_SUMMARY_EDGES},
        "thresholds_by_source": thr_by_src,
        "reports": reports,
        "demo": {"meta": demo_meta, "paths": demo_paths},
        "notes":{
            "interprocedural":"DFG/CFG/CALL/ARG2PARAM/RET2CALL/RET2LHS + optional DFG_THIN summary; multi-root OK.",
            "beam_guidance":"learned per-relation gates via rel_emb + edge_head; no fixed ordering.",
            "CCS":"p(X) vs p(do(X')) by zeroing causal-slice features; slice from labels or provided causal_nodes.",
            "CFAM":"grad-norm attribution mass on hidden h over causal vs causal+spurious; uses provided sets if present.",
            "calibration":"per-source thresholds chosen on valid to maximize F1; applied to other splits.",
            "provenance":"each graph carries 'source' from sidecar JSON; per-source training weights applied."
        }
    }
    with open(OUT_ROOT/"reports.json","w",encoding="utf-8") as f: json.dump(out, f, indent=2)
    print("Saved:", OUT_ROOT/"model.pt", "and", OUT_ROOT/"reports.json")

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:", e)
    traceback.print_exc()


c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Saving to: out\cvul\1760248531
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_31804\50842969.py:86: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location="cpu")
C:\Users\MSHUV

[train] epoch 1 loss=0.000000 (graphs=0)
[train] epoch 2 loss=0.000000 (graphs=0)
[train] epoch 3 loss=0.000000 (graphs=0)
[train] epoch 4 loss=0.000000 (graphs=0)
[train] epoch 5 loss=0.000000 (graphs=0)
[train] epoch 6 loss=0.000000 (graphs=0)
[train] epoch 7 loss=0.000000 (graphs=0)
[train] epoch 8 loss=0.000000 (graphs=0)
[train] epoch 9 loss=0.000000 (graphs=0)
[train] epoch 10 loss=0.000000 (graphs=0)
[WARN] No training graphs had labels. Add node labels or sidecar JSON with 'nodes[].y' or 'sinks' or 'vulnerable_paths'.
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
Saved: out\cvul\1760248531\model.pt and out\cvul\1760248531\reports.json


In [2]:
# Causal-Vul (Inter-proc, Beam, CCS/CFAM) – strict label-driven, no heuristics for causal/spurious
# Saves to out/cvul/<ts>/
# -----------------------------------------------------------------------------------------------

# ======= CONFIG =================================================================================
EPOCHS           = 10
HIDDEN           = 64
LAYERS           = 3
LR               = 2e-3
WEIGHT_DECAY     = 1e-4
NEG_POS_RATIO    = 20
USE_AMP          = True
ADD_SUMMARY_EDGES= True     # thin pass-through edges across calls
BEAM_W           = 16
MAX_HOPS         = 6
ALPHA_NODE       = 0.7

# Provenance-aware dataset weights (overridden by sidecar JSON "source"/"dataset" if present)
DATASET_WEIGHTS = {"default": 1.0}

# Label field names to accept (deterministic mapping; adjust if your data uses different keys)
LABEL_KEYS = ["y", "label", "is_sink", "vulnerable"]  # node-level fields (0/1)
SINK_KEYS  = ["sinks", "sink_nodes"]                  # sidecar JSON arrays of sink node ids
PATH_KEYS  = ["vulnerable_paths", "paths_idx"]        # sidecar JSON arrays of positive paths

# Dirs
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"
# ================================================================================================

import os, json, time, random, math, gc, csv, traceback
from pathlib import Path
from collections import defaultdict, deque
from typing import Dict, List, Tuple, Optional

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric is required to load HeteroData graphs") from e

# -------------------- device --------------------
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.empty(1024, device="cuda"); del _
            print(f"[env] torch={torch.__version__} device=cuda")
            try: print("GPU:", torch.cuda.get_device_name(0))
            except: pass
            return torch.device("cuda")
        except Exception:
            pass
    print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device()

# -------------------- IO --------------------
def near_json(pt_path: str) -> Optional[str]:
    p = Path(pt_path)
    for c in [p.with_suffix(".json"), p.with_suffix(".aug.json"), p.with_suffix(".unified.json")]:
        if c.exists(): return str(c)
    return None

def load_json(path: Optional[str]) -> Optional[dict]:
    if not path or not os.path.exists(path): return None
    try:
        with open(path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def graph_source(j: Optional[dict]) -> str:
    if isinstance(j, dict):
        for k in ("source","dataset","origin","corpus"):
            v=j.get(k, None)
            if isinstance(v, str) and v.strip(): return v.strip()
    return "default"

def safe_load(path: str):
    return torch.load(path, map_location="cpu")

class GraphDir(Dataset):
    def __init__(self, d: str):
        self.paths = sorted([str(p) for p in Path(d).glob("*.pt")])
        if not self.paths:
            print(f"[WARN] No .pt files in: {d}")
        else:
            print(f"[DATA] {len(self.paths)} graphs in {d}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt)
        g.__dict__["_aug_json_path"] = near_json(pt)
        g.__dict__["_path"] = pt
        return g

# -------------------- edge utils --------------------
NODE_TYPE = "node"
FORWARD_RELS = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS")

def get_edge_index(g: HeteroData, et: Tuple[str,str,str]) -> torch.Tensor:
    if et not in g.edge_types:
        dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else "cpu")
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[et]
    ei = getattr(store, "edge_index", None)
    if ei is None:
        adj_t = getattr(store, "adj_t", None)
        if adj_t is not None:
            row,col,_ = adj_t.coo()
            ei = torch.stack([col,row], dim=0)
        else:
            dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else "cpu")
            ei = torch.zeros((2,0), dtype=torch.long, device=dev)
    return ei

def add_thin_summary_edges(hdev, edges: Dict[str,torch.Tensor]) -> Dict[str,torch.Tensor]:
    if not ADD_SUMMARY_EDGES: return edges
    e_ap = edges.get("ARG2PARAM", torch.zeros((2,0), dtype=torch.long, device=hdev))
    e_rl = edges.get("RET2LHS",   torch.zeros((2,0), dtype=torch.long, device=hdev))
    if e_ap.numel()==0 and e_rl.numel()==0: return edges
    edges = dict(edges)
    edges["DFG_THIN"] = torch.cat([x for x in (e_ap,e_rl) if x.numel()>0], dim=1)
    return edges

# -------------------- labels & strict slice (no heuristics) --------------------
def labels_from_graph_or_json(g: HeteroData) -> Optional[torch.Tensor]:
    st=g[NODE_TYPE]
    # in-graph node label fields
    for k in LABEL_KEYS:
        if hasattr(st, k):
            y=getattr(st,k)
            if isinstance(y, torch.Tensor):
                y=y.float()
                if y.dim()==2 and y.size(1)==1: y=y.view(-1)
                if y.dim()==1 and y.numel()==st.num_nodes:
                    return y
    # sidecar sinks
    j=load_json(getattr(g,"_aug_json_path", None))
    if isinstance(j, dict):
        for k in SINK_KEYS:
            if k in j and isinstance(j[k], list):
                y=torch.zeros(st.num_nodes, dtype=torch.float32)
                for v in j[k]:
                    if isinstance(v,(int,str)) and str(v).lstrip('-').isdigit():
                        v=int(v)
                        if 0<=v<y.numel(): y[v]=1.0
                if (y>0.5).any():
                    return y
        # paths → terminal node as sink
        for k in PATH_KEYS:
            if k in j and isinstance(j[k], list):
                y=torch.zeros(st.num_nodes, dtype=torch.float32)
                for path in j[k]:
                    if isinstance(path,(list,tuple)) and path:
                        last=path[-1]
                        if isinstance(last,(int,str)) and str(last).lstrip('-').isdigit():
                            last=int(last)
                            if 0<=last<y.numel(): y[last]=1.0
                if (y>0.5).any():
                    return y
    return None

def build_adj(g: HeteroData):
    fwd=defaultdict(list); rev=defaultdict(list)
    for r in FORWARD_RELS:
        et=(NODE_TYPE,r,NODE_TYPE)
        if et not in g.edge_types: continue
        ei=get_edge_index(g,et)
        if ei.numel()==0: continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            fwd[u].append(v); rev[v].append(u)
    return fwd,rev

def slice_from_labels(g: HeteroData, y: torch.Tensor):
    sinks=[i for i in range(y.numel()) if float(y[i])>0.5]
    if not sinks: return [],[],[]
    fwd,rev = build_adj(g)
    B=set(); dq=deque(sinks)
    while dq:
        u=dq.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: dq.append(v)
    F=set(); dq=deque(list(B))
    while dq:
        u=dq.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: dq.append(v)
    causal=sorted(list(B.intersection(F)))
    roots=sorted([u for u in B if not any((v in B) for v in rev.get(u, []))])  # multi-root OK
    spurious=[i for i in range(y.numel()) if i not in set(causal)]
    return causal, roots, spurious

# -------------------- model --------------------
class ProjectIn(nn.Module):
    def __init__(self, hidden:int):
        super().__init__()
        self.hidden=hidden
        self.proj=nn.ModuleDict()
    def forward(self, x):
        d=x.size(1); k=str(d)
        if k not in self.proj:
            lin=nn.Linear(d, self.hidden)
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)
            self.proj[k]=lin.to(x.device, dtype=x.dtype)
        return self.proj[k](x)

class RelLayer(nn.Module):
    def __init__(self, hidden:int, rels:List[str]):
        super().__init__()
        self.rels=rels
        self.self_lin=nn.Linear(hidden, hidden)
        self.msg_lin=nn.ModuleDict({r: nn.Linear(hidden, hidden) for r in rels})
        self.ln=nn.LayerNorm(hidden)
    def forward(self, h, edges:Dict[str,torch.Tensor]):
        out=self.self_lin(h); N=h.size(0)
        for r in self.rels:
            ei=edges.get(r,None)
            if ei is None or ei.numel()==0: continue
            src,dst=ei[0],ei[1]
            m=self.msg_lin[r](h[src])
            agg=torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            deg=torch.zeros(N, device=h.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
            out=out + agg/deg.clamp(min=1).unsqueeze(-1)
        return self.ln(F.silu(out))

class CausalVulNet(nn.Module):
    def __init__(self, hidden:int, rels:List[str]):
        super().__init__()
        self.rels=rels
        self.project_in=ProjectIn(hidden)
        self.layers=nn.ModuleList([RelLayer(hidden, rels) for _ in range(LAYERS)])
        self.node_head=nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.rel_emb=nn.Embedding(num_embeddings=len(rels), embedding_dim=hidden)
        self.edge_head=nn.Sequential(nn.Linear(hidden*3, hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.part_head=nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden,1))
    def encode(self, g: HeteroData):
        st=g[NODE_TYPE]
        xs=[]
        if hasattr(st,"x_text"): xs.append(st.x_text.float())
        if hasattr(st,"x"):      xs.append(st.x.float())
        if not xs: xs=[torch.zeros(st.num_nodes,1, device=st.x.device if hasattr(st,"x") else "cpu")]
        x=xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
        h=F.relu(self.project_in(x))
        edges={}
        for r in self.rels:
            et=(NODE_TYPE,r,NODE_TYPE)
            edges[r]=get_edge_index(g, et).to(h.device) if et in g.edge_types else torch.zeros((2,0), dtype=torch.long, device=h.device)
        edges=add_thin_summary_edges(h.device, edges)
        for layer in self.layers: h=layer(h, edges)
        return h,edges
    def forward(self, g: HeteroData):
        h,edges=self.encode(g)
        logit=self.node_head(h).squeeze(-1)
        return logit,h,edges
    def edge_scores(self, h, edges):
        out={}
        rel_list=list(self.rels)
        if "DFG_THIN" in edges and "DFG_THIN" not in rel_list:
            rel_list=rel_list+["DFG_THIN"]
        for i,r in enumerate(rel_list):
            ei=edges.get(r,None)
            if ei is None or ei.numel()==0:
                out[r]=torch.empty(0, device=h.device); continue
            s,d=ei[0], ei[1]
            j=min(i, self.rel_emb.num_embeddings-1)
            t=self.rel_emb(torch.tensor(j, device=h.device)).view(1,-1).expand(s.numel(),-1)
            out[r]=self.edge_head(torch.cat([h[s],h[d],t], dim=-1)).squeeze(-1)
        return out

# -------------------- losses --------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.97, gamma=2.0): super().__init__(); self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p, y, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps)
        pt=torch.where(y>0.5, p, 1-p)
        w=(self.alpha_pos*(y>0.5).float() + (1-self.alpha_pos)*(y<=0.5).float())
        if weight is not None: w=w*weight
        return (-((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w).mean()

def class_weights(y):
    pos=(y>0.5).sum().item(); neg=y.numel()-pos
    w=torch.ones_like(y)
    if pos>0: w[y>0.5]=(neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p, y, max_ratio=NEG_POS_RATIO):
    pos_idx=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx=(y<=0.5).nonzero(as_tuple=False).view(-1)
    k=int(min(len(neg_idx), max_ratio*max(1,len(pos_idx))))
    if k<=0: return (y>0.5)
    if len(neg_idx)==0: return (y>0.5)
    _, inds = torch.topk(p[neg_idx], k=k)
    keep_neg=neg_idx[inds]
    m=torch.zeros_like(y, dtype=torch.bool)
    if len(pos_idx)>0: m[pos_idx]=True
    m[keep_neg]=True
    return m

def edge_participation_loss(h, edges, model: CausalVulNet, p):
    loss=0.0; cnt=0
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei[0], ei[1]
        logits=model.part_head(torch.cat([h[s],h[d]], dim=-1)).squeeze(-1)
        target=torch.minimum(p[s], p[d]).detach()
        loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        cnt+=1
    return loss/max(1,cnt)

def monotonicity_loss_on_paths(logit, paths):
    loss=0.0; cnt=0
    for path in paths:
        if len(path)<2: continue
        z=logit[path]; diffs=z[1:]-z[:-1]
        loss=loss+F.relu(-diffs).mean(); cnt+=1
    return loss/max(1,cnt)

def path_ranking_loss(logit, pos_paths, neg_paths, margin=0.3):
    if not pos_paths or not neg_paths: return logit.new_zeros(())
    def score(path): return torch.sigmoid(logit[path]).mean()
    L=logit.new_tensor(0.0); m=logit.new_tensor(margin); cnt=0
    for pp in pos_paths:
        sp=score(pp)
        for np in neg_paths[:3]:
            sn=score(np)
            L=L+F.relu(m-(sp-sn)); cnt+=1
    return L/max(1,cnt)

# -------------------- beam search --------------------
def run_beam(p, edge_scores, edges, seeds, beam_width=BEAM_W, max_hops=MAX_HOPS, alpha_node=ALPHA_NODE):
    adj=defaultdict(lambda: defaultdict(list))
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d): adj[r][u].append(v)
    BeamPath=lambda score,nodes: {"score":score,"nodes":nodes}
    beams=[BeamPath(float(torch.log(p[s].clamp(1e-9,1-1e-9))), [s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u=b["nodes"][-1]
            for r,nbrs in adj.items():
                if u not in nbrs: continue
                ei=edges.get(r,None); es=edge_scores.get(r,None)
                for v in nbrs[u]:
                    es_val=0.0
                    if ei is not None and es is not None and ei.numel()>0 and es.numel()>0:
                        mask=((ei[0]==u)&(ei[1]==v))
                        if mask.any(): es_val=float(es[mask].max().item())
                    score=b["score"]+alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9)))+(1-alpha_node)*es_val
                    cand.append(BeamPath(score, b["nodes"]+[v]))
        if not cand: break
        cand.sort(key=lambda x:x["score"], reverse=True)
        beams=cand[:beam_width]
        finished.extend(beams[:beam_width//2])
    seen=set(); uniq=[]
    for bp in sorted(finished, key=lambda x:x["score"], reverse=True):
        key=tuple(bp["nodes"][-4:])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# -------------------- CCS & CFAM --------------------
def ccs_for_graph(model, g, causal_nodes, sink_mask):
    if not causal_nodes: return None
    dev=next(model.parameters()).device
    g=g.to(dev, non_blocking=(dev.type=="cuda"))
    with torch.no_grad():
        logit,h,edges=model(g); p=torch.sigmoid(logit)
        p_sink=p[sink_mask].mean() if sink_mask.any() else p.mean()
    st=g[NODE_TYPE]
    xs=[]
    if hasattr(st,"x_text"): xs.append(st.x_text.float().clone())
    if hasattr(st,"x"):      xs.append(st.x.float().clone())
    if not xs: xs=[torch.zeros(st.num_nodes,1, device=dev)]
    x=xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
    x[causal_nodes]=0.0  # do(X')
    h0=F.relu(model.project_in(x))
    edges={}
    for r in model.rels:
        et=(NODE_TYPE,r,NODE_TYPE)
        edges[r]=get_edge_index(g, et).to(dev) if et in g.edge_types else torch.zeros((2,0), dtype=torch.long, device=dev)
    edges=add_thin_summary_edges(dev, edges)
    for layer in model.layers: h0=layer(h0, edges)
    logit2=model.node_head(h0).squeeze(-1); p2=torch.sigmoid(logit2)
    p2_sink=p2[sink_mask].mean() if sink_mask.any() else p2.mean()
    return float((p_sink - p2_sink).pow(2).item())

def cfam_for_graph(model, g, causal_nodes, spurious_nodes, sink_mask):
    if not causal_nodes: return None
    dev=next(model.parameters()).device
    g=g.to(dev, non_blocking=(dev.type=="cuda"))
    model.train()
    h,edges=model.encode(g); h.retain_grad()
    logit=model.node_head(h).squeeze(-1)
    tgt = logit[sink_mask].sum() if sink_mask.any() else logit.mean()
    model.zero_grad(set_to_none=True); tgt.backward()
    attr=h.grad.norm(dim=1)
    c=torch.tensor(causal_nodes, device=dev, dtype=torch.long)
    s=torch.tensor(spurious_nodes, device=dev, dtype=torch.long) if spurious_nodes else torch.tensor([], device=dev, dtype=torch.long)
    sum_c=float(attr[c].sum().item()) if c.numel()>0 else 0.0
    sum_s=float(attr[s].sum().item()) if s.numel()>0 else 0.0
    if sum_c+sum_s==0.0: return None
    return float(sum_c/(sum_c+sum_s))

# -------------------- metrics & calibration --------------------
def micro_metrics(all_pairs, thr_map):
    # all_pairs: list of (p [N], y [N], src)
    tp=fp=fn=0.0
    for p,y,src in all_pairs:
        thr=thr_map.get(src, 0.25)
        pred=(p>=thr).float()
        tp+=(pred*y).sum().item()
        fp+=(pred*(1-y)).sum().item()
        fn+=((1-pred)*y).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec, "recall":rec, "f1":f1}

def calibrate_thresholds_by_source(model, dir_valid: str):
    if not Path(dir_valid).exists(): return {}
    ds=GraphDir(dir_valid); 
    if not ds.paths: return {}
    pairs=defaultdict(list)  # src -> [(p,y),...]
    for pt in ds.paths:
        g=safe_load(pt); y=labels_from_graph_or_json(g)
        if y is None: continue
        j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
        g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with torch.no_grad():
            logit,_,_=model(g); p=torch.sigmoid(logit).detach().cpu()
        pairs[src].append((p, y.cpu()))
    thr_grid=[0.05]+[i/20 for i in range(1,20)]  # 0.05..0.95
    out={}
    for src, lst in pairs.items():
        best_thr=0.25; best_f1=-1
        for thr in thr_grid:
            tp=fp=fn=0.0
            for p,y in lst:
                pred=(p>=thr).float()
                tp+=(pred*y).sum().item()
                fp+=(pred*(1-y)).sum().item()
                fn+=((1-pred)*y).sum().item()
            prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
            if f1>best_f1: best_f1=f1; best_thr=thr
        out[src]=best_thr
    return out

# -------------------- train & eval --------------------
TIMESTAMP=str(int(time.time()))
OUT_ROOT=Path("out")/"cvul"/TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

def train_and_eval():
    random.seed(0); torch.manual_seed(0)
    if DEVICE.type=="cuda":
        torch.cuda.manual_seed_all(0)
        try: torch.backends.cudnn.benchmark=True
        except: pass

    ds_tr=GraphDir(TRAIN_DIR)
    dl_tr=DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=lambda xs: xs[0])

    # derive relation set from a probe
    probe=None
    for ds_path in (ds_tr.paths[:1] if ds_tr.paths else []):
        probe=safe_load(ds_path); break
    if probe is None:
        for d in [VALID_DIR, TEST_DIR]:
            if Path(d).exists():
                tmp=GraphDir(d)
                if tmp.paths: probe=safe_load(tmp.paths[0]); break
    if probe is None: raise RuntimeError("No graphs found in train/valid/test.")
    rels = sorted(list(set([r for (s,r,t) in probe.edge_types if r in FORWARD_RELS])))

    model=CausalVulNet(HIDDEN, rels).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    focal=FocalBCELoss(alpha_pos=0.97, gamma=2.0)
    scaler=torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

    used_train_graphs=0
    skip_no_label=0
    for ep in range(1,EPOCHS+1):
        model.train(); loss_sum=0.0; n_seen=0
        for g in dl_tr:
            try:
                g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y=labels_from_graph_or_json(g)
                if y is None or float((y>0.5).sum().item())==0:
                    skip_no_label+=1
                    continue
                used_train_graphs+=1

                j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
                ds_w=DATASET_WEIGHTS.get(src, DATASET_WEIGHTS["default"])

                # positive paths (supervision for mono/rank if present)
                pos_paths=[]
                if isinstance(j,dict):
                    for k in PATH_KEYS:
                        if k in j and isinstance(j[k], list):
                            for path in j[k]:
                                if isinstance(path,(list,tuple)) and path:
                                    plist=[int(k) for k in path if isinstance(k,(int,str)) and str(k).lstrip('-').isdigit()]
                                    if plist: pos_paths.append(plist)

                with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type=="cuda")):
                    logit,h,edges=model(g); p=torch.sigmoid(logit)
                    keep=hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                    y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                    w=class_weights(y_m)

                    loss_node=focal(p_m, y_m, weight=w) * ds_w
                    loss_edge=edge_participation_loss(h, edges, model, p) * (0.10*ds_w)
                    loss_mono=monotonicity_loss_on_paths(logit, pos_paths) * (0.10*ds_w) if pos_paths else logit.new_zeros(())
                    loss_rank=path_ranking_loss(logit, pos_paths, [], margin=0.3) * (0.10*ds_w) if pos_paths else logit.new_zeros(())
                    loss = loss_node + loss_edge + loss_mono + loss_rank

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                scaler.step(opt); scaler.update()

                loss_sum += float(loss.detach().cpu()); n_seen+=1
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={(loss_sum/max(1,n_seen)):.6f} (graphs={n_seen})")

    if used_train_graphs==0:
        print(f"[WARN] No training graphs used. Skipped for missing/zero-positive labels: {skip_no_label}.")
        print("       Check LABEL_KEYS / SINK_KEYS / PATH_KEYS to match your dataset’s fields.")

    # save model & config
    torch.save({"state_dict":model.state_dict(),"rels":model.rels,"hidden":HIDDEN,"layers":LAYERS}, OUT_ROOT/"model.pt")
    with open(OUT_ROOT/"cfg.json","w",encoding="utf-8") as f:
        json.dump({"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                   "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE),"dataset_weights":DATASET_WEIGHTS,
                   "relations":rels,"add_summary_edges":ADD_SUMMARY_EDGES,
                   "label_keys":LABEL_KEYS,"sink_keys":SINK_KEYS,"path_keys":PATH_KEYS}, f, indent=2)

    # per-source threshold calibration
    thr_by_src = calibrate_thresholds_by_source(model, VALID_DIR)
    with open(OUT_ROOT/"thresholds_by_source.json","w",encoding="utf-8") as f:
        json.dump(thr_by_src, f, indent=2)

    def eval_split(dir_path: str, split_name: str, thr_map: Dict[str,float]):
        preds_out=[]; all_pairs=[]; per_src=defaultdict(lambda: {"tp":0.0,"fp":0.0,"fn":0.0})
        ccs_vals=[]; cfam_vals=[]; n_used=0
        if not Path(dir_path).exists():
            return {"split":split_name,"overall":{"precision":None,"recall":None,"f1":0.0},
                    "thresholds":thr_map,"n_graphs_used":0,"per_source":{},
                    "CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}

        ds=GraphDir(dir_path)
        for pt in ds.paths:
            try:
                g=safe_load(pt); y=labels_from_graph_or_json(g)
                if y is None: continue
                j=load_json(getattr(g,"_aug_json_path", None)); src=graph_source(j)
                thr = thr_map.get(src, 0.25 if split_name=="valid" else 0.05)
                g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                with torch.no_grad():
                    logit,h,edges=model(g); p=torch.sigmoid(logit).detach().cpu()
                n_used+=1
                # micro aggregation
                all_pairs.append((p, y.cpu(), src))
                # per-src counts at this thr
                pred=(p>=thr).float()
                per_src[src]["tp"] += (pred*y).sum().item()
                per_src[src]["fp"] += (pred*(1-y)).sum().item()
                per_src[src]["fn"] += ((1-pred)*y).sum().item()
                # save preds
                for i,pv in enumerate(p.tolist()):
                    preds_out.append({"graph": pt, "node": i, "p": float(pv), "y": float(y[i].item()), "source": src})
                # CCS/CFAM
                causal=None; spurious=None
                if isinstance(j,dict) and ("causal_nodes" in j or "spurious_nodes" in j):
                    if isinstance(j.get("causal_nodes"), list):
                        causal=[int(k) for k in j["causal_nodes"] if str(k).lstrip('-').isdigit()]
                    if isinstance(j.get("spurious_nodes"), list):
                        spurious=[int(k) for k in j["spurious_nodes"] if str(k).lstrip('-').isdigit()]
                if causal is None:
                    c, roots, s = slice_from_labels(g, y)
                    causal, spurious = c, s
                sink_mask=(y>0.5).to(DEVICE)
                if causal:
                    ccs=ccs_for_graph(model, g, causal, sink_mask)
                    cfam=cfam_for_graph(model, g, causal, spurious, sink_mask)
                    if ccs is not None: ccs_vals.append(ccs)
                    if cfam is not None: cfam_vals.append(cfam)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise

        # overall micro metrics
        overall = micro_metrics(all_pairs, thr_map) if all_pairs else {"precision":None,"recall":None,"f1":0.0}
        # per-source pretty
        per_source={}
        for src, agg in per_src.items():
            tp,fp,fn=agg["tp"],agg["fp"],agg["fn"]
            prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
            per_source[src]={"precision":prec,"recall":rec,"f1":f1,"thr":thr_map.get(src, None)}

        # write preds CSV
        csv_path = OUT_ROOT/f"{split_name}_preds.csv"
        with open(csv_path,"w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f, fieldnames=["graph","node","p","y","source"]); w.writeheader()
            for r in preds_out: w.writerow(r)

        return {
            "split": split_name,
            "overall": overall,
            "thresholds": thr_map,
            "n_graphs_used": n_used,
            "per_source": per_source,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count": len(ccs_vals),
            "CFAM_count": len(cfam_vals),
        }

    thr_by_src = calibrate_thresholds_by_source(model, VALID_DIR)
    reports={}
    reports["valid"] = eval_split(VALID_DIR, "valid", thr_map=thr_by_src)
    reports["train"] = eval_split(TRAIN_DIR, "train", thr_map=thr_by_src or {"default":0.05})
    reports["test"]  = eval_split(TEST_DIR,  "test",  thr_map=thr_by_src or {"default":0.05})

    # Demo beam (first labeled graph found)
    demo_paths=[]; demo_meta={"split":None,"graph_path":None}
    for d,name in [(VALID_DIR,"valid"),(TEST_DIR,"test"),(TRAIN_DIR,"train")]:
        if not Path(d).exists(): continue
        ds=GraphDir(d)
        for pt in ds.paths:
            g=safe_load(pt)
            y=labels_from_graph_or_json(g)
            if y is None: continue
            g=g.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
            with torch.no_grad():
                logit,h,edges=model(g); p=torch.sigmoid(logit); scores=model.edge_scores(h,edges)
            c, roots, _ = slice_from_labels(g, y)
            seeds = roots if roots else torch.topk(p, k=min(12, p.numel())).indices.tolist()
            beam = run_beam(p, scores, edges, seeds)
            demo_paths=[{"score":bp["score"],"nodes":bp["nodes"]} for bp in beam[:10]]
            demo_meta={"split":name,"graph_path":pt}
            break
        if demo_paths: break

    out = {
        "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                  "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE),"dataset_weights":DATASET_WEIGHTS,
                  "relations":rels,"add_summary_edges":ADD_SUMMARY_EDGES,
                  "label_keys":LABEL_KEYS, "sink_keys":SINK_KEYS, "path_keys":PATH_KEYS},
        "thresholds_by_source": thr_by_src,
        "reports": reports,
        "demo": {"meta": demo_meta, "paths": demo_paths},
        "notes":{
            "interprocedural":"DFG/CFG/CALL/ARG2PARAM/RET2CALL/RET2LHS + thin DFG_THIN; multi-root supported.",
            "beam_guidance":"learned per-relation gates via rel_emb + edge_head; no fixed ordering.",
            "CCS":"p(X) vs p(do(X')) by zeroing causal-slice features strictly derived from ground-truth labels/sinks/paths.",
            "CFAM":"grad-norm attribution over hidden h; CFAM = Σ_causal / (Σ_causal + Σ_spurious).",
            "calibration":"per-source thresholds selected on valid; applied to train/test.",
            "provenance":"per-graph 'source' from sidecar; training loss weighted per DATASET_WEIGHTS."
        }
    }
    with open(OUT_ROOT/"reports.json","w",encoding="utf-8") as f: json.dump(out, f, indent=2)
    print("Saved:", OUT_ROOT/"model.pt", "and", OUT_ROOT/"reports.json")

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:", e)
    traceback.print_exc()


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Saving to: out\cvul\1760250975
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_31804\2604120376.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location="cpu")
C:\Users\MSH

[train] epoch 1 loss=0.000000 (graphs=0)
[train] epoch 2 loss=0.000000 (graphs=0)
[train] epoch 3 loss=0.000000 (graphs=0)
[train] epoch 4 loss=0.000000 (graphs=0)
[train] epoch 5 loss=0.000000 (graphs=0)
[train] epoch 6 loss=0.000000 (graphs=0)
[train] epoch 7 loss=0.000000 (graphs=0)
[train] epoch 8 loss=0.000000 (graphs=0)
[train] epoch 9 loss=0.000000 (graphs=0)
[train] epoch 10 loss=0.000000 (graphs=0)
[WARN] No training graphs used. Skipped for missing/zero-positive labels: 34380.
       Check LABEL_KEYS / SINK_KEYS / PATH_KEYS to match your dataset’s fields.
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert
[DATA] 3438 graphs in Dataset/train/he

In [8]:
# %% Causal-Vul (inter-proc + CCS/CFAM) — fixed beam Path + AMP API
import os, json, time, math, gc, random, traceback, warnings
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, deque

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed") from e

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`")

# ---------------- config ----------------
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"
SIDECAR_DIRS = []  # optional extra dirs with sidecar JSON

OUT_ROOT = Path("out/cvul") / str(int(time.time()))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (DEVICE.type == "cuda")

EPOCHS = 10
BATCH_SIZE = 1
HIDDEN = 64
LAYERS = 3
LR = 2e-3
WD = 1e-4
NEG_POS_RATIO = 20

RELATIONS = ["ARG2PARAM", "CALL", "CFG", "DFG", "RET2CALL", "RET2LHS"]
ADD_SUMMARY_EDGES = True  # thin DFG pass-through

MAX_NODES_FOR_GPU = 3500
MAX_EDGES_FOR_GPU = 12000
CLEAR_CACHE_EVERY = 12

LABEL_KEYS = ["y","label","labels","vulnerable","is_sink","target","targets"]
SINK_KEYS  = ["sinks","sink_nodes"]
PATH_KEYS  = ["vulnerable_paths","paths_idx"]
DATASET_WEIGHTS = {"default":1.0}

def set_seed(s=23):
    random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def primary_node_type(g: "HeteroData"):
    for nt in g.node_types:
        st = g[nt]
        if hasattr(st,"num_nodes") and (hasattr(st,"x") or hasattr(st,"x_text") or any(hasattr(st,k) for k in LABEL_KEYS)):
            return nt
    return g.node_types[0] if g.node_types else None

def get_edge_index(g, et):
    st = g[et]
    if hasattr(st, "edge_index") and st.edge_index is not None:
        return st.edge_index
    if hasattr(st, "adj_t") and st.adj_t is not None:
        row,col,_ = st.adj_t.coo()
        return torch.stack([col,row],0)
    return torch.zeros((2,0), dtype=torch.long)

def add_thin_dfg(g, nt, rel="DFG", rel_name="DFG_THIN", max_new=15000):
    et = (nt, rel, nt)
    if et not in g.edge_types: return
    ei = get_edge_index(g, et)
    if ei.numel()==0: return
    src,dst = ei
    adj = defaultdict(list)
    for u,v in zip(src.tolist(), dst.tolist()):
        if len(adj[u])<64: adj[u].append(v)
    s2,d2=[],[]
    for u, nbs in adj.items():
        for v in nbs[:32]:
            for w in adj.get(v,[])[:16]:
                if u!=w:
                    s2.append(u); d2.append(w)
                    if len(s2)>=max_new: break
            if len(s2)>=max_new: break
        if len(s2)>=max_new: break
    if s2:
        e2 = torch.tensor([s2,d2], dtype=torch.long)
        g[(nt, rel_name, nt)].edge_index = e2

def find_sidecar_json(stem):
    for d in SIDECAR_DIRS:
        p = Path(d)/f"{stem}.json"
        if p.exists(): return p
    return None

class GraphDir(Dataset):
    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob("*.pt"))
        print(f"[DATA] {len(self.paths)} graphs in {folder}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        try:
            g = torch.load(p, map_location="cpu")
        except Exception:
            from torch.serialization import add_safe_globals
            from torch_geometric.data.storage import BaseStorage
            add_safe_globals([BaseStorage])
            g = torch.load(p, map_location="cpu")
        g.__dict__["_sidecar"] = find_sidecar_json(p.stem)
        return g

# ---------- imbalance/noise ----------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.97, gamma=2.0):
        super().__init__(); self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p, y, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps); pt=torch.where(y>0,p,1-p)
        w = (self.alpha_pos*(y>0).float() + (1-self.alpha_pos)*(y<=0).float())
        if weight is not None: w = w*weight
        loss = -((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w
        return loss.mean()

def class_weights(y):
    pos=y.sum(); neg=y.numel()-pos
    w=torch.ones_like(y)
    if pos>0: w[y>0.5]=(neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p,y,max_ratio=20):
    y=y.view(-1); p=p.view(-1)
    pos_idx=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx=(y<=0.5).nonzero(as_tuple=False).view(-1)
    k=int(min(neg_idx.numel(), max_ratio*max(1,pos_idx.numel())))
    if k<=0 or neg_idx.numel()==0:
        return (y>0.5)
    topk=torch.topk(p[neg_idx], k=k).indices
    keep_neg = neg_idx[topk]
    mask=torch.zeros_like(y, dtype=torch.bool)
    if pos_idx.numel()>0: mask[pos_idx]=True
    if keep_neg.numel()>0: mask[keep_neg]=True
    return mask

# ---------- model ----------
class RelGate(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__(); self.relations=relations
        self.emb=nn.Embedding(len(relations), hidden)
        self.log_prior=nn.Parameter(torch.zeros(len(relations)))
        self.edge_mlp=nn.Sequential(nn.Linear(hidden*3,hidden), nn.SiLU(), nn.Linear(hidden,1))
    def forward(self, h, edge_index_by_rel):
        out={}
        for i,r in enumerate(self.relations):
            ei=edge_index_by_rel.get(r,None)
            if ei is None or ei.numel()==0:
                out[r]=h.new_empty((0,)); continue
            u,v=ei[0],ei[1]
            te=self.emb.weight[i].view(1,-1).expand(u.numel(),-1).to(h.dtype)
            score=self.edge_mlp(torch.cat([h[u],h[v],te],-1)).squeeze(-1)
            out[r]=score+self.log_prior[i].to(score.dtype)
        return out

class Block(nn.Module):
    def __init__(self, hidden, relations, drop=0.1):
        super().__init__(); self.relations=relations
        self.msg=nn.ModuleDict({r: nn.Linear(hidden,hidden) for r in relations})
        self.self_lin=nn.Linear(hidden,hidden)
        self.norm=nn.LayerNorm(hidden); self.drop=nn.Dropout(drop)
        self.skip_alpha=nn.Parameter(torch.tensor(0.5))
    def forward(self,h,edge_index_by_rel):
        dt, dv = h.dtype, h.device
        out = self.self_lin(h).to(dt)
        agg = torch.zeros_like(h, dtype=dt, device=dv)
        deg = torch.zeros(h.size(0), device=dv, dtype=dt)
        for r in self.relations:
            ei=edge_index_by_rel.get(r,None)
            if ei is None or ei.numel()==0: continue
            u,v=ei[0],ei[1]
            m=self.msg[r](h[u]).to(dt)
            agg.index_add_(0, v, m)
            deg.index_add_(0, v, torch.ones(v.numel(), device=dv, dtype=dt))
        deg=deg.clamp(min=1).unsqueeze(-1)
        out=out + agg/deg
        out=self.drop(out)
        ka = self.skip_alpha.to(dt)
        out = ka*h + (1-ka)*out
        return self.norm(F.silu(out).to(dt))

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=RELATIONS):
        super().__init__()
        self.hidden=hidden
        self.relations=relations
        self.proj_pool=nn.ModuleDict()
        self.blocks=nn.ModuleList([Block(hidden, relations) for _ in range(layers)])
        self.node_head=nn.Sequential(nn.Linear(hidden,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.edge_head=nn.Sequential(nn.Linear(hidden*2,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.next_hop_head=nn.Sequential(nn.Linear(hidden*2,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.rel_gate=RelGate(hidden, relations)
    def _get_proj(self, d, device, dtype):
        key=f"in_{d}"
        if key not in self.proj_pool:
            lin=nn.Linear(d, self.hidden)
            nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))
            lin = lin.to(device=device, dtype=dtype)
            self.proj_pool[key]=lin
        else:
            mod=self.proj_pool[key]
            if next(mod.parameters()).device != device or next(mod.parameters()).dtype != dtype:
                self.proj_pool[key]=mod.to(device=device, dtype=dtype)
        return self.proj_pool[key]
    def encode(self, x):
        proj = self._get_proj(x.size(1), x.device, x.dtype)
        return F.relu(proj(x))
    def forward(self, g: "HeteroData", nt: str):
        xs=[]
        if hasattr(g[nt],"x_text"): xs.append(g[nt].x_text.float().to(next(self.parameters()).dtype))
        if hasattr(g[nt],"x"):      xs.append(g[nt].x.float().to(next(self.parameters()).dtype))
        if not xs: raise RuntimeError("No features (x/x_text) on node store")
        x = xs[0] if len(xs)==1 else torch.cat(xs,1)
        h = self.encode(x)
        e_by_rel={}
        avail = set(et[1] for et in g.edge_types)
        for r in (self.relations + (["DFG_THIN"] if "DFG_THIN" in avail else [])):
            et=(nt,r,nt)
            if et in g.edge_types:
                e_by_rel[r]=get_edge_index(g, et)
        for blk in self.blocks:
            h = blk(h, e_by_rel)
        logit=self.node_head(h).squeeze(-1)
        return logit, h, e_by_rel

# ---------- helpers ----------
def graph_shrink(g, nt, must_keep=None):
    N=g[nt].num_nodes
    tot=0
    for et in g.edge_types: tot += int(get_edge_index(g, et).size(1))
    if (N<=MAX_NODES_FOR_GPU) and (tot<=MAX_EDGES_FOR_GPU): return g
    keep=set(must_keep or [])
    for r in RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in [et[1] for et in g.edge_types]) else []):
        et=(nt,r,nt)
        if et in g.edge_types:
            ei=get_edge_index(g, et)
            if ei.numel()>0:
                s,d=ei
                sd=set(s.tolist()); dd=set(d.tolist())
                if keep & sd or keep & dd: keep |= sd; keep |= dd
    rnd=list(range(N)); random.shuffle(rnd)
    tgt=min(N, MAX_NODES_FOR_GPU)
    for u in rnd:
        if len(keep)>=tgt: break
        keep.add(u)
    keep=sorted(keep); imap={old:i for i,old in enumerate(keep)}
    g2=HeteroData()
    st=g[nt]
    if hasattr(st,"x_text"): g2[nt].x_text=st.x_text[keep]
    if hasattr(st,"x"):      g2[nt].x=st.x[keep]
    for k in LABEL_KEYS + ["nid","source","causal_nodes","spurious_nodes"]:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v) and v.numel()==st.num_nodes:
                g2[nt][k]=v[keep]
            else:
                g2[nt][k]=v
    g2[nt].num_nodes=len(keep)
    for et in g.edge_types:
        ei=get_edge_index(g, et)
        if ei.numel()==0: 
            g2[et].edge_index=ei; continue
        s,d=ei
        sf,df=[],[]
        for u,v in zip(s.tolist(), d.tolist()):
            if u in imap and v in imap:
                sf.append(imap[u]); df.append(imap[v])
        e=torch.tensor([sf,df], dtype=torch.long) if sf else torch.zeros((2,0), dtype=torch.long)
        g2[et].edge_index=e
    return g2

def labels_from_graph(g, nt):
    y=None; sinks=[]; paths=[]
    st=g[nt]
    for k in LABEL_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v) and v.numel()==st.num_nodes:
                y=v.float().view(-1); break
    for k in SINK_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v): sinks=v.view(-1).tolist()
            elif isinstance(v,(list,tuple)): sinks=list(v)
            break
    for k in PATH_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if isinstance(v,list) and v and isinstance(v[0],(list,tuple)):
                paths=[[int(ix) for ix in path] for path in v]
            break
    return y,sinks,paths

def build_adj(edge_index_by_rel):
    adj={r: defaultdict(list) for r in edge_index_by_rel.keys()}
    for r,ei in edge_index_by_rel.items():
        if ei is None or ei.numel()==0: continue
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            if len(adj[r][u])<64: adj[r][u].append(v)
    return adj

def program_slice(g, nt, roots, sinks, allowed=None):
    allowed = allowed or RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in [et[1] for et in g.edge_types]) else [])
    adj=defaultdict(list); radj=defaultdict(list)
    for r in allowed:
        et=(nt,r,nt)
        if et not in g.edge_types: continue
        ei=get_edge_index(g, et)
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            adj[u].append(v); radj[v].append(u)
    causal=set()
    for s in roots:
        q=deque([s]); vis=set([s])
        while q:
            u=q.popleft(); causal.add(u)
            for v in adj.get(u,[]):
                if v not in vis: vis.add(v); q.append(v)
    keep_back=set(); dq=deque([t for t in sinks])
    while dq:
        v=dq.popleft(); keep_back.add(v)
        for u in radj.get(v,[]):
            if u not in keep_back: keep_back.add(u); dq.append(u)
    return sorted(list(causal & keep_back))

@dataclass
class BeamPath:
    score: float
    nodes: list

def run_beam(p, edge_scores, edges, seeds, beam_width=32, max_hops=6, alpha_node=0.7):
    rels=sorted(set(edges.keys()) & set(edge_scores.keys()))
    adj=build_adj({r:edges[r] for r in rels})
    def clog(z): return float(torch.log(z.clamp(1e-9,1-1e-9)))
    if not seeds:
        K=min(8, p.numel())
        seeds = torch.topk(p, k=K).indices.tolist()
    beams=[BeamPath(clog(p[s]), [int(s)]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u=b.nodes[-1]
            for r in rels:
                for v in adj[r].get(u,[]):
                    ei=edges[r]; m=(ei[0]==u)&(ei[1]==v)
                    es = float(edge_scores[r][m].max().item()) if m.any() else 0.0
                    cand.append(BeamPath(b.score + alpha_node*clog(p[v]) + (1-alpha_node)*es, b.nodes+[v]))
        cand.sort(key=lambda x: x.score, reverse=True)
        beams=cand[:beam_width]; finished.extend(beams[:beam_width//2])
        if not beams: break
    uniq=[]; seen=set()
    for bp in sorted(finished, key=lambda x: x.score, reverse=True):
        key=tuple(bp.nodes[-min(4,len(bp.nodes)):])
        if key in seen: continue
        seen.add(key); uniq.append({"score":bp.score,"nodes":bp.nodes})
        if len(uniq)>=beam_width: break
    return uniq

def metrics_at(p,y,thr):
    pred=(p>=thr).float()
    tp=(pred*y).sum().item(); fp=(pred*(1-y)).sum().item(); fn=((1-pred)*y).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def edge_mask_from_paths(paths, N):
    m=torch.zeros(N, dtype=torch.bool)
    if not paths: return m
    for path in paths:
        for j in path:
            if 0<=j<N: m[j]=True
    return m

def attribution_cf(h, causal_idx, spurious_idx=None):
    g = h.grad.detach().abs() if (h.grad is not None) else h.detach().abs()
    mass = g.norm(p=2, dim=-1)
    causal_sum = mass[causal_idx].sum().item() if len(causal_idx)>0 else 0.0
    if spurious_idx is None:
        total = mass.sum().item() + 1e-9
        return causal_sum/total
    sp_sum = mass[spurious_idx].sum().item() if len(spurious_idx)>0 else 0.0
    return causal_sum / (causal_sum + sp_sum + 1e-9)

def compute_CCS(model, g, nt, causal_nodes):
    with torch.no_grad():
        logit,h,edges = model(g, nt)
        p = torch.sigmoid(logit)
    if not causal_nodes: return None
    g2 = g.clone()
    if hasattr(g2[nt],"x_text"): g2[nt].x_text = g2[nt].x_text.clone()
    if hasattr(g2[nt],"x"):      g2[nt].x      = g2[nt].x.clone()
    idx = torch.tensor(causal_nodes, dtype=torch.long)
    if hasattr(g2[nt],"x_text"): g2[nt].x_text[idx]=0.0
    if hasattr(g2[nt],"x"):      g2[nt].x[idx]=0.0
    with torch.no_grad():
        logit2,_,_ = model(g2, nt)
        p2 = torch.sigmoid(logit2)
    return float((p - p2).pow(2).mean().item())

# --------------- train & eval ---------------
def train_and_eval():
    set_seed(23)
    print(f"[env] torch={torch.__version__} device={DEVICE.type}")
    (OUT_ROOT/"cfg.json").write_text(json.dumps({
        "epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
        "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE.type,"relations":RELATIONS,
        "add_summary_edges":ADD_SUMMARY_EDGES,"label_keys":LABEL_KEYS,
        "sink_keys":SINK_KEYS,"path_keys":PATH_KEYS
    }, indent=2))

    ds_tr, ds_va, ds_te = GraphDir(TRAIN_DIR), GraphDir(VALID_DIR), GraphDir(TEST_DIR)

    # probe to finalize relation set (adds DFG_THIN if created)
    probe = None
    for ds in (ds_tr, ds_va, ds_te):
        if len(ds)>0: probe = ds[0]; break
    if probe is None:
        print("No graphs found."); return
    nt_probe = primary_node_type(probe) or "node"
    if ADD_SUMMARY_EDGES: add_thin_dfg(probe, nt_probe)
    relations_eff = RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in [et[1] for et in probe.edge_types]) else [])

    model = CausalVulNet(hidden=HIDDEN, layers=LAYERS, relations=relations_eff).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    # new AMP API
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    focal = FocalBCELoss()

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda xs: xs[0])
    dl_va = DataLoader(ds_va, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0])
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0])

    def train_epoch(ep):
        model.train(); loss_avg=0.0; steps=0
        pbar=tqdm(total=len(dl_tr), desc=f"[train] epoch {ep}", leave=False)
        for i,g in enumerate(dl_tr,1):
            try:
                nt = primary_node_type(g) or "node"
                if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
                # must-keep before shrinking
                y,sinks,paths = labels_from_graph(g, nt)
                keep=[]
                if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
                elif paths: 
                    for pth in paths: keep += pth
                elif sinks: keep += sinks
                g = graph_shrink(g, nt, must_keep=keep)

                # move to device
                dev = next(model.parameters()).device
                if hasattr(g[nt],"x_text"): g[nt].x_text=g[nt].x_text.to(dev)
                if hasattr(g[nt],"x"):      g[nt].x=g[nt].x.to(dev)
                for et in g.edge_types:
                    if hasattr(g[et],"edge_index"): g[et].edge_index=g[et].edge_index.to(dev)

                with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                    logit,h,edges = model(g, nt)
                    p = torch.sigmoid(logit)

                    # strict labels
                    y,sinks,paths = labels_from_graph(g, nt)
                    if y is None and sinks:
                        y=torch.zeros(g[nt].num_nodes, dtype=p.dtype, device=dev)
                        for s in sinks:
                            if 0<=s<y.numel(): y[s]=1.0
                    if y is None and not paths:
                        pbar.update(1); continue
                    if y is None:
                        y=torch.zeros(g[nt].num_nodes, dtype=p.dtype, device=dev)

                    # imbalance
                    keep_mask=hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                    y_m, p_m = y[keep_mask].to(torch.float32), p[keep_mask].to(torch.float32)
                    w=class_weights(y_m)
                    loss_node = focal(p_m, y_m, w)

                    # edge participation (paths only)
                    loss_edge = 0.0
                    if paths:
                        node_mask=edge_mask_from_paths(paths, g[nt].num_nodes).to(dev)
                        part_labels=[]; part_scores=[]
                        for r,ei in edges.items():
                            if ei is None or ei.numel()==0: continue
                            u,v=ei
                            m = node_mask[u] & node_mask[v]
                            part_labels.append(m.float())
                            score = torch.sigmoid(model.edge_head(torch.cat([h[u],h[v]],-1)).squeeze(-1))
                            part_scores.append(score.float())
                        if part_scores:
                            part_scores=torch.cat(part_scores,0); part_labels=torch.cat(part_labels,0)
                            loss_edge=F.binary_cross_entropy(part_scores, part_labels)

                    # next-hop ranking + monotonicity
                    loss_nh=0.0; loss_mono=0.0
                    if paths:
                        for path in paths:
                            idx=torch.tensor(path, dtype=torch.long, device=dev)
                            if idx.numel()<2: continue
                            l=logit[idx]
                            loss_mono += sum(F.relu(l[i]-l[i+1]) for i in range(len(l)-1))/max(1,(len(l)-1))
                            for i_step in range(len(idx)-1):
                                u=int(idx[i_step]); v=int(idx[i_step+1])
                                pos=model.next_hop_head(torch.cat([h[u],h[v]],-1))
                                negs=[]
                                for r,ei in edges.items():
                                    if ei is None or ei.numel()==0: continue
                                    vs=ei[1][(ei[0]==u)]
                                    if vs.numel()>0:
                                        perm=torch.randperm(vs.numel(), device=dev)[:6]
                                        for vv in vs[perm]:
                                            if int(vv)!=v:
                                                negs.append(model.next_hop_head(torch.cat([h[u],h[int(vv)]],-1)))
                                if negs:
                                    neg=torch.stack(negs).squeeze(-1)
                                    loss_nh += F.relu(1.0 - pos.squeeze(-1) + neg).mean()
                        loss_mono = loss_mono / max(1,len(paths))

                    loss = loss_node + 0.25*loss_edge + 0.20*loss_nh + 0.20*loss_mono

                opt.zero_grad(set_to_none=True)
                if USE_AMP:
                    scaler.scale(loss).backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()

                steps+=1; loss_avg+=float(loss.detach().cpu())
                if (i % CLEAR_CACHE_EVERY)==0 and DEVICE.type=="cuda":
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    if DEVICE.type=="cuda": torch.cuda.empty_cache()
                else:
                    raise
            pbar.set_postfix(loss=f"{(loss_avg/max(1,steps)):.4f}"); pbar.update(1)
        pbar.close()

    for ep in range(1,EPOCHS+1):
        train_epoch(ep)

    # ----- eval with per-source thresholds (valid) -----
    def eval_split(name, ds):
        if len(ds)==0:
            return {"split":name,"overall":{"precision":None,"recall":None,"f1":0.0},
                    "thresholds":{},"n_graphs_used":0,"per_source":{},
                    "CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        report={"split":name,"per_source":{}}
        THRS={"default": (0.25 if name=="valid" else 0.05)}
        # Calibrate on valid only
        if name=="valid":
            by_src_p=defaultdict(list); by_src_y=defaultdict(list)
            for g in ds:
                nt=primary_node_type(g) or "node"
                if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
                y,_,paths = labels_from_graph(g, nt)
                keep=[]
                if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
                elif paths: 
                    for pth in paths: keep += pth
                g = graph_shrink(g, nt, must_keep=keep)
                dev=next(model.parameters()).device
                if hasattr(g[nt],"x_text"): g[nt].x_text=g[nt].x_text.to(dev)
                if hasattr(g[nt],"x"):      g[nt].x=g[nt].x.to(dev)
                for et in g.edge_types:
                    if hasattr(g[et],"edge_index"): g[et].edge_index=g[et].edge_index.to(dev)
                with torch.no_grad():
                    logit,_,_=model(g, nt); p=torch.sigmoid(logit).detach().cpu()
                if (y is None) and paths:
                    y=torch.zeros(g[nt].num_nodes); y[edge_mask_from_paths(paths, g[nt].num_nodes)]=1.0
                if y is None: 
                    continue
                src=getattr(g[nt],"source","default")
                by_src_p[src].append(p); by_src_y[src].append(y.cpu())
            for s in by_src_p:
                P=torch.cat(by_src_p[s]); Y=torch.cat(by_src_y[s]).float()
                best_thr,best_f1=0.5,-1
                for thr in torch.linspace(0.05,0.95,19):
                    m=metrics_at(P,Y,float(thr))
                    if m["f1"]>best_f1: best_f1=m["f1"]; best_thr=float(thr)
                THRS[s]=best_thr

        used=0; ccs_vals=[]; cfam_vals=[]; per_src=report["per_source"]
        for g in ds:
            nt=primary_node_type(g) or "node"
            if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
            y,sinks,paths = labels_from_graph(g, nt)
            keep=[]
            if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
            elif paths: 
                for pth in paths: keep += pth
            elif sinks:
                keep += sinks
            g = graph_shrink(g, nt, must_keep=keep)
            dev=next(model.parameters()).device
            if hasattr(g[nt],"x_text"): g[nt].x_text=g[nt].x_text.to(dev)
            if hasattr(g[nt],"x"):      g[nt].x=g[nt].x.to(dev)
            for et in g.edge_types:
                if hasattr(g[et],"edge_index"): g[et].edge_index=g[et].edge_index.to(dev)

            with torch.no_grad():
                logit,h,edges = model(g, nt)
                p = torch.sigmoid(logit)
            if (y is None) and paths:
                y=torch.zeros(g[nt].num_nodes, device=p.device)
                y[edge_mask_from_paths(paths, g[nt].num_nodes).to(p.device)]=1.0
            if y is None and not (sinks or paths):
                continue
            if y is None:
                y=torch.zeros(g[nt].num_nodes, device=p.device)
                for s in sinks:
                    if 0<=s<y.numel(): y[s]=1.0
            src=getattr(g[nt],"source","default")
            thr=THRS.get(src, THRS.get("default",0.5))
            m=metrics_at(p.detach().cpu(), y.detach().cpu(), thr)
            per_src.setdefault(src, {"n_graphs":0,"metrics":[]})
            per_src[src]["n_graphs"] += 1
            per_src[src]["metrics"].append(m)
            used += 1

            # CCS/CFAM (strict)
            causal = list(getattr(g[nt],"causal_nodes", [])) if hasattr(g[nt],"causal_nodes") else []
            spurious = list(getattr(g[nt],"spurious_nodes", [])) if hasattr(g[nt],"spurious_nodes") else []
            if not causal:
                if paths:
                    causal = sorted(list({u for path in paths for u in path}))
                else:
                    roots=[]
                    if (nt,'ARG2PARAM',nt) in g.edge_types:
                        ei=get_edge_index(g,(nt,'ARG2PARAM',nt))
                        if ei.numel()>0: roots = torch.unique(ei[0]).tolist()
                    sinks_idx = (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
                    if roots and sinks_idx:
                        causal = program_slice(g, nt, roots, sinks_idx)
            if not spurious and causal:
                all_nodes=set(range(g[nt].num_nodes))
                spurious = sorted(list(all_nodes - set(causal)))
            if causal:
                ccs = compute_CCS(model, g, nt, causal)
                if ccs is not None: ccs_vals.append(ccs)
                # CFAM grad pass
                model.zero_grad(set_to_none=True)
                logit2,h2,_ = model(g, nt)  # grad-enabled
                logit2.sum().backward(retain_graph=False)
                cf = attribution_cf(h2, torch.tensor(causal, dtype=torch.long, device=h2.device),
                                    torch.tensor(spurious, dtype=torch.long, device=h2.device) if spurious else None)
                cfam_vals.append(cf)
                model.zero_grad(set_to_none=True)

        overall={"precision":0.0,"recall":0.0,"f1":0.0}
        if per_src:
            for s in per_src:
                ms=per_src[s]["metrics"]
                avg={k: sum(m[k] for m in ms)/len(ms) for k in ms[0]}
                per_src[s]["avg"]=avg
                for k in ("precision","recall","f1"): overall[k]+=avg[k]
            n=len(per_src)
            for k in overall: overall[k]/=n
        return {
            "split":name,
            "overall":overall,
            "thresholds":THRS,
            "n_graphs_used":used,
            "per_source":per_src,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count":len(ccs_vals),
            "CFAM_count":len(cfam_vals)
        }

    rep_train = eval_split("train", ds_tr)
    rep_valid = eval_split("valid", ds_va)
    rep_test  = eval_split("test",  ds_te)

    # demo beam (multi-root)
    demo={"meta":{"split":None,"graph_path":None},"paths":[]}
    ds_demo = ds_va if len(ds_va)>0 else (ds_tr if len(ds_tr)>0 else [])
    if len(ds_demo)>0:
        g0 = ds_demo[0]; nt0=primary_node_type(g0) or "node"
        if ADD_SUMMARY_EDGES: add_thin_dfg(g0, nt0)
        g0 = graph_shrink(g0, nt0, must_keep=[])
        dev=next(model.parameters()).device
        if hasattr(g0[nt0],"x_text"): g0[nt0].x_text=g0[nt0].x_text.to(dev)
        if hasattr(g0[nt0],"x"):      g0[nt0].x=g0[nt0].x.to(dev)
        for et in g0.edge_types:
            if hasattr(g0[et],"edge_index"): g0[et].edge_index=g0[et].edge_index.to(dev)
        with torch.no_grad():
            logit0,h0,edges0=model(g0, nt0); p0=torch.sigmoid(logit0)
            edge_scores0=model.rel_gate(h0,{r:edges0[r] for r in edges0.keys()})
        y0,_,paths0 = labels_from_graph(g0, nt0)
        if (y0 is None) and paths0:
            y0=torch.zeros(g0[nt0].num_nodes); y0[edge_mask_from_paths(paths0, g0[nt0].num_nodes)]=1.0
        seeds = (y0>0.5).nonzero(as_tuple=False).view(-1).tolist() if (y0 is not None and (y0>0.5).any()) else torch.topk(p0, k=min(12,p0.numel())).indices.tolist()
        beam_out = run_beam(p0, edge_scores0, edges0, seeds, 32, 6, 0.7)
        demo["meta"]["split"]=("valid" if len(ds_va)>0 else "train")
        demo["meta"]["graph_path"]=str(ds_demo.paths[0])
        demo["paths"]=beam_out[:10]

    # save artifacts
    torch.save({"model":model.state_dict(), "config":{"hidden":HIDDEN,"layers":LAYERS,"relations":RELATIONS}}, OUT_ROOT/"model.pt")
    report={
        "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
                  "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE.type,"dataset_weights":DATASET_WEIGHTS,
                  "relations":RELATIONS,"add_summary_edges":ADD_SUMMARY_EDGES,
                  "label_keys":LABEL_KEYS,"sink_keys":SINK_KEYS,"path_keys":PATH_KEYS},
        "thresholds_by_source": rep_valid.get("thresholds",{}),
        "reports":{"train":rep_train,"valid":rep_valid,"test":rep_test},
        "demo":demo,
        "notes":{
            "interprocedural":"DFG/CFG/CALL/ARG2PARAM/RET2CALL/RET2LHS (+DFG_THIN optional); multi-root beam.",
            "beam_guidance":"learned per-relation gates (RelGate); no fixed ordering.",
            "objective":"node (focal+class-weights+hard-neg), edge participation, next-hop ranking, monotonicity.",
            "CCS":"p(x) vs p(do(x')) by zeroing strict causal-slice (annotated or sliced roots→sinks).",
            "CFAM":"grad-norm attribution mass on causal / (causal+spurious).",
            "calibration":"per-source thresholds tuned on valid; used in summaries.",
            "provenance":"source field if present; else 'default'."
        }
    }
    (OUT_ROOT/"report.json").write_text(json.dumps(report, indent=2))
    print("Saved to:", OUT_ROOT)

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:")
    traceback.print_exc()


[env] torch=2.4.1+cu121 device=cuda
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert


Saved to: out\cvul\1760395552


In [9]:
# %% Causal-Vul — inter-proc + CCS/CFAM + sidecar labels (no heuristics)
import os, json, time, math, gc, random, traceback, warnings
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed") from e

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`")

# ----------- config -----------
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"

OUT_ROOT = Path("out/cvul") / str(int(time.time()))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (DEVICE.type == "cuda")

EPOCHS = 10
BATCH_SIZE = 1
HIDDEN = 64
LAYERS = 3
LR = 2e-3
WD = 1e-4
NEG_POS_RATIO = 20

RELATIONS = ["ARG2PARAM","CALL","CFG","DFG","RET2CALL","RET2LHS"]
ADD_SUMMARY_EDGES = True

MAX_NODES_FOR_GPU = 3500
MAX_EDGES_FOR_GPU = 12000
CLEAR_CACHE_EVERY = 12

# keys we will honor IN GRAPH
LABEL_KEYS = ["y","label","labels","vulnerable","is_sink","target","targets"]
SINK_KEYS  = ["sinks","sink_nodes"]
PATH_KEYS  = ["vulnerable_paths","paths_idx"]

# keys we will honor IN SIDECAR JSON (same-stem .json by each .pt)
SIDECAR_LABEL_IDX_KEYS = ["labels_idx","label_idx","y_idx","pos_idx","positive_nodes","vulnerable_nodes","pos_nodes"]
SIDECAR_SINK_KEYS      = ["sinks","sink_nodes"]
SIDECAR_PATH_KEYS      = ["paths","paths_idx","vulnerable_paths"]
SIDECAR_SOURCE_KEYS    = ["source","dataset","origin"]

DATASET_WEIGHTS = {"default":1.0}

def set_seed(s=23):
    random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def primary_node_type(g: "HeteroData"):
    for nt in g.node_types:
        st=g[nt]
        if hasattr(st,"num_nodes") and (hasattr(st,"x") or hasattr(st,"x_text") or any(hasattr(st,k) for k in LABEL_KEYS)):
            return nt
    return g.node_types[0] if g.node_types else None

def get_edge_index(g, et):
    st=g[et]
    if hasattr(st,"edge_index") and st.edge_index is not None:
        return st.edge_index
    if hasattr(st,"adj_t") and st.adj_t is not None:
        row,col,_ = st.adj_t.coo()
        return torch.stack([col,row],0)
    return torch.zeros((2,0), dtype=torch.long)

def add_thin_dfg(g, nt, rel="DFG", rel_name="DFG_THIN", max_new=15000):
    et=(nt,rel,nt)
    if et not in g.edge_types: return
    ei=get_edge_index(g, et)
    if ei.numel()==0: return
    src,dst=ei
    adj=defaultdict(list)
    for u,v in zip(src.tolist(), dst.tolist()):
        if len(adj[u])<64: adj[u].append(v)
    s2,d2=[],[]
    for u,nbs in adj.items():
        for v in nbs[:32]:
            for w in adj.get(v,[])[:16]:
                if u!=w:
                    s2.append(u); d2.append(w)
                    if len(s2)>=max_new: break
            if len(s2)>=max_new: break
        if len(s2)>=max_new: break
    if s2:
        g[(nt,rel_name,nt)].edge_index = torch.tensor([s2,d2], dtype=torch.long)

class GraphDir(Dataset):
    def __init__(self, folder):
        self.paths=sorted(Path(folder).glob("*.pt"))
        print(f"[DATA] {len(self.paths)} graphs in {folder}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p=self.paths[i]
        try:
            g=torch.load(p, map_location="cpu")
        except Exception:
            from torch.serialization import add_safe_globals
            from torch_geometric.data.storage import BaseStorage
            add_safe_globals([BaseStorage])
            g=torch.load(p, map_location="cpu")
        g.__dict__["_path"]=str(p)
        return g

def load_sidecar_dict(pt_path: str):
    p = Path(pt_path)
    cand = [p.with_suffix(".json"), p.with_suffix(".JSON"), p.with_suffix(".Json")]
    for c in cand:
        if c.exists():
            try:
                return json.loads(c.read_text())
            except Exception:
                return None
    return None

def apply_sidecar(g:"HeteroData", nt:str):
    sd = load_sidecar_dict(g.__dict__.get("_path",""))
    if not sd: return
    st=g[nt]
    # source
    for k in SIDECAR_SOURCE_KEYS:
        if k in sd:
            setattr(st,"source", sd[k])
            break
    # labels by indices
    y_idx=None
    for k in SIDECAR_LABEL_IDX_KEYS:
        if k in sd and isinstance(sd[k], (list,tuple)):
            y_idx=[int(v) for v in sd[k]]
            break
    if y_idx is not None and hasattr(st,"num_nodes"):
        y=torch.zeros(st.num_nodes, dtype=torch.float32)
        for v in y_idx:
            if 0<=v<st.num_nodes: y[v]=1.0
        setattr(st, "labels", y)
    # sinks
    for k in SIDECAR_SINK_KEYS:
        if k in sd and isinstance(sd[k], (list,tuple)):
            setattr(st, "sinks", [int(v) for v in sd[k]])
            break
    # paths
    for k in SIDECAR_PATH_KEYS:
        if k in sd and isinstance(sd[k], (list,tuple)) and sd[k] and isinstance(sd[k][0], (list,tuple)):
            setattr(st, "paths_idx", [[int(v) for v in path] for path in sd[k]])
            break
    # causal/spurious sets if provided
    for nm in ("causal_nodes","spurious_nodes"):
        if nm in sd and isinstance(sd[nm], (list,tuple)):
            setattr(st, nm, [int(v) for v in sd[nm]])

# ---------- imbalance/noise ----------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.97, gamma=2.0):
        super().__init__(); self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p, y, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps); pt=torch.where(y>0,p,1-p)
        w = (self.alpha_pos*(y>0).float() + (1-self.alpha_pos)*(y<=0).float())
        if weight is not None: w = w*weight
        return (-((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w).mean()

def class_weights(y):
    pos=y.sum(); neg=y.numel()-pos
    w=torch.ones_like(y)
    if pos>0: w[y>0.5]=(neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p,y,max_ratio=20):
    y=y.view(-1); p=p.view(-1)
    pos_idx=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx=(y<=0.5).nonzero(as_tuple=False).view(-1)
    k=int(min(neg_idx.numel(), max_ratio*max(1,pos_idx.numel())))
    if k<=0 or neg_idx.numel()==0:
        return (y>0.5)
    topk=torch.topk(p[neg_idx], k=k).indices
    keep_neg=neg_idx[topk]
    mask=torch.zeros_like(y, dtype=torch.bool)
    if pos_idx.numel()>0: mask[pos_idx]=True
    if keep_neg.numel()>0: mask[keep_neg]=True
    return mask

# ---------- model ----------
class RelGate(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__(); self.relations=relations
        self.emb=nn.Embedding(len(relations), hidden)
        self.log_prior=nn.Parameter(torch.zeros(len(relations)))
        self.edge_mlp=nn.Sequential(nn.Linear(hidden*3,hidden), nn.SiLU(), nn.Linear(hidden,1))
    def forward(self, h, edge_index_by_rel):
        out={}
        for i,r in enumerate(self.relations):
            ei=edge_index_by_rel.get(r,None)
            if ei is None or ei.numel()==0:
                out[r]=h.new_empty((0,)); continue
            u,v=ei[0],ei[1]
            te=self.emb.weight[i].view(1,-1).expand(u.numel(),-1).to(h.dtype)
            score=self.edge_mlp(torch.cat([h[u],h[v],te],-1)).squeeze(-1)
            out[r]=score+self.log_prior[i].to(score.dtype)
        return out

class Block(nn.Module):
    def __init__(self, hidden, relations, drop=0.1):
        super().__init__(); self.relations=relations
        self.msg=nn.ModuleDict({r: nn.Linear(hidden,hidden) for r in relations})
        self.self_lin=nn.Linear(hidden,hidden)
        self.norm=nn.LayerNorm(hidden); self.drop=nn.Dropout(drop)
        self.skip_alpha=nn.Parameter(torch.tensor(0.5))
    def forward(self,h,edge_index_by_rel):
        dt, dv = h.dtype, h.device
        out = self.self_lin(h).to(dt)
        agg = torch.zeros_like(h, dtype=dt, device=dv)
        deg = torch.zeros(h.size(0), device=dv, dtype=dt)
        for r in self.relations:
            ei=edge_index_by_rel.get(r,None)
            if ei is None or ei.numel()==0: continue
            u,v=ei[0],ei[1]
            m=self.msg[r](h[u]).to(dt)
            agg.index_add_(0, v, m)
            deg.index_add_(0, v, torch.ones(v.numel(), device=dv, dtype=dt))
        deg=deg.clamp(min=1).unsqueeze(-1)
        out=out + agg/deg
        out=self.drop(out)
        ka=self.skip_alpha.to(dt)
        out = ka*h + (1-ka)*out
        return self.norm(F.silu(out).to(dt))

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=RELATIONS):
        super().__init__()
        self.hidden=hidden; self.relations=relations
        self.proj_pool=nn.ModuleDict()
        self.blocks=nn.ModuleList([Block(hidden, relations) for _ in range(layers)])
        self.node_head=nn.Sequential(nn.Linear(hidden,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.edge_head=nn.Sequential(nn.Linear(hidden*2,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.next_hop_head=nn.Sequential(nn.Linear(hidden*2,hidden), nn.SiLU(), nn.Linear(hidden,1))
        self.rel_gate=RelGate(hidden, relations)
    def _get_proj(self, d, device, dtype):
        key=f"in_{d}"
        if key not in self.proj_pool:
            lin=nn.Linear(d, self.hidden)
            nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))
            self.proj_pool[key]=lin.to(device=device, dtype=dtype)
        else:
            mod=self.proj_pool[key]
            if next(mod.parameters()).device != device or next(mod.parameters()).dtype != dtype:
                self.proj_pool[key]=mod.to(device=device, dtype=dtype)
        return self.proj_pool[key]
    def encode(self, x):
        proj = self._get_proj(x.size(1), x.device, x.dtype)
        return F.relu(proj(x))
    def forward(self, g:"HeteroData", nt:str):
        xs=[]
        if hasattr(g[nt],"x_text"): xs.append(g[nt].x_text.float().to(next(self.parameters()).dtype))
        if hasattr(g[nt],"x"):      xs.append(g[nt].x.float().to(next(self.parameters()).dtype))
        if not xs: raise RuntimeError("No features (x/x_text) on node store")
        x = xs[0] if len(xs)==1 else torch.cat(xs,1)
        h = self.encode(x)
        e_by_rel={}
        avail=set(et[1] for et in g.edge_types)
        for r in (self.relations + (["DFG_THIN"] if "DFG_THIN" in avail else [])):
            et=(nt,r,nt)
            if et in g.edge_types:
                e_by_rel[r]=get_edge_index(g, et)
        for blk in self.blocks:
            h = blk(h, e_by_rel)
        logit=self.node_head(h).squeeze(-1)
        return logit, h, e_by_rel

# ---------- helpers ----------
def graph_shrink(g, nt, must_keep=None):
    N=g[nt].num_nodes
    tot=sum(int(get_edge_index(g,et).size(1)) for et in g.edge_types)
    if (N<=MAX_NODES_FOR_GPU) and (tot<=MAX_EDGES_FOR_GPU): return g
    keep=set(must_keep or [])
    avail=[et[1] for et in g.edge_types]
    for r in RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in avail) else []):
        et=(nt,r,nt)
        if et in g.edge_types:
            ei=get_edge_index(g, et)
            if ei.numel()>0:
                s,d=ei
                if keep & set(s.tolist()) or keep & set(d.tolist()):
                    keep |= set(s.tolist()); keep |= set(d.tolist())
    tgt=min(N, MAX_NODES_FOR_GPU)
    rnd=list(range(N)); random.shuffle(rnd)
    for u in rnd:
        if len(keep)>=tgt: break
        keep.add(u)
    keep=sorted(keep); imap={old:i for i,old in enumerate(keep)}
    g2=HeteroData()
    st=g[nt]
    if hasattr(st,"x_text"): g2[nt].x_text=st.x_text[keep]
    if hasattr(st,"x"):      g2[nt].x=st.x[keep]
    for k in LABEL_KEYS + ["nid","source","causal_nodes","spurious_nodes","sinks","paths_idx"]:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v) and v.numel()==st.num_nodes:
                g2[nt][k]=v[keep]
            else:
                g2[nt][k]=v
    g2[nt].num_nodes=len(keep)
    for et in g.edge_types:
        ei=get_edge_index(g, et)
        if ei.numel()==0:
            g2[et].edge_index=ei; continue
        s,d=ei
        sf,df=[],[]
        for u,v in zip(s.tolist(), d.tolist()):
            if u in imap and v in imap:
                sf.append(imap[u]); df.append(imap[v])
        g2[et].edge_index = (torch.tensor([sf,df], dtype=torch.long) if sf else torch.zeros((2,0), dtype=torch.long))
    return g2

def labels_from_graph(g, nt):
    # apply sidecar FIRST so fields are present
    apply_sidecar(g, nt)
    y=None; sinks=[]; paths=[]
    st=g[nt]
    # exact vector y
    for k in LABEL_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v) and v.numel()==st.num_nodes:
                y=v.float().view(-1); break
    # if vector not provided but indices were put into st.labels from sidecar
    if y is None and hasattr(st,"labels") and torch.is_tensor(st.labels) and st.labels.numel()==st.num_nodes:
        y=st.labels.float().view(-1)
    # sinks
    for k in SINK_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if torch.is_tensor(v): sinks=v.view(-1).tolist()
            elif isinstance(v,(list,tuple)): sinks=list(map(int,v))
            break
    # paths
    for k in PATH_KEYS:
        if hasattr(st,k):
            v=getattr(st,k)
            if isinstance(v,list) and v and isinstance(v[0],(list,tuple)):
                paths=[[int(ix) for ix in path] for path in v]
            break
    return y,sinks,paths

def build_adj(edge_index_by_rel):
    adj={r: defaultdict(list) for r in edge_index_by_rel.keys()}
    for r,ei in edge_index_by_rel.items():
        if ei is None or ei.numel()==0: continue
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            if len(adj[r][u])<64: adj[r][u].append(v)
    return adj

def program_slice(g, nt, roots, sinks, allowed=None):
    allowed = allowed or RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in [et[1] for et in g.edge_types]) else [])
    adj=defaultdict(list); radj=defaultdict(list)
    for r in allowed:
        et=(nt,r,nt)
        if et not in g.edge_types: continue
        ei=get_edge_index(g, et)
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            adj[u].append(v); radj[v].append(u)
    causal=set()
    for s in roots:
        q=deque([s]); vis=set([s])
        while q:
            u=q.popleft(); causal.add(u)
            for v in adj.get(u,[]):
                if v not in vis: vis.add(v); q.append(v)
    keep_back=set(); dq=deque([t for t in sinks])
    while dq:
        v=dq.popleft(); keep_back.add(v)
        for u in radj.get(v,[]):
            if u not in keep_back: keep_back.add(u); dq.append(u)
    return sorted(list(causal & keep_back))

@dataclass
class BeamPath:
    score: float
    nodes: list

def run_beam(p, edge_scores, edges, seeds, beam_width=32, max_hops=6, alpha_node=0.7):
    rels=sorted(set(edges.keys()) & set(edge_scores.keys()))
    adj=build_adj({r:edges[r] for r in rels})
    def clog(z): return float(torch.log(z.clamp(1e-9,1-1e-9)))
    if not seeds:
        K=min(8, p.numel()); seeds=torch.topk(p, k=K).indices.tolist()
    beams=[BeamPath(clog(p[s]), [int(s)]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u=b.nodes[-1]
            for r in rels:
                for v in adj[r].get(u,[]):
                    ei=edges[r]; m=(ei[0]==u)&(ei[1]==v)
                    es=float(edge_scores[r][m].max().item()) if m.any() else 0.0
                    cand.append(BeamPath(b.score + alpha_node*clog(p[v]) + (1-alpha_node)*es, b.nodes+[v]))
        cand.sort(key=lambda x: x.score, reverse=True)
        beams=cand[:beam_width]; finished.extend(beams[:beam_width//2])
        if not beams: break
    uniq=[]; seen=set()
    for bp in sorted(finished, key=lambda x: x.score, reverse=True):
        key=tuple(bp.nodes[-min(4,len(bp.nodes)):])
        if key in seen: continue
        seen.add(key); uniq.append({"score":bp.score,"nodes":bp.nodes})
        if len(uniq)>=beam_width: break
    return uniq

def metrics_at(p,y,thr):
    pred=(p>=thr).float()
    tp=(pred*y).sum().item(); fp=(pred*(1-y)).sum().item(); fn=((1-pred)*y).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def edge_mask_from_paths(paths, N):
    m=torch.zeros(N, dtype=torch.bool)
    if not paths: return m
    for path in paths:
        for j in path:
            if 0<=j<N: m[j]=True
    return m

def attribution_cf(h, causal_idx, spurious_idx=None):
    g = h.grad.detach().abs() if (h.grad is not None) else h.detach().abs()
    mass = g.norm(p=2, dim=-1)
    causal_sum = mass[causal_idx].sum().item() if len(causal_idx)>0 else 0.0
    if spurious_idx is None:
        total = mass.sum().item() + 1e-9
        return causal_sum/total
    sp_sum = mass[spurious_idx].sum().item() if len(spurious_idx)>0 else 0.0
    return causal_sum / (causal_sum + sp_sum + 1e-9)

def compute_CCS(model, g, nt, causal_nodes):
    with torch.no_grad():
        logit,_,_ = model(g, nt); p = torch.sigmoid(logit)
    if not causal_nodes: return None
    g2 = g.clone()
    if hasattr(g2[nt],"x_text"): g2[nt].x_text = g2[nt].x_text.clone()
    if hasattr(g2[nt],"x"):      g2[nt].x      = g2[nt].x.clone()
    idx = torch.tensor(causal_nodes, dtype=torch.long)
    if hasattr(g2[nt],"x_text"): g2[nt].x_text[idx]=0.0
    if hasattr(g2[nt],"x"):      g2[nt].x[idx]=0.0
    with torch.no_grad():
        logit2,_,_ = model(g2, nt); p2 = torch.sigmoid(logit2)
    return float((p - p2).pow(2).mean().item())

# --------------- train & eval ---------------
def train_and_eval():
    set_seed(23)
    print(f"[env] torch={torch.__version__} device={DEVICE.type}")
    (OUT_ROOT/"cfg.json").write_text(json.dumps({
        "epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
        "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE.type,"relations":RELATIONS,
        "add_summary_edges":ADD_SUMMARY_EDGES,
        "label_keys":LABEL_KEYS,"sink_keys":SINK_KEYS,"path_keys":PATH_KEYS
    }, indent=2))

    ds_tr, ds_va, ds_te = GraphDir(TRAIN_DIR), GraphDir(VALID_DIR), GraphDir(TEST_DIR)
    # probe
    probe=None
    for ds in (ds_tr, ds_va, ds_te):
        if len(ds)>0: probe=ds[0]; break
    if probe is None:
        print("No graphs found."); return
    nt_probe = primary_node_type(probe) or "node"
    if ADD_SUMMARY_EDGES: add_thin_dfg(probe, nt_probe)
    relations_eff = RELATIONS + (["DFG_THIN"] if ("DFG_THIN" in [et[1] for et in probe.edge_types]) else [])

    model=CausalVulNet(hidden=HIDDEN, layers=LAYERS, relations=relations_eff).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler=torch.amp.GradScaler('cuda', enabled=USE_AMP)
    focal=FocalBCELoss()

    dl_tr=DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda xs: xs[0])
    dl_va=DataLoader(ds_va, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0])
    dl_te=DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0])

    def move_to_device(g, nt, dev):
        if hasattr(g[nt],"x_text"): g[nt].x_text=g[nt].x_text.to(dev)
        if hasattr(g[nt],"x"):      g[nt].x=g[nt].x.to(dev)
        for et in g.edge_types:
            if hasattr(g[et],"edge_index"): g[et].edge_index=g[et].edge_index.to(dev)

    def train_epoch(ep):
        model.train(); loss_avg=0.0; steps=0
        pbar=tqdm(total=len(dl_tr), desc=f"[train] epoch {ep}", leave=False)
        for i,g in enumerate(dl_tr,1):
            try:
                nt=primary_node_type(g) or "node"
                if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
                # derive labels/paths/sinks (apply sidecar inside)
                y,sinks,paths = labels_from_graph(g, nt)
                # if nothing available, skip this graph
                if (y is None) and not sinks and not paths:
                    pbar.update(1); continue
                # keep set for shrink
                keep=[]
                if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
                if paths: 
                    for pth in paths: keep += pth
                if sinks: keep += sinks
                g = graph_shrink(g, nt, must_keep=keep)
                move_to_device(g, nt, next(model.parameters()).device)

                with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                    logit,h,edges = model(g, nt)
                    p = torch.sigmoid(logit)

                    # strict labels again (after shrink)
                    y,sinks,paths = labels_from_graph(g, nt)
                    if (y is None) and paths:
                        y=torch.zeros(g[nt].num_nodes, dtype=p.dtype, device=p.device)
                        y[edge_mask_from_paths(paths, g[nt].num_nodes).to(p.device)]=1.0
                    if (y is None) and sinks:
                        y=torch.zeros(g[nt].num_nodes, dtype=p.dtype, device=p.device)
                        for s in sinks:
                            if 0<=s<y.numel(): y[s]=1.0
                    if y is None:
                        pbar.update(1); continue

                    keep_mask=hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                    y_m, p_m = y[keep_mask].to(torch.float32), p[keep_mask].to(torch.float32)
                    w=class_weights(y_m)
                    loss_node=focal(p_m, y_m, w)

                    loss_edge=0.0
                    if paths:
                        node_m=edge_mask_from_paths(paths, g[nt].num_nodes).to(p.device)
                        labs=[]; scr=[]
                        for r,ei in edges.items():
                            if ei is None or ei.numel()==0: continue
                            u,v=ei
                            m = node_m[u] & node_m[v]
                            labs.append(m.float())
                            scr.append(torch.sigmoid(model.edge_head(torch.cat([h[u],h[v]],-1)).squeeze(-1)).float())
                        if scr:
                            scr=torch.cat(scr,0); labs=torch.cat(labs,0)
                            loss_edge=F.binary_cross_entropy(scr, labs)

                    loss_nh=0.0; loss_mono=0.0
                    if paths:
                        for path in paths:
                            idx=torch.tensor(path, dtype=torch.long, device=p.device)
                            if idx.numel()<2: continue
                            l=logit[idx]
                            loss_mono += sum(F.relu(l[i]-l[i+1]) for i in range(len(l)-1))/max(1,(len(l)-1))
                            for i_step in range(len(idx)-1):
                                u=int(idx[i_step]); v=int(idx[i_step+1])
                                pos=model.next_hop_head(torch.cat([h[u],h[v]],-1))
                                negs=[]
                                for r,ei in edges.items():
                                    if ei is None or ei.numel()==0: continue
                                    vs=ei[1][(ei[0]==u)]
                                    if vs.numel()>0:
                                        perm=torch.randperm(vs.numel(), device=p.device)[:6]
                                        for vv in vs[perm]:
                                            if int(vv)!=v:
                                                negs.append(model.next_hop_head(torch.cat([h[u],h[int(vv)]],-1)))
                                if negs:
                                    neg=torch.stack(negs).squeeze(-1)
                                    loss_nh += F.relu(1.0 - pos.squeeze(-1) + neg).mean()
                        loss_mono = loss_mono / max(1,len(paths))

                    loss = loss_node + 0.25*loss_edge + 0.20*loss_nh + 0.20*loss_mono

                opt.zero_grad(set_to_none=True)
                if USE_AMP:
                    scaler.scale(loss).backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()

                steps+=1; loss_avg+=float(loss.detach().cpu())
                if (i % CLEAR_CACHE_EVERY)==0 and DEVICE.type=="cuda":
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    if DEVICE.type=="cuda": torch.cuda.empty_cache()
                else:
                    raise
            pbar.set_postfix(loss=f"{(loss_avg/max(1,steps)):.4f}"); pbar.update(1)
        pbar.close()

    for ep in range(1,EPOCHS+1):
        train_epoch(ep)

    # ----- eval with per-source thresholds on valid -----
    def eval_split(name, ds):
        if len(ds)==0:
            return {"split":name,"overall":{"precision":None,"recall":None,"f1":0.0},
                    "thresholds":{},"n_graphs_used":0,"per_source":{},
                    "CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        report={"split":name,"per_source":{}}
        THRS={"default": (0.25 if name=="valid" else 0.05)}
        # Calibrate on valid
        if name=="valid":
            by_src_p=defaultdict(list); by_src_y=defaultdict(list)
            for g in ds:
                nt=primary_node_type(g) or "node"
                if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
                y,sinks,paths = labels_from_graph(g, nt)
                if (y is None) and not sinks and not paths:
                    continue
                keep=[]
                if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
                if paths:
                    for pth in paths: keep += pth
                if sinks: keep += sinks
                g = graph_shrink(g, nt, must_keep=keep)
                move_to_device(g, nt, next(model.parameters()).device)
                with torch.no_grad():
                    logit,_,_=model(g, nt); p=torch.sigmoid(logit).detach().cpu()
                if (y is None) and paths:
                    y=torch.zeros(g[nt].num_nodes); y[edge_mask_from_paths(paths, g[nt].num_nodes)]=1.0
                if (y is None) and sinks:
                    y=torch.zeros(g[nt].num_nodes); 
                    for s in sinks:
                        if 0<=s<y.numel(): y[s]=1.0
                if y is None:
                    continue
                src=getattr(g[nt],"source","default")
                by_src_p[src].append(p); by_src_y[src].append(y.cpu())
            for s in by_src_p:
                P=torch.cat(by_src_p[s]); Y=torch.cat(by_src_y[s]).float()
                best_thr,best_f1=0.5,-1
                for thr in torch.linspace(0.05,0.95,19):
                    m=metrics_at(P,Y,float(thr))
                    if m["f1"]>best_f1: best_f1=m["f1"]; best_thr=float(thr)
                THRS[s]=best_thr

        used=0; ccs_vals=[]; cfam_vals=[]; per_src=report["per_source"]
        for g in ds:
            nt=primary_node_type(g) or "node"
            if ADD_SUMMARY_EDGES: add_thin_dfg(g, nt)
            y,sinks,paths = labels_from_graph(g, nt)
            if (y is None) and not sinks and not paths:
                continue
            keep=[]
            if y is not None: keep += (y>0.5).nonzero(as_tuple=False).view(-1).tolist()
            if paths:
                for pth in paths: keep += pth
            if sinks: keep += sinks
            g = graph_shrink(g, nt, must_keep=keep)
            move_to_device(g, nt, next(model.parameters()).device)

            with torch.no_grad():
                logit,h,edges = model(g, nt)
                p = torch.sigmoid(logit)
            if (y is None) and paths:
                y=torch.zeros(g[nt].num_nodes, device=p.device)
                y[edge_mask_from_paths(paths, g[nt].num_nodes).to(p.device)]=1.0
            if (y is None) and sinks:
                y=torch.zeros(g[nt].num_nodes, device=p.device)
                for s in sinks:
                    if 0<=s<y.numel(): y[s]=1.0
            if y is None:
                continue
            src=getattr(g[nt],"source","default")
            thr=THRS.get(src, THRS.get("default",0.5))
            m=metrics_at(p.detach().cpu(), y.detach().cpu(), thr)
            per_src.setdefault(src, {"n_graphs":0,"metrics":[]})
            per_src[src]["n_graphs"]+=1
            per_src[src]["metrics"].append(m)
            used+=1

            # strict CCS/CFAM
            causal=list(getattr(g[nt],"causal_nodes", [])) if hasattr(g[nt],"causal_nodes") else []
            spurious=list(getattr(g[nt],"spurious_nodes", [])) if hasattr(g[nt],"spurious_nodes") else []
            if not causal:
                if paths:
                    causal=sorted(list({u for path in paths for u in path}))
                else:
                    # if no explicit causal, we cannot invent; skip CCS/CFAM
                    causal=[]
            if not spurious and causal:
                all_nodes=set(range(g[nt].num_nodes))
                spurious=sorted(list(all_nodes - set(causal)))
            if causal:
                ccs=compute_CCS(model, g, nt, causal)
                if ccs is not None: ccs_vals.append(ccs)
                model.zero_grad(set_to_none=True)
                logit2,h2,_=model(g, nt)
                logit2.sum().backward(retain_graph=False)
                cf=attribution_cf(h2, torch.tensor(causal, dtype=torch.long, device=h2.device),
                                  torch.tensor(spurious, dtype=torch.long, device=h2.device) if spurious else None)
                cfam_vals.append(cf)
                model.zero_grad(set_to_none=True)

        overall={"precision":0.0,"recall":0.0,"f1":0.0}
        if per_src:
            for s in per_src:
                ms=per_src[s]["metrics"]
                avg={k: sum(m[k] for m in ms)/len(ms) for k in ms[0]}
                per_src[s]["avg"]=avg
                for k in ("precision","recall","f1"): overall[k]+=avg[k]
            n=len(per_src)
            for k in overall: overall[k]/=n
        return {
            "split":name,
            "overall":overall,
            "thresholds":THRS,
            "n_graphs_used":used,
            "per_source":per_src,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count":len(ccs_vals),
            "CFAM_count":len(cfam_vals)
        }

    # train already ran; now eval
    rep_train = eval_split("train", ds_tr)
    rep_valid = eval_split("valid", ds_va)
    rep_test  = eval_split("test",  ds_te)

    # demo beam (multi-root)
    demo={"meta":{"split":None,"graph_path":None},"paths":[]}
    ds_demo = ds_va if len(ds_va)>0 else (ds_tr if len(ds_tr)>0 else [])
    if len(ds_demo)>0:
        g0 = ds_demo[0]; nt0=primary_node_type(g0) or "node"
        if ADD_SUMMARY_EDGES: add_thin_dfg(g0, nt0)
        g0 = graph_shrink(g0, nt0, must_keep=[])
        move_to_device(g0, nt0, next(model.parameters()).device)
        with torch.no_grad():
            logit0,h0,edges0=model(g0, nt0); p0=torch.sigmoid(logit0)
            edge_scores0=model.rel_gate(h0,{r:edges0[r] for r in edges0.keys()})
        y0,sinks0,paths0 = labels_from_graph(g0, nt0)
        seeds = (y0>0.5).nonzero(as_tuple=False).view(-1).tolist() if (y0 is not None and (y0>0.5).any()) else []
        if not seeds and paths0:
            seeds = sorted(list({path[0] for path in paths0 if len(path)>0}))
        if not seeds:
            seeds = torch.topk(p0, k=min(12,p0.numel())).indices.tolist()
        beam_out = run_beam(p0, edge_scores0, edges0, seeds, 32, 6, 0.7)
        demo["meta"]["split"]=("valid" if len(ds_va)>0 else "train")
        demo["meta"]["graph_path"]=str(ds_demo.paths[0])
        demo["paths"]=beam_out[:10]

    # save artifacts
    torch.save({"model":model.state_dict(), "config":{"hidden":HIDDEN,"layers":LAYERS,"relations":RELATIONS}}, OUT_ROOT/"model.pt")
    report={
        "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
                  "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE.type,"dataset_weights":DATASET_WEIGHTS,
                  "relations":RELATIONS,"add_summary_edges":ADD_SUMMARY_EDGES,
                  "label_keys":LABEL_KEYS,"sink_keys":SINK_KEYS,"path_keys":PATH_KEYS},
        "thresholds_by_source": rep_valid.get("thresholds",{}),
        "reports":{"train":rep_train,"valid":rep_valid,"test":rep_test},
        "demo":demo,
        "notes":{
            "interprocedural":"DFG/CFG/CALL/ARG2PARAM/RET2CALL/RET2LHS (+DFG_THIN optional); multi-root beam.",
            "beam_guidance":"learned per-relation gates (RelGate); no fixed ordering.",
            "objective":"node (focal+class-weights+hard-neg), edge participation, next-hop ranking, monotonicity.",
            "CCS":"p(x) vs p(do(x')) by zeroing strict causal-slice (annotated or from provided paths).",
            "CFAM":"grad-norm attribution mass on causal / (causal+spurious).",
            "calibration":"per-source thresholds tuned on valid; used in summaries.",
            "provenance":"per-graph 'source' via sidecar or in-graph; default otherwise."
        }
    }
    (OUT_ROOT/"report.json").write_text(json.dumps(report, indent=2))
    print("Saved to:", OUT_ROOT)

# ---- run ----
try:
    print(f"[env] torch={torch.__version__} device={DEVICE.type}")
    if DEVICE.type=="cuda":
        try:
            print("GPU:", torch.cuda.get_device_name(0))
        except Exception: pass
    train_and_eval()
except Exception as e:
    print("!! Exception:")
    traceback.print_exc()


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[env] torch=2.4.1+cu121 device=cuda
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert


Saved to: out\cvul\1760397721


In [ ]:
# ========================= Causal-Vul Training (robust hetero loader; beam-only slice; inter-proc) =========================
import os, json, time, random, math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# --------------------------- Config ---------------------------
DATA_DIRS = {
    "train": r"Dataset/train/hetero_ready_gcbert",
    "valid": r"Dataset/valid/hetero_ready_gcbert",
    "test" : r"Dataset/test/hetero_ready_gcbert",
}
RELATIONS = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]
ADD_SUMMARY_EDGES = True  # adds DFG_THIN = DFG + call-family edges

EPOCHS  = 5
HIDDEN  = 64
LAYERS  = 3
LR      = 2e-3
WD      = 1e-4
NEG_POS_RATIO = 20
USE_AMP = True

# Beam (no heuristics; model-guided)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7

DATASET_WEIGHTS = {"default": 1.0}
THRESHOLDS_BY_SOURCE = {"default": 0.25}

LABEL_KEYS = ["y","label","labels","vulnerable","is_sink","target","targets"]
SINK_KEYS  = ["sinks","sink_nodes"]
PATH_KEYS  = ["paths_idx","vulnerable_paths"]

OUT_ROOT = Path("out/cvul")/str(int(time.time()))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------- Device & reproducibility ---------------------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except: return "cuda"

DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")
if DEVICE=="cuda":
    try: print("GPU:", torch.cuda.get_device_name(0))
    except: pass
random.seed(23)
torch.manual_seed(23)
if DEVICE=="cuda":
    try: torch.cuda.manual_seed_all(23)
    except: pass

def vram_info():
    if DEVICE!="cuda": return {"device":"cpu"}
    try:
        free,total = torch.cuda.mem_get_info()
        return {"device":"cuda","freeMB":free//(1024**2),"totalMB":total//(1024**2)}
    except: return {"device":"cuda"}

# --------------------------- IO helpers ---------------------------
def safe_load(path: str):
    p=Path(path)
    if p.suffix.lower()==".json":
        return json.loads(p.read_text())
    # .pt or others -> torch
    return torch.load(p, map_location="cpu")

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N: int, ei: torch.Tensor) -> torch.Tensor:
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d = ei
    m = (s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any():
        return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def coerce_x_any(obj):
    # direct feature fields
    for k in ["x","features","node_features","feat","emb","gcbert","gcb","gcb_x","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    # assemble from known subfields
    xs=[]
    for k in ["x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            xs.append(torch.as_tensor(obj[k]).float())
    if xs:
        xs=[x if x.ndim==2 else x.view(-1,1) for x in xs]
        return torch.cat(xs, dim=1)
    # nodes: [{x:...}, ...]
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["x","feat","emb","features","gcbert","gcb_x"]:
                if k in nd:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N: int) -> Dict[str, torch.Tensor]:
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    # explicit dict of relations
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES and "DFG_THIN" in obj["edges"]:
            E["DFG_THIN"]=sanitize_edges(N, to_long_2(obj["edges"]["DFG_THIN"]))
        return E
    # top-level relation keys
    if isinstance(obj, dict):
        for r in RELATIONS:
            for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if k in obj:
                    E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    # list-of-dicts edges
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], list):
        buckets={r:[[],[]] for r in RELATIONS}
        for ed in obj["edges"]:
            if not isinstance(ed, dict): continue
            t=(ed.get("type") or ed.get("rel") or ed.get("r") or "").upper()
            s=ed.get("src") or ed.get("s"); d=ed.get("dst") or ed.get("t")
            if t in buckets and s is not None and d is not None:
                buckets[t][0].append(int(s)); buckets[t][1].append(int(d))
        for r,(ss,dd) in buckets.items():
            if ss:
                E[r]=sanitize_edges(N, to_long_2([ss,dd]))
    # DFG_THIN
    if ADD_SUMMARY_EDGES:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        parts=[base]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel()>0: parts.append(E[r])
        E["DFG_THIN"]=torch.cat(parts, dim=1) if len(parts)>1 else parts[0]
    return E

def normalize_hetero(obj):
    """
    Robustly handle torch_geometric.HeteroData WITHOUT calling data.x_dict/y_dict
    (they trigger collect() and KeyError if absent). We use node_stores/edge_stores only.
    """
    node_stores=getattr(obj,"node_stores",None)
    edge_stores=getattr(obj,"edge_stores",None)
    if node_stores is None or edge_stores is None:
        return None

    # choose node store with the largest x
    cands=[]
    for st in node_stores:
        xv=getattr(st,"x",None)
        if xv is None: continue
        xt=torch.as_tensor(xv).float()
        if xt.ndim==1: xt=xt.view(-1,1)
        key=getattr(st,"_key", None) or getattr(st,"type", None) or "code"
        cands.append((key, xt))
    if not cands: return None
    nt, x = max(cands, key=lambda kv: kv[1].size(0))
    N=x.size(0)

    # collect edges that keep src/dst == nt where available, else accept homogeneous
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei=getattr(es,"edge_index",None)
        if ei is None: continue
        rel=getattr(es,"edge_type",None) or getattr(es,"_key",None)
        src_t=getattr(es,"src_type",None); dst_t=getattr(es,"dst_type",None)
        rel=str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E and (src_t is None or dst_t is None or (src_t==nt and dst_t==nt)):
            E[rel]=sanitize_edges(N, to_long_2(ei))

    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base

    # labels (optional)
    y=None
    for st in node_stores:
        st_key=getattr(st,"_key", None) or getattr(st,"type", None)
        if st_key==nt:
            yy=getattr(st,"y",None)
            if yy is not None:
                yy=torch.as_tensor(yy).float().view(-1)
                if yy.numel()==N: y=yy
            break

    return {"x":x, "edges":E, "y":y, "source":"default"}

def normalize_graph(obj):
    # unwrap {graph: {...}}
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj=obj["graph"]

    # dict schema
    if isinstance(obj, dict):
        x=coerce_x_any(obj)
        if x is not None:
            N=x.size(0)
            E=build_edges_any(obj, N)
            y=None
            for k in LABEL_KEYS:
                if k in obj:
                    try:
                        yy=torch.as_tensor(obj[k]).float().view(-1)
                        if yy.numel()==N: y=yy
                    except: pass
                    break
            src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"
            g={"x":x, "edges":E, "y":y, "source":src}
            if any(k in obj for k in (SINK_KEYS+PATH_KEYS)):
                g["aux"]={k:obj.get(k) for k in (SINK_KEYS+PATH_KEYS)}
            return g

    # torch_geometric Data (homogeneous)
    if hasattr(obj,"x") and hasattr(obj,"__class__") and "torch_geometric" in str(type(obj)):
        # Prefer hetero normalization if possible
        g_het = normalize_hetero(obj)
        if g_het is not None: return g_het

        # Fallback: homogeneous Data
        x=torch.as_tensor(getattr(obj,"x")).float()
        if x.ndim==1: x=x.view(-1,1)
        N=x.size(0)
        E={}
        for r in RELATIONS:
            found=False
            for nm in [f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if hasattr(obj, nm) and getattr(obj, nm) is not None:
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,nm))); found=True; break
            if not found:
                if hasattr(obj,"edge_index") and r=="DFG":
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,"edge_index")))
                else:
                    E[r]=torch.zeros((2,0),dtype=torch.long)
        if ADD_SUMMARY_EDGES:
            base=E["DFG"]
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        y=None
        if hasattr(obj,"y") and obj.y is not None and len(obj.y)==N:
            y=torch.as_tensor(obj.y).float().view(-1)
        return {"x":x, "edges":E, "y":y, "source":"default"}

    # torch_geometric HeteroData-like that didn't match previous clause
    if "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj)
        if g_het is not None: return g_het

    raise RuntimeError("Unsupported graph object type")

class GraphDir:
    def __init__(self, root):
        root=Path(root)
        self.paths = sorted([str(p) for p in root.glob("*.json")] + [str(p) for p in root.glob("*.pt")])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        obj=safe_load(self.paths[i])
        g=normalize_graph(obj)
        g["path"]=self.paths[i]
        return g

# --------------------------- Model ---------------------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden, layers, relations):
        super().__init__()
        self.relations = relations + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
    def _proj(self, D:int):
        key=str(D)
        if key not in self.proj_cache:
            layer=nn.Linear(D, HIDDEN).to(next(self.parameters()).device)
            self.proj_cache[key]=layer
        return self.proj_cache[key]
    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

# --------------------------- Beam & slicing ---------------------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei
        ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def run_beam(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i]); val=float(es[i].item())
            if 0<=u<N and 0<=v<N: mp[(u,v)] = max(mp.get((u,v), -1e9), val)
        uv[r]=mp
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)]) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es=uv.get(r,{}).get((uu,vv), 0.0)
                sc=b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams=nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

def slice_from_paths(g, paths):
    if not paths:
        return {"x": g["x"][:1], "edges":{r:torch.zeros((2,0),dtype=torch.long) for r in g["edges"]}, "orig_idx":[0]}
    idx = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<g["x"].size(0)))
    if not idx: idx=[0]
    idmap={old:i for i,old in enumerate(idx)}
    x=g["x"][idx]
    E={}
    keep=torch.tensor(idx)
    for r,ei in g["edges"].items():
        if ei is None or ei.numel()==0: E[r]=torch.zeros((2,0),dtype=torch.long); continue
        s,d=ei
        m=torch.isin(s,keep)&torch.isin(d,keep)
        if m.any():
            s2=torch.tensor([idmap[int(v)] for v in s[m].tolist()], dtype=torch.long)
            d2=torch.tensor([idmap[int(v)] for v in d[m].tolist()], dtype=torch.long)
            E[r]=torch.stack([s2,d2], dim=0)
        else:
            E[r]=torch.zeros((2,0),dtype=torch.long)
    return {"x":x, "edges":E, "orig_idx":idx}

def remap_paths_to_slice(paths, orig_idx):
    idmap={old:i for i,old in enumerate(orig_idx)}
    out=[]
    for bp in paths:
        ns=[]
        for n in bp.nodes:
            if n in idmap: ns.append(idmap[n])
            else: ns=[]; break
        if len(ns)>=2: out.append(BeamPath(bp.score, ns))
    return out

# --------------------------- Losses ---------------------------
def focal_bce_with_logits(logit, target, alpha_pos=0.5, gamma=2.0):
    ce=F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    p=torch.sigmoid(logit)
    pt=p*target + (1-p)*(1-target)
    w=(alpha_pos*target + (1-alpha_pos)*(1-target)) * ((1-pt).pow(gamma))
    return (w*ce).mean()

def class_weights(y):
    pos=max(float((y>0.5).sum().item()), 1.0)
    neg=max(float((y<=0.5).sum().item()), 1.0)
    return neg/(pos+neg)

def hard_negative_mask(p,y,max_ratio=20):
    pos=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg=(y<=0.5).nonzero(as_tuple=False).view(-1)
    if neg.numel()==0: return torch.ones_like(y, dtype=torch.bool)
    k=min(neg.numel(), int(max_ratio*max(1,pos.numel())))
    if k<=0: k=min(10,neg.numel())
    topk=torch.topk(p[neg], k=k).indices
    keep_neg=neg[topk]
    mask=torch.zeros_like(y, dtype=torch.bool)
    mask[pos]=True; mask[keep_neg]=True
    return mask

def loss_edge_participation(edge_sc, E, paths):
    dev = next((v.device for v in edge_sc.values() if hasattr(v,'device')), torch.device("cpu"))
    if not paths: return torch.tensor(0.0, device=dev)
    L=torch.tensor(0.0, device=dev); n=0
    pos=set()
    for bp in paths[:3]:
        for u,v in zip(bp.nodes[:-1], bp.nodes[1:]): pos.add((u,v))
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; es=edge_sc[r]
        if es.numel()==0: continue
        pm = torch.tensor([ (int(s[i]),int(d[i])) in pos for i in range(s.numel()) ], device=es.device)
        if pm.any():
            L = L + F.binary_cross_entropy_with_logits(es[pm], torch.ones_like(es[pm])); n+=1
        nm = ~pm
        if nm.any():
            idx=torch.nonzero(nm, as_tuple=False).view(-1)
            if idx.numel()>0:
                sel=idx[torch.randperm(idx.numel(), device=idx.device)[:max(1, pm.sum().item())]]
                L = L + F.binary_cross_entropy_with_logits(es[sel], torch.zeros_like(es[sel])); n+=1
    return L/(n or 1)

def loss_monotonicity(p, paths, margin=0.05):
    if not paths: return torch.tensor(0.0, device=p.device)
    L=torch.tensor(0.0, device=p.device); n=0
    for bp in paths[:5]:
        ns=[n for n in bp.nodes if 0<=n<p.numel()]
        for i in range(len(ns)-1):
            L = L + F.relu(p[ns[i]] - p[ns[i+1]] + margin); n+=1
    return L/(n or 1)

def loss_path_ranking(p, paths):
    if not paths: return torch.tensor(0.0, device=p.device)
    def ps(bp):
        idx=[n for n in bp.nodes if 0<=n<p.numel()]
        if len(idx)<2: return None
        idx=torch.as_tensor(idx, device=p.device, dtype=torch.long)
        return torch.log(p[idx].clamp(1e-9,1-1e-9)).sum()
    pos=[t for t in (ps(bp) for bp in paths[:3]) if t is not None]
    if not pos: return torch.tensor(0.0, device=p.device)
    pos=torch.stack(pos)
    rnd=[]
    N=p.numel()
    for _ in range(len(pos)):
        L=max(2, min(N, 6))
        idx=torch.randperm(N, device=p.device)[:L]
        rnd.append(torch.log(p[idx].clamp(1e-9,1-1e-9)).sum())
    rnd=torch.stack(rnd)
    return F.relu(0.1 + rnd - pos).mean()

# --------------------------- Metrics / Attribution ---------------------------
@torch.no_grad()
def f1_from_probs(p, y, thr=0.5):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def compute_CFAM(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _,nl,_,_ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        s=torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    idx=list(causal); mask[torch.tensor(idx, device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE)
    with torch.no_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p0=torch.sigmoid(nl).mean().item()
    cf=x.clone()
    causal=sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal:
        cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _,nl2,_,_=model.forward_full(cf, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p1=torch.sigmoid(nl2).mean().item()
    return (p0-p1)**2

def to_device_graph(g):
    return {"x":g["x"].to(DEVICE),
            "edges":{r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": (g.get("y").to(DEVICE) if g.get("y") is not None else None),
            "source": g.get("source","default")}

# --------------------------- Trainer ---------------------------
@dataclass
class BeamSummary:
    avg_len: Optional[float]=None
    interproc_share: Optional[float]=None

class CausalTrainer:
    def __init__(self):
        self.model = CausalVulNet(hidden=HIDDEN, layers=LAYERS, relations=RELATIONS).to(DEVICE)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=LR, weight_decay=WD)
        self.scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE=='cuda'))
    def train_and_eval(self):
        ds_tr, ds_v, ds_te = GraphDir(DATA_DIRS["train"]), GraphDir(DATA_DIRS["valid"]), GraphDir(DATA_DIRS["test"])
        print(f"[DATA] {len(ds_tr)} graphs in {DATA_DIRS['train']}")
        print(f"[DATA] {len(ds_v)} graphs in {DATA_DIRS['valid']}")
        print(f"[DATA] {len(ds_te)} graphs in {DATA_DIRS['test']}")
        (OUT_ROOT/"cfg.json").write_text(json.dumps({
            "epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
            "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
            "add_summary_edges":ADD_SUMMARY_EDGES,
            "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}
        }, indent=2))
        step=0
        for ep in range(1, EPOCHS+1):
            self.model.train()
            running={"node":0.0,"edge":0.0,"mono":0.0,"rank":0.0}
            for i in tqdm(range(len(ds_tr)), desc=f"[train] epoch {ep}"):
                step+=1
                g_cpu = ds_tr[i]; g_dev = to_device_graph(g_cpu)
                with (torch.amp.autocast('cuda', enabled=(USE_AMP and DEVICE=='cuda')) if DEVICE=='cuda' else torch.autocast("cpu", enabled=False)):
                    # Full-graph seed & beam → Beam-only slice (no heuristics)
                    s_full, n_full, h_full, e_full = self.model.forward_full(g_dev["x"], g_dev["edges"])
                    seeds = pick_seeds(s_full.detach(), g_dev["edges"], SEED_K)
                    p_full = torch.sigmoid(n_full)
                    beam = run_beam(p_full.detach(), e_full, g_dev["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)

                    g_slice_cpu = slice_from_paths({"x":g_cpu["x"], "edges":g_cpu["edges"]}, beam)
                    g_slice = {"x": g_slice_cpu["x"].to(DEVICE),
                               "edges": {r:e.to(DEVICE) for r,e in g_slice_cpu["edges"].items()},
                               "orig_idx": g_slice_cpu["orig_idx"]}
                    beam_slice = remap_paths_to_slice(beam, g_slice["orig_idx"])

                    s_s, n_s, h_s, e_s = self.model.forward_full(g_slice["x"], g_slice["edges"])
                    p_s = torch.sigmoid(n_s)

                    # Node loss (if labels exist on full graph)
                    if g_dev["y"] is not None and g_dev["y"].numel()==p_full.numel():
                        alpha_pos = class_weights(g_dev["y"])
                        mask = hard_negative_mask(p_full.detach(), g_dev["y"], NEG_POS_RATIO)
                        l_node = focal_bce_with_logits(n_full[mask], g_dev["y"][mask], alpha_pos=alpha_pos, gamma=2.0)
                        ds_w = DATASET_WEIGHTS.get(g_cpu.get("source","default"), DATASET_WEIGHTS["default"])
                        l_node = l_node * float(ds_w)
                    else:
                        l_node = torch.tensor(0.0, device=DEVICE)

                    l_edge = loss_edge_participation(e_s, g_slice["edges"], beam_slice)
                    l_mono = loss_monotonicity(p_s, beam_slice, 0.05)
                    l_rank = loss_path_ranking(p_s, beam_slice)

                    loss = l_node + 0.20*l_edge + 0.25*l_mono + 0.20*l_rank

                self.opt.zero_grad(set_to_none=True)
                if DEVICE=='cuda' and USE_AMP:
                    self.scaler.scale(loss).backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
                    self.scaler.step(self.opt); self.scaler.update()
                else:
                    loss.backward(); nn.utils.clip_grad_norm_(self.model.parameters(), 2.0); self.opt.step()

                running["node"]+=float(l_node.detach().item()); running["edge"]+=float(l_edge.detach().item())
                running["mono"]+=float(l_mono.detach().item()); running["rank"]+=float(l_rank.detach().item())

                # Early causal probes every ~1200 steps
                if step % 1200 == 0 and len(ds_v)>0:
                    self.model.eval()
                    with torch.no_grad():
                        blen=[]; inter=[]; cfam=[]; ccs=[]
                        for j in range(min(3,len(ds_v))):
                            gv_cpu=ds_v[j]; gv=to_device_graph(gv_cpu)
                            sd,nl,hv,es = self.model.forward_full(gv["x"], gv["edges"])
                            seeds=pick_seeds(sd.detach(), gv["edges"], min(SEED_K,4))
                            pv=torch.sigmoid(nl)
                            b=run_beam(pv, es, gv["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                            blen.append(sum(len(bp.nodes) for bp in b)/(len(b) or 1))
                            inter_edges={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
                            out_steps=0; inter_steps=0
                            adj,_=build_adj(gv["edges"])
                            for bp in b:
                                for u,v in zip(bp.nodes[:-1], bp.nodes[1:]):
                                    out_steps+=1
                                    if any(v in adj.get(r,{}).get(u,[]) for r in inter_edges):
                                        inter_steps+=1
                            inter.append(inter_steps/(out_steps or 1))
                            cf = compute_CFAM(self.model, gv_cpu, b); cc = compute_CCS(self.model, gv_cpu, b)
                            if cf is not None: cfam.append(cf)
                            if cc is not None: ccs.append(cc)
                    print(f"[early] ep{ep} step{step} | beam.len={sum(blen)/len(blen):.2f} "
                          f"inter%={100*(sum(inter)/len(inter) if inter else 0):.1f} "
                          f"| CFAM~{(sum(cfam)/len(cfam) if cfam else float('nan')):.3f} "
                          f"CCS~{(sum(ccs)/len(ccs) if ccs else float('nan')):.3f} | VRAM={vram_info()}")
                    self.model.train()

            avg={k:v/max(1,len(ds_tr)) for k,v in running.items()}
            print(f"[epoch {ep}] loss={sum(avg.values()):.4f} (node={avg['node']:.4f}, edge={avg['edge']:.4f}, mono={avg['mono']:.4f}, rank={avg['rank']:.4f})")

        # Save model
        torch.save({"state_dict":self.model.state_dict(),
                    "hidden":HIDDEN,"layers":LAYERS,
                    "relations":RELATIONS + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])}, OUT_ROOT/"model.pt")

        # Evaluate + Demo
        self.model.eval()
        reports={}
        for SPLIT, DS in [("train", GraphDir(DATA_DIRS["train"])),
                          ("valid", GraphDir(DATA_DIRS["valid"])),
                          ("test",  GraphDir(DATA_DIRS["test"]))]:
            n_used=0; s_prec=0.0; s_rec=0.0; s_f1=0.0; ccs=[]; cfam=[]
            for i in tqdm(range(min(256, len(DS))), desc=f"[eval:{SPLIT}]"):
                g_cpu=DS[i]; g=to_device_graph(g_cpu)
                with torch.no_grad():
                    sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                    seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                    p=torch.sigmoid(nl)
                    b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                m=f1_from_probs(p.detach().cpu(), g_cpu.get("y"),
                                THRESHOLDS_BY_SOURCE.get(g_cpu.get("source","default"), THRESHOLDS_BY_SOURCE["default"]))
                if m:
                    s_prec+=m["precision"]; s_rec+=m["recall"]; s_f1+=m["f1"]
                cf=compute_CFAM(self.model, g_cpu, b); cc=compute_CCS(self.model, g_cpu, b)
                if cf is not None: cfam.append(cf)
                if cc is not None: ccs.append(cc)
                n_used+=1
            rep={
                "split":SPLIT,
                "overall":{"precision":(s_prec/n_used if n_used and s_prec>0 else None),
                           "recall":   (s_rec/n_used if n_used and s_rec>0 else None),
                           "f1":       (s_f1/n_used if n_used and s_f1>0 else 0.0)},
                "thresholds": THRESHOLDS_BY_SOURCE,
                "n_graphs_used": n_used,
                "CCS_mean": (sum(ccs)/len(ccs) if ccs else None),
                "CFAM_mean":(sum(cfam)/len(cfam) if cfam else None)
            }
            reports[SPLIT]=rep

        demo={"meta":{"split":None,"graph_path":None},"paths":[]}
        if len(GraphDir(DATA_DIRS["valid"]))>0:
            g_cpu=GraphDir(DATA_DIRS["valid"])[0]; g=to_device_graph(g_cpu)
            with torch.no_grad():
                sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                p=torch.sigmoid(nl)
                b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
            demo["meta"]={"split":"valid","graph_path":g_cpu.get("path")}
            demo["paths"]=[{"score":bp.score, "nodes":bp.nodes} for bp in b[:10]]

        (OUT_ROOT/"reports.json").write_text(json.dumps({
            "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
                      "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
                      "add_summary_edges":ADD_SUMMARY_EDGES,"slice_mode":"beam",
                      "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}},
            "thresholds_by_source":THRESHOLDS_BY_SOURCE,
            "reports":reports,
            "notes":{
                "interprocedural":"CALL/ARG2PARAM/RET2CALL/RET2LHS (+DFG_THIN).",
                "beam_guidance":"learned relation gates; no fixed ordering.",
                "objective":"focal+class-weights+hard-negatives; edge-participation; monotonicity; path ranking.",
                "CCS":"p(x) vs p(do(x')) on beam slice.",
                "CFAM":"grad-norm attribution mass on beam slice / total.",
                "multi_root":"top-K seeds; beam tracks multiple chains."
            }
        }, indent=2))
        print(f"[saved] artifacts in: {OUT_ROOT}")

# --------------------------- Run ---------------------------
try:
    CausalTrainer().train_and_eval()
except Exception as e:
    print("!! Exception:", e, "| VRAM:", vram_info())
    raise


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert


[train] epoch 1:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_18800\1637410006.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

!! Exception: element 0 of tensors does not require grad and does not have a grad_fn | VRAM: {'device': 'cuda', 'freeMB': 6790, 'totalMB': 8187}


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [1]:
# ================= Causal-Vul: inter-proc, beam-only slice, CCS/CFAM, early probes =================
import os, json, time, math, random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- Config ----------------
DATA_DIRS = {
    "train": r"Dataset/train/hetero_ready_gcbert",
    "valid": r"Dataset/valid/hetero_ready_gcbert",
    "test" : r"Dataset/test/hetero_ready_gcbert",
}
RELATIONS = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]
ADD_SUMMARY_EDGES = True  # adds DFG_THIN = DFG ∪ call-family

EPOCHS  = 5
HIDDEN  = 64
LAYERS  = 3
LR      = 2e-3
WD      = 1e-4
NEG_POS_RATIO = 20
USE_AMP = True
RNG_SEED = 23

# Beam (model-guided; no heuristics)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7

# Provenance / calibration
DATASET_WEIGHTS = {"default": 1.0}
THRESHOLDS_BY_SOURCE = {"default": 0.25}  # will be re-fit on valid if possible

# Eval budget
EVAL_MAX = 256   # set None to evaluate ALL graphs

OUT_ROOT = Path("out/cvul")/str(int(time.time()))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------- Device & determinism --------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except: return "cuda"

DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")
if DEVICE=="cuda":
    try: print("GPU:", torch.cuda.get_device_name(0))
    except: pass
random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if DEVICE=="cuda":
    try: torch.cuda.manual_seed_all(RNG_SEED)
    except: pass

def vram_info():
    if DEVICE!="cuda": return {"device":"cpu"}
    try:
        free,total = torch.cuda.mem_get_info()
        return {"device":"cuda","freeMB":free//(1024**2),"totalMB":total//(1024**2)}
    except: return {"device":"cuda"}

# -------------- IO & normalization --------------
def safe_load(p): 
    p=Path(p)
    if p.suffix.lower()==".json": return json.loads(p.read_text())
    return torch.load(p, map_location="cpu")

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d=ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def coerce_x_any(obj):
    # common fields
    for k in ["x","features","node_features","feat","emb","gcbert","gcb_x","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    xs=[]
    for k in ["x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            xs.append(t if t.ndim==2 else t.view(-1,1))
    if xs:
        d=max(x.size(1) for x in xs)
        xs=[F.pad(x,(0, d-x.size(1))) for x in xs]
        return torch.cat(xs, dim=1)
    # node list
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["x","feat","emb","features","gcbert","gcb_x"]:
                if k in nd:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES and "DFG_THIN" in obj["edges"]:
            E["DFG_THIN"]=sanitize_edges(N, to_long_2(obj["edges"]["DFG_THIN"]))
        return E
    if isinstance(obj, dict):
        for r in RELATIONS:
            for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if k in obj:
                    E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_hetero(obj):
    node_stores=getattr(obj,"node_stores",None)
    edge_stores=getattr(obj,"edge_stores",None)
    if node_stores is None or edge_stores is None: return None
    cands=[]
    for st in node_stores:
        xv=getattr(st,"x",None)
        if xv is None: continue
        xt=torch.as_tensor(xv).float()
        if xt.ndim==1: xt=xt.view(-1,1)
        key=getattr(st,"_key", None) or getattr(st,"type", None) or "code"
        cands.append((key, xt))
    if not cands: return None
    nt,x=max(cands, key=lambda kv: kv[1].size(0))
    N=x.size(0)
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei=getattr(es,"edge_index",None)
        if ei is None: continue
        rel=getattr(es,"edge_type",None) or getattr(es,"_key",None)
        src_t=getattr(es,"src_type",None); dst_t=getattr(es,"dst_type",None)
        rel=str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E and (src_t is None or dst_t is None or (src_t==nt and dst_t==nt)):
            E[rel]=sanitize_edges(N, to_long_2(ei))
    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    y=None
    for st in node_stores:
        st_key=getattr(st,"_key", None) or getattr(st,"type", None)
        if st_key==nt:
            yy=getattr(st,"y",None)
            if yy is not None:
                yy=torch.as_tensor(yy).float().view(-1)
                if yy.numel()==N: y=yy
            break
    return {"x":x, "edges":E, "y":y, "source":"default"}

def normalize_graph(obj):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj=obj["graph"]
    # dict
    if isinstance(obj, dict):
        x=coerce_x_any(obj)
        if x is not None:
            N=x.size(0)
            E=build_edges_any(obj, N)
            y=None
            for k in ["y","label","labels","vulnerable","is_sink","target","targets"]:
                if k in obj:
                    try:
                        yy=torch.as_tensor(obj[k]).float().view(-1)
                        if yy.numel()==N: y=yy
                    except: pass
                    break
            src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"
            g={"x":x, "edges":E, "y":y, "source":src}
            if any(k in obj for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]):
                g["aux"]={k:obj.get(k) for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]}
            return g
    # torch_geometric
    if hasattr(obj,"x") and "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj)
        if g_het is not None: return g_het
        x=torch.as_tensor(getattr(obj,"x")).float()
        if x.ndim==1: x=x.view(-1,1)
        N=x.size(0)
        E={}
        for r in RELATIONS:
            found=False
            for nm in [f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if hasattr(obj, nm) and getattr(obj,nm) is not None:
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,nm))); found=True; break
            if not found:
                if hasattr(obj,"edge_index") and r=="DFG":
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,"edge_index")))
                else:
                    E[r]=torch.zeros((2,0),dtype=torch.long)
        if ADD_SUMMARY_EDGES:
            base=E["DFG"]
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        y=None
        if hasattr(obj,"y") and obj.y is not None and len(obj.y)==N:
            y=torch.as_tensor(obj.y).float().view(-1)
        return {"x":x, "edges":E, "y":y, "source":"default"}
    # last chance hetero
    if "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj)
        if g_het is not None: return g_het
    raise RuntimeError("Unsupported graph object type")

class GraphDir:
    def __init__(self, root):
        root=Path(root)
        self.paths = sorted([str(p) for p in root.glob("*.json")] + [str(p) for p in root.glob("*.pt")])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        obj=safe_load(self.paths[i])
        g=normalize_graph(obj)
        g["path"]=self.paths[i]
        return g

# -------------- Model --------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden, layers, relations):
        super().__init__()
        self.relations = relations + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, HIDDEN).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]
    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

# -------------- Beam & slicing --------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(),d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def run_beam(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                if (u,v) in mp: mp[(u,v)]=max(mp[(u,v)],val)
                else: mp[(u,v)]=val
        uv[r]=mp
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)]) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es=uv.get(r,{}).get((uu,vv), 0.0)
                sc=b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams=nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

def slice_from_paths(g, paths):
    if not paths:
        return {"x": g["x"][:1], "edges":{r:torch.zeros((2,0),dtype=torch.long) for r in g["edges"]}, "orig_idx":[0]}
    idx = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<g["x"].size(0)))
    if not idx: idx=[0]
    idmap={old:i for i,old in enumerate(idx)}
    x=g["x"][idx]
    E={}
    keep=torch.tensor(idx)
    for r,ei in g["edges"].items():
        if ei is None or ei.numel()==0: E[r]=torch.zeros((2,0),dtype=torch.long); continue
        s,d=ei
        m=torch.isin(s,keep)&torch.isin(d,keep)
        if m.any():
            s2=torch.tensor([idmap[int(v)] for v in s[m].tolist()], dtype=torch.long)
            d2=torch.tensor([idmap[int(v)] for v in d[m].tolist()], dtype=torch.long)
            E[r]=torch.stack([s2,d2], dim=0)
        else:
            E[r]=torch.zeros((2,0),dtype=torch.long)
    return {"x":x, "edges":E, "orig_idx":idx}

def remap_paths_to_slice(paths, orig_idx):
    idmap={old:i for i,old in enumerate(orig_idx)}
    out=[]
    for bp in paths:
        ns=[]
        for n in bp.nodes:
            if n in idmap: ns.append(idmap[n])
            else: ns=[]; break
        if len(ns)>=2: out.append(BeamPath(bp.score, ns))
    return out

# -------------- Losses --------------
def focal_bce_with_logits(logit, target, alpha_pos=0.5, gamma=2.0):
    ce=F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    p=torch.sigmoid(logit)
    pt=p*target + (1-p)*(1-target)
    w=(alpha_pos*target + (1-alpha_pos)*(1-target)) * ((1-pt).pow(gamma))
    return (w*ce).mean()

def class_weights(y):
    pos=max(float((y>0.5).sum().item()), 1.0)
    neg=max(float((y<=0.5).sum().item()), 1.0)
    return neg/(pos+neg)

def hard_negative_mask(p,y,max_ratio=20):
    pos=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg=(y<=0.5).nonzero(as_tuple=False).view(-1)
    if neg.numel()==0: return torch.ones_like(y, dtype=torch.bool)
    k=min(neg.numel(), int(max_ratio*max(1,pos.numel())))
    if k<=0: k=min(10,neg.numel())
    topk=torch.topk(p[neg], k=k).indices
    keep_neg=neg[topk]
    mask=torch.zeros_like(y, dtype=torch.bool)
    mask[pos]=True; mask[keep_neg]=True
    return mask

def loss_edge_participation(edge_sc, E, paths):
    # encourage edges that are used in top beam paths
    any_vec = next(iter(edge_sc.values()), torch.tensor([], device='cpu'))
    dev = any_vec.device if hasattr(any_vec,'device') else torch.device('cpu')
    if not paths: return torch.tensor(0.0, device=dev)
    L=torch.tensor(0.0, device=dev); n=0
    pos=set()
    for bp in paths[:3]:
        for u,v in zip(bp.nodes[:-1], bp.nodes[1:]): pos.add((u,v))
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; es=edge_sc[r]
        if es.numel()==0: continue
        pm = torch.tensor([ (int(s[i]),int(d[i])) in pos for i in range(s.numel()) ], device=es.device)
        if pm.any():
            L = L + F.binary_cross_entropy_with_logits(es[pm], torch.ones_like(es[pm])); n+=1
        nm = ~pm
        if nm.any():
            idx=torch.nonzero(nm, as_tuple=False).view(-1)
            if idx.numel()>0:
                sel=idx[torch.randperm(idx.numel(), device=idx.device)[:max(1, pm.sum().item())]]
                L = L + F.binary_cross_entropy_with_logits(es[sel], torch.zeros_like(es[sel])); n+=1
    return L/(n or 1)

def loss_monotonicity(p, paths, margin=0.05):
    if not paths: return torch.tensor(0.0, device=p.device)
    L=torch.tensor(0.0, device=p.device); n=0
    for bp in paths[:5]:
        ns=[n for n in bp.nodes if 0<=n<p.numel()]
        for i in range(len(ns)-1):
            L = L + F.relu(p[ns[i]] - p[ns[i+1]] + margin); n+=1
    return L/(n or 1)

def loss_path_ranking(p, paths):
    if not paths: return torch.tensor(0.0, device=p.device)
    def ps(bp):
        idx=[n for n in bp.nodes if 0<=n<p.numel()]
        if len(idx)<2: return None
        idx=torch.as_tensor(idx, device=p.device, dtype=torch.long)
        return torch.log(p[idx].clamp(1e-9,1-1e-9)).sum()
    pos=[t for t in (ps(bp) for bp in paths[:3]) if t is not None]
    if not pos: return torch.tensor(0.0, device=p.device)
    pos=torch.stack(pos)
    rnd=[]
    N=p.numel()
    for _ in range(len(pos)):
        L=max(2, min(N, 6))
        idx=torch.randperm(N, device=p.device)[:L]
        rnd.append(torch.log(p[idx].clamp(1e-9,1-1e-9)).sum())
    rnd=torch.stack(rnd)
    return F.relu(0.1 + rnd - pos).mean()

# -------------- Metrics / Attribution --------------
@torch.no_grad()
def f1_from_probs(p, y, thr=0.5):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def compute_CFAM(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _,nl,_,_ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        s=torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE)
    with torch.no_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p0=torch.sigmoid(nl).mean().item()
    cf=x.clone()
    causal=sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal: cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _,nl2,_,_=model.forward_full(cf, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p1=torch.sigmoid(nl2).mean().item()
    return (p0-p1)**2

def to_device_graph(g):
    return {"x":g["x"].to(DEVICE),
            "edges":{r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": (g.get("y").to(DEVICE) if g.get("y") is not None else None),
            "source": g.get("source","default")}

# -------------- Calibration --------------
@torch.no_grad()
def calibrate_thresholds(model, ds):
    per_source={}
    for i in range(min(EVAL_MAX or 10**9, len(ds))):
        g_cpu=ds[i]; g=to_device_graph(g_cpu)
        sd,nl,_,_=model.forward_full(g["x"], g["edges"])
        p=torch.sigmoid(nl).detach().cpu()
        y=g_cpu.get("y")
        if y is None or y.numel()!=p.numel(): continue
        src=g_cpu.get("source","default")
        per_source.setdefault(src, {"p":[], "y": []})
        per_source[src]["p"].append(p); per_source[src]["y"].append(y)
    out={}
    for src,buf in per_source.items():
        P=torch.cat(buf["p"]); Y=torch.cat([torch.as_tensor(x).float().view(-1) for x in buf["y"]])
        best=(0.5,0.0)
        for thr in [i/100 for i in range(5,96,5)]:
            m=f1_from_probs(P, Y, thr)
            f=m["f1"] if m else 0.0
            if f>best[1]: best=(thr,f)
        out[src]=best[0]
    if not out: out={"default": THRESHOLDS_BY_SOURCE.get("default",0.25)}
    return out

# -------------- Trainer --------------
class CausalTrainer:
    def __init__(self):
        self.model = CausalVulNet(hidden=HIDDEN, layers=LAYERS, relations=RELATIONS).to(DEVICE)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=LR, weight_decay=WD)
        self.scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE=='cuda'))
    def train_and_eval(self):
        ds_tr, ds_v, ds_te = GraphDir(DATA_DIRS["train"]), GraphDir(DATA_DIRS["valid"]), GraphDir(DATA_DIRS["test"])
        print(f"[DATA] {len(ds_tr)} graphs in {DATA_DIRS['train']}")
        print(f"[DATA] {len(ds_v)} graphs in {DATA_DIRS['valid']}")
        print(f"[DATA] {len(ds_te)} graphs in {DATA_DIRS['test']}")
        (OUT_ROOT/"cfg.json").write_text(json.dumps({
            "epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
            "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
            "add_summary_edges":ADD_SUMMARY_EDGES,
            "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}
        }, indent=2))
        step=0
        for ep in range(1, EPOCHS+1):
            self.model.train()
            running={"node":0.0,"edge":0.0,"mono":0.0,"rank":0.0}
            skipped=0
            for i in tqdm(range(len(ds_tr)), desc=f"[train] epoch {ep}"):
                step+=1
                g_cpu = ds_tr[i]; g_dev = to_device_graph(g_cpu)
                with (torch.amp.autocast('cuda', enabled=(USE_AMP and DEVICE=='cuda')) if DEVICE=='cuda' else torch.autocast("cpu", enabled=False)):
                    # seed/beam on full graph (cheap seed head) → beam-only slice; NO heuristics
                    s_full, n_full, h_full, e_full = self.model.forward_full(g_dev["x"], g_dev["edges"])
                    seeds = pick_seeds(s_full.detach(), g_dev["edges"], SEED_K)
                    p_full = torch.sigmoid(n_full)
                    beam = run_beam(p_full.detach(), e_full, g_dev["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)

                    if len(beam)==0:
                        skipped+=1
                        continue  # nothing to learn from

                    g_slice_cpu = slice_from_paths({"x":g_cpu["x"], "edges":g_cpu["edges"]}, beam)
                    g_slice = {"x": g_slice_cpu["x"].to(DEVICE),
                               "edges": {r:e.to(DEVICE) for r,e in g_slice_cpu["edges"].items()},
                               "orig_idx": g_slice_cpu["orig_idx"]}
                    beam_slice = remap_paths_to_slice(beam, g_slice["orig_idx"])
                    if len(beam_slice)==0:
                        skipped+=1
                        continue

                    s_s, n_s, h_s, e_s = self.model.forward_full(g_slice["x"], g_slice["edges"])
                    p_s = torch.sigmoid(n_s)

                    # Node loss if labels exist; else 0
                    if g_dev["y"] is not None and g_dev["y"].numel()==p_full.numel():
                        alpha_pos = class_weights(g_dev["y"])
                        mask = hard_negative_mask(p_full.detach(), g_dev["y"], NEG_POS_RATIO)
                        l_node = focal_bce_with_logits(n_full[mask], g_dev["y"][mask], alpha_pos=alpha_pos, gamma=2.0)
                        ds_w = DATASET_WEIGHTS.get(g_cpu.get("source","default"), DATASET_WEIGHTS["default"])
                        l_node = l_node * float(ds_w)
                    else:
                        l_node = torch.tensor(0.0, device=DEVICE)

                    l_edge = loss_edge_participation(e_s, g_slice["edges"], beam_slice)
                    l_mono = loss_monotonicity(p_s, beam_slice, 0.05)
                    l_rank = loss_path_ranking(p_s, beam_slice)

                    loss = l_node + 0.20*l_edge + 0.25*l_mono + 0.20*l_rank

                # if everything was constant (rare), skip backward
                if not loss.requires_grad:
                    skipped+=1
                    continue

                self.opt.zero_grad(set_to_none=True)
                if DEVICE=='cuda' and USE_AMP:
                    self.scaler.scale(loss).backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
                    self.scaler.step(self.opt); self.scaler.update()
                else:
                    loss.backward(); nn.utils.clip_grad_norm_(self.model.parameters(), 2.0); self.opt.step()

                running["node"]+=float(l_node.detach().item())
                running["edge"]+=float(l_edge.detach().item())
                running["mono"]+=float(l_mono.detach().item())
                running["rank"]+=float(l_rank.detach().item())

                # Early causal probes every ~1200 steps (inter-proc %, beam length, CFAM/CCS snapshot)
                if step % 1200 == 0 and len(ds_v)>0:
                    self.model.eval()
                    with torch.no_grad():
                        blen=[]; inter=[]; cfam=[]; ccs=[]
                        for j in range(min(3,len(ds_v))):
                            gv_cpu=ds_v[j]; gv=to_device_graph(gv_cpu)
                            sd,nl,hv,es = self.model.forward_full(gv["x"], gv["edges"])
                            seeds=pick_seeds(sd.detach(), gv["edges"], min(SEED_K,4))
                            pv=torch.sigmoid(nl)
                            b=run_beam(pv, es, gv["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                            if b:
                                blen.append(sum(len(bp.nodes) for bp in b)/(len(b) or 1))
                                inter_edges={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
                                out_steps=0; inter_steps=0
                                adj,_=build_adj(gv["edges"])
                                for bp in b:
                                    for u,v in zip(bp.nodes[:-1], bp.nodes[1:]):
                                        out_steps+=1
                                        if any(v in adj.get(r,{}).get(u,[]) for r in inter_edges):
                                            inter_steps+=1
                                inter.append(inter_steps/(out_steps or 1))
                            cf = compute_CFAM(self.model, gv_cpu, b); cc = compute_CCS(self.model, gv_cpu, b)
                            if cf is not None: cfam.append(cf)
                            if cc is not None: ccs.append(cc)
                    print(f"[early] ep{ep} step{step} | beam.len={ (sum(blen)/len(blen) if blen else float('nan')):.2f} "
                          f"inter%={ 100*(sum(inter)/len(inter) if inter else 0):.1f} "
                          f"| CFAM~{(sum(cfam)/len(cfam) if cfam else float('nan')):.3f} "
                          f"CCS~{(sum(ccs)/len(ccs) if ccs else float('nan')):.3f} | VRAM={vram_info()}")
                    self.model.train()

            avg={k:v/max(1,(len(ds_tr)-skipped)) for k,v in running.items()}
            print(f"[epoch {ep}] loss={sum(avg.values()):.4f} (node={avg['node']:.4f}, edge={avg['edge']:.4f}, mono={avg['mono']:.4f}, rank={avg['rank']:.4f}) | skipped={skipped}")

        # Save model
        torch.save({"state_dict":self.model.state_dict(),
                    "hidden":HIDDEN,"layers":LAYERS,
                    "relations":RELATIONS + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])}, OUT_ROOT/"model.pt")

        # Calibrate thresholds on valid (if labels exist)
        if len(ds_v)>0: 
            thrs = calibrate_thresholds(self.model, ds_v)
        else:
            thrs = THRESHOLDS_BY_SOURCE
        (OUT_ROOT/"thresholds.json").write_text(json.dumps(thrs, indent=2))

        # Evaluate + Demo
        self.model.eval()
        reports={}
        for SPLIT, DS in [("train", GraphDir(DATA_DIRS["train"])),
                          ("valid", GraphDir(DATA_DIRS["valid"])),
                          ("test",  GraphDir(DATA_DIRS["test"]))]:
            n_used=0; s_prec=0.0; s_rec=0.0; s_f1=0.0; ccs_vals=[]; cfam_vals=[]
            U = min(EVAL_MAX or 10**9, len(DS))
            for i in tqdm(range(U), desc=f"[eval:{SPLIT}]"):
                g_cpu=DS[i]; g=to_device_graph(g_cpu)
                with torch.no_grad():
                    sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                    seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                    p=torch.sigmoid(nl)
                    b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                m=f1_from_probs(p.detach().cpu(), g_cpu.get("y"), thrs.get(g_cpu.get("source","default"), thrs.get("default", 0.25)))
                if m:
                    s_prec+=m["precision"]; s_rec+=m["recall"]; s_f1+=m["f1"]
                cf=compute_CFAM(self.model, g_cpu, b); cc=compute_CCS(self.model, g_cpu, b)
                if cf is not None: cfam_vals.append(cf)
                if cc is not None: ccs_vals.append(cc)
                n_used+=1
            rep={
                "split":SPLIT,
                "overall":{"precision":(s_prec/n_used if n_used and s_prec>0 else None),
                           "recall":   (s_rec/n_used if n_used and s_rec>0 else None),
                           "f1":       (s_f1/n_used if n_used and s_f1>0 else 0.0)},
                "thresholds": thrs,
                "n_graphs_used": n_used,
                "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
                "CFAM_mean":(sum(cfam_vals)/len(cfam_vals) if cfam_vals else None)
            }
            reports[SPLIT]=rep

        demo={"meta":{"split":None,"graph_path":None},"paths":[]}
        if len(GraphDir(DATA_DIRS["valid"]))>0:
            g_cpu=GraphDir(DATA_DIRS["valid"])[0]; g=to_device_graph(g_cpu)
            with torch.no_grad():
                sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                p=torch.sigmoid(nl)
                b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
            demo["meta"]={"split":"valid","graph_path":g_cpu.get("path")}
            demo["paths"]=[{"score":bp.score, "nodes":bp.nodes} for bp in b[:10]]

        (OUT_ROOT/"reports.json").write_text(json.dumps({
            "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
                      "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
                      "add_summary_edges":ADD_SUMMARY_EDGES,"slice_mode":"beam",
                      "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}},
            "thresholds_by_source":thrs,
            "reports":reports,
            "notes":{
                "interprocedural":"CALL/ARG2PARAM/RET2CALL/RET2LHS (+DFG_THIN) participate in beam transitions.",
                "beam_guidance":"learned relation gates + bilinear edge scorer; no fixed ordering.",
                "objective":"focal(class-weights + hard-negatives) + edge participation + monotonicity + path ranking.",
                "CCS":"p(x) vs p(do(x')) by zeroing beam slice features.",
                "CFAM":"grad-norm attribution mass on beam slice / total.",
                "multi_root":"top-K seed nodes → multiple chains.",
                "saving": str(OUT_ROOT)
            }
        }, indent=2))
        print(f"[saved] artifacts in: {OUT_ROOT}")

# -------------- Run --------------
try:
    CausalTrainer().train_and_eval()
except Exception as e:
    print("!! Exception:", e, "| VRAM:", vram_info())
    raise


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert


[train] epoch 1:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_24636\581379378.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  

[epoch 1] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[train] epoch 2: 100%|██████████| 3438/3438 [02:08<00:00, 26.66it/s]


[epoch 2] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


...

[epoch 4] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[train] epoch 5: 100%|██████████| 3438/3438 [01:52<00:00, 30.49it/s]


[epoch 5] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[eval:test]: 100%|██████████| 256/256 [00:06<00:00, 40.15it/s]


[saved] artifacts in: out\cvul\1760985443


Create CKG

In [7]:
# ===================== CKG mining + CKG-guided inference (drop-in cell) =====================
import os, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- CONFIG (edit these) ----------
CKPT       = r"out/cvul/1760989836/model.pt"                    # <-- set a real run id
DATA_MINE  = r"Dataset/train/hetero_ready_gcbert"            # mine CKG from here
DATA_DEMO  = r"Dataset/demo/hetero_ready_gcbert"             # may be empty; will fallback
CKG_PATH   = r"ckg/ckg.json"                                 # CKG output
DEMO_OUT   = r"out/ckg_demo/demo.json"                       # demo outputs

# Beam defaults (match your training)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7

# Small CKG prior weight during inference (0.1–0.3 is typical)
LAMBDA_PRIOR  = 0.15

# ---------- device ----------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"
DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ---------- utils ----------
def safe_load(p):
    p = Path(p)
    if p.suffix.lower() == ".json":
        return json.loads(p.read_text())
    return torch.load(p, map_location="cpu")

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0:
        return torch.zeros((2,0), dtype=torch.long)
    s,d = ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any():
        return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

# ---------- robust feature extraction for HeteroData ----------
def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim == 1: t = t.view(-1, 1)
    return t

def _get_store_feat(store):
    cand = ["x", "gcb_x", "x_text", "x_num", "features", "feat", "emb"]
    for nm in cand:
        if hasattr(store, nm):
            val = getattr(store, nm)
            if val is not None:
                t = _first_2d_float(val)
                if t.numel() > 0:
                    return t
    if hasattr(store, "__dict__"):
        for nm in cand:
            if nm in store.__dict__:
                t = _first_2d_float(store.__dict__[nm])
                if t.numel() > 0:
                    return t
    return None

def normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    node_stores = getattr(obj, "node_stores", None)
    edge_stores = getattr(obj, "edge_stores", None)
    if node_stores is None or edge_stores is None:
        return None

    candidates=[]
    for st in node_stores:
        key = getattr(st, "_key", None) or getattr(st, "type", None) or "code"
        feat = _get_store_feat(st)
        if feat is not None and feat.numel() > 0:
            candidates.append((key, feat))

    if not candidates:
        return None

    nt, x = max(candidates, key=lambda kv: kv[1].size(0))
    N = x.size(0)

    E = {r: torch.zeros((2,0), dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei = getattr(es, "edge_index", None)
        if ei is None: continue
        src_t = getattr(es, "src_type", None)
        dst_t = getattr(es, "dst_type", None)
        if (src_t is not None and dst_t is not None) and not (src_t == nt and dst_t == nt):
            continue
        rel = getattr(es, "edge_type", None) or getattr(es, "_key", None)
        if isinstance(rel, (tuple, list)) and len(rel) == 3:
            rel = str(rel[1])
        rel = str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E:
            E[rel] = sanitize_edges(N, to_long_2(ei))

    if ADD_SUMMARY_EDGES:
        base = E.get("DFG", torch.zeros((2,0), dtype=torch.long))
        for r in ("ARG2PARAM", "RET2CALL", "RET2LHS"):
            if r in E and E[r].numel() > 0:
                base = torch.cat([base, E[r]], dim=1)
        E["DFG_THIN"] = base

    y = None
    for st in node_stores:
        st_key = getattr(st, "_key", None) or getattr(st, "type", None)
        if st_key == nt:
            for lab in ["y", "labels", "target", "targets"]:
                if hasattr(st, lab):
                    yy = getattr(st, lab)
                    if yy is not None:
                        yy = torch.as_tensor(yy).float().view(-1)
                        if yy.numel() == N:
                            y = yy
                            break
            break

    return {"x": x, "edges": E, "y": y, "source": "default"}

def coerce_x_any(obj):
    for k in ["x","features","node_features","feat","emb","gcbert","gcb_x","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    xs=[]
    for k in ["x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            xs.append(t if t.ndim==2 else t.view(-1,1))
    if xs:
        d=max(x.size(1) for x in xs)
        xs=[F.pad(x,(0,d-x.size(1))) for x in xs]
        return torch.cat(xs, dim=1)
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["x","feat","emb","features","gcbert","gcb_x"]:
                if k in nd:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES and "DFG_THIN" in obj["edges"]:
            E["DFG_THIN"]=sanitize_edges(N, to_long_2(obj["edges"]["DFG_THIN"]))
        return E
    if isinstance(obj, dict):
        for r in RELATIONS:
            for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if k in obj:
                    E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj = obj["graph"]

    if "torch_geometric" in str(type(obj)):
        g_het = normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g_het is not None:
            return g_het
        if hasattr(obj, "x"):
            x = _first_2d_float(getattr(obj, "x"))
            N = x.size(0)
            E={}
            for r in RELATIONS:
                found=False
                for nm in [f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                    if hasattr(obj, nm) and getattr(obj, nm) is not None:
                        E[r]=sanitize_edges(N, to_long_2(getattr(obj, nm))); found=True; break
                if not found:
                    if hasattr(obj, "edge_index") and r=="DFG":
                        E[r]=sanitize_edges(N, to_long_2(getattr(obj,"edge_index")))
                    else:
                        E[r]=torch.zeros((2,0),dtype=torch.long)
            if ADD_SUMMARY_EDGES:
                base = E["DFG"]
                for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                    if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
                E["DFG_THIN"] = base
            y=None
            if hasattr(obj, "y") and obj.y is not None and len(obj.y)==N:
                y = torch.as_tensor(obj.y).float().view(-1)
            return {"x":x, "edges":E, "y":y, "source":"default"}
        return None

    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is not None:
            N = x.size(0)
            E = build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES)
            y=None
            for k in ["y","label","labels","vulnerable","is_sink","target","targets"]:
                if k in obj:
                    try:
                        yy=torch.as_tensor(obj[k]).float().view(-1)
                        if yy.numel()==N: y=yy
                    except: pass
                    break
            src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"
            g={"x":x, "edges":E, "y":y, "source":src}
            if any(k in obj for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]):
                g["aux"]={k:obj.get(k) for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]}
            return g

    raise RuntimeError("Unsupported graph object type or missing features")

def iter_graphs(root_dir, RELATIONS, ADD_SUMMARY_EDGES=True, pattern=("*.pt","*.json")):
    root = Path(root_dir)
    files=[]
    for pat in pattern:
        files.extend(sorted(root.glob(pat)))
    for fp in files:
        try:
            obj = safe_load(fp)
            g = normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
            if g is None or g["x"] is None or g["x"].numel()==0:
                print(f"[skip] {fp} no usable node features after normalization")
                continue
            g["path"] = str(fp)
            yield g
        except Exception as ex:
            print(f"[skip] {fp} error: {ex}")

def pick_demo_graph_or_fallback(demo_dir, RELATIONS, *fallback_dirs):
    for g in iter_graphs(demo_dir, RELATIONS):
        return g
    for d in fallback_dirs:
        for g in iter_graphs(d, RELATIONS):
            return g
    raise RuntimeError("No usable graph found in demo or fallback directories.")

# ---------- model (match your training class) ----------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations = relations
        self.lin_rel = nn.ModuleDict({r: nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self= nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H = h
        for r in self.relations:
            ei = E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d = ei
            msg = self.lin_rel[r](H)
            agg = torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H = H + agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None, add_summary=True):
        super().__init__()
        if relations is None:
            relations = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]
        self.relations = list(relations) + (["DFG_THIN"] if add_summary and "DFG_THIN" not in relations else [])
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden    = hidden
        self.layers    = layers

    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]

    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h

    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out

    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path:str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"])
    add_summary = ("DFG_THIN" in relations)
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations, add_summary=add_summary).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    base_relations = [r for r in relations if r != "DFG_THIN"]
    return model, base_relations, add_summary

# ---------- beam (plain) ----------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]   # relation type per hop

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc, E, N:int):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

def run_beam_plain(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                sc = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# ---------- beam with CKG prior ----------
def run_beam_with_ckg(p, edge_sc, E, seeds, ckg, width=24, max_hops=5, alpha_node=0.7, lambda_prior=0.15):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)

    # priors (log domain)
    ep = ckg.get("edge_prior_prob", {})
    bp = ckg.get("bigram_prob", {})
    def logp_edge(r):
        pr = float(ep.get(r, 1e-6))
        return math.log(max(pr, 1e-9))
    def logp_bigram(prev_r, r):
        tbl = bp.get(prev_r, {})
        pr = float(tbl.get(r, 1e-6))
        return math.log(max(pr, 1e-9))

    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            prev_r = b.rels[-1] if b.rels else None
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                base = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                # add small CKG prior
                base += lambda_prior * logp_edge(r)
                if prev_r is not None:
                    base += lambda_prior * logp_bigram(prev_r, r)
                nxt.append(BeamPath(base, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# ---------- helpers ----------
def to_device_graph(g):
    return {
        "x": g["x"].to(DEVICE),
        "edges": {r:e.to(DEVICE) for r,e in g["edges"].items()},
        "y": (g.get("y").to(DEVICE) if g.get("y") is not None else None),
        "source": g.get("source","default")
    }

def compute_CFAM(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _,nl,_,_ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        s = torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE)
    with torch.no_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p0 = torch.sigmoid(nl).mean().item()
    cf = x.clone()
    causal = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal:
        cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _,nl2,_,_ = model.forward_full(cf, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p1 = torch.sigmoid(nl2).mean().item()
    return (p0 - p1) ** 2

# ---------- CKG mining ----------
def mine_ckg(model, data_dir, base_relations, add_summary=True, limit=200, top_paths=3, width=24, hops=5, alpha=0.7):
    # stats
    edge_count = {r:0 for r in base_relations + (["DFG_THIN"] if add_summary else [])}
    bigram_count = {r:{q:0 for q in edge_count} for r in edge_count}
    hop_hist = {}
    used = 0

    rels = base_relations + (["DFG_THIN"] if add_summary else [])
    for i, g_cpu in enumerate(iter_graphs(data_dir, base_relations, add_summary)):
        if i >= limit: break
        g = to_device_graph(g_cpu)
        with torch.no_grad():
            sd, nl, h, es = model.forward_full(g["x"], g["edges"])
            seeds = pick_seeds(sd.detach(), g["edges"], SEED_K)
            p  = torch.sigmoid(nl)
            bs = run_beam_plain(p, es, g["edges"], seeds, width=width, max_hops=hops, alpha_node=alpha)
        if not bs: continue
        used += 1
        # accumulate counts from top K paths
        for bp in bs[:top_paths]:
            hop_hist[len(bp.nodes)-1] = hop_hist.get(len(bp.nodes)-1, 0) + 1
            # count relations along this path
            prev = None
            for r in bp.rels:
                if r not in edge_count: continue
                edge_count[r] += 1
                if prev is not None and prev in bigram_count and r in bigram_count[prev]:
                    bigram_count[prev][r] += 1
                prev = r

    # normalize priors
    total_edges = sum(edge_count.values()) or 1
    edge_prior_prob = {r: edge_count[r]/total_edges for r in edge_count}
    bigram_prob = {}
    for r in bigram_count:
        s = sum(bigram_count[r].values()) or 1
        bigram_prob[r] = {q: bigram_count[r][q]/s for q in bigram_count[r]}

    # simple motif mining: top tri-gram sequences by count
    motifs = []
    for r1 in bigram_count:
        for r2 in bigram_count[r1]:
            if bigram_count[r1][r2] == 0: continue
            # approximate 3rd step by chaining the most frequent next of r2
            r3 = max(bigram_count[r2], key=lambda q: bigram_count[r2][q]) if sum(bigram_count[r2].values())>0 else None
            if r3 is None: continue
            score = min(bigram_count[r1][r2], bigram_count[r2][r3])
            motifs.append((score, [r1,r2,r3]))
    motifs.sort(key=lambda x:x[0], reverse=True)
    motifs_topk = [{"count":int(c),"rels":rels} for c,rels in motifs[:20]]

    ckg = {
        "meta": {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "graphs_used": used,
            "mined_from": str(Path(data_dir).resolve())
        },
        "relations": list(edge_count.keys()),
        "edge_prior_count": edge_count,
        "edge_prior_prob": edge_prior_prob,
        "bigram_count": bigram_count,
        "bigram_prob": bigram_prob,
        "start_count": {},   # optional future use
        "end_count":   {},   # optional future use
        "hop_histogram": hop_hist,
        "motifs_topk": motifs_topk
    }
    return ckg

# ---------- Inference wrapper (plain vs CKG) ----------
def infer_one_with_ckg(model, g_cpu, ckg, width=24, hops=5, alpha=0.7, lambda_prior=0.15):
    g = to_device_graph(g_cpu)
    with torch.no_grad():
        sd, nl, h, es = model.forward_full(g["x"], g["edges"])
        seeds = pick_seeds(sd.detach(), g["edges"], SEED_K)
        p  = torch.sigmoid(nl)
        beams_plain = run_beam_plain(p, es, g["edges"], seeds, width=width, max_hops=hops, alpha_node=alpha)
        beams_ckg   = run_beam_with_ckg(p, es, g["edges"], seeds, ckg, width=width, max_hops=hops, alpha_node=alpha, lambda_prior=lambda_prior)
    return beams_plain, beams_ckg

# ===================== MAIN =====================
Path(CKG_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(DEMO_OUT).parent.mkdir(parents=True, exist_ok=True)

# 1) Load model & relations
assert os.path.exists(CKPT), f"Checkpoint not found: {CKPT}"
model, BASE_REL, ADD_SUMMARY = load_model(CKPT)
print(f"[model] hidden={model.hidden} layers={model.layers} relations={model.relations}")

# 2) Mine CKG (post-hoc) from your train split
ckg = mine_ckg(model, DATA_MINE, base_relations=BASE_REL, add_summary=ADD_SUMMARY,
               limit=200, top_paths=3, width=BEAM_WIDTH, hops=BEAM_MAX_HOPS, alpha=ALPHA_NODE)
Path(CKG_PATH).write_text(json.dumps(ckg, indent=2))
print(f"[CKG] saved to {CKG_PATH} | graphs_used={ckg['meta']['graphs_used']} | motifs={len(ckg['motifs_topk'])}")

# 3) Pick a demo graph (demo dir → valid → train)
g_demo = pick_demo_graph_or_fallback(
    DATA_DEMO, BASE_REL,
    r"Dataset/valid/hetero_ready_gcbert", r"Dataset/train/hetero_ready_gcbert"
)
print("[demo] graph:", g_demo.get("path","<in-memory>"))

# 4) Infer plain vs CKG-guided beams
beams_plain, beams_ckg = infer_one_with_ckg(
    model, g_demo, ckg,
    width=BEAM_WIDTH, hops=BEAM_MAX_HOPS, alpha=ALPHA_NODE, lambda_prior=LAMBDA_PRIOR
)

# 5) Quick metrics on the CKG-guided beams
cfam = compute_CFAM(model, g_demo, beams_ckg)
ccs  = compute_CCS(model, g_demo,  beams_ckg)

# 6) Save demo outputs
demo = {
    "meta": {
        "graph_path": g_demo.get("path"),
        "ckpt": str(Path(CKPT).resolve()),
        "ckg": str(Path(CKG_PATH).resolve())
    },
    "plain_beams": [{"score":bp.score, "nodes":bp.nodes, "rels":bp.rels} for bp in beams_plain],
    "ckg_beams":   [{"score":bp.score, "nodes":bp.nodes, "rels":bp.rels} for bp in beams_ckg],
    "metrics": {"CFAM": cfam, "CCS": ccs}
}
Path(DEMO_OUT).write_text(json.dumps(demo, indent=2))
print(f"[demo] saved to {DEMO_OUT}")
print(f"[metrics] CFAM={cfam:.4f} CCS={ccs:.4f}" if (cfam is not None and ccs is not None) else "[metrics] CFAM/CCS not available")


[env] torch=2.4.1+cu121 device=cuda
[model] hidden=64 layers=3 relations=['DFG', 'CFG', 'CALL', 'ARG2PARAM', 'RET2CALL', 'RET2LHS', 'DFG_THIN']


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_28696\981697541.py:339: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
C:\User

[CKG] saved to ckg/ckg.json | graphs_used=200 | motifs=20
[demo] graph: Dataset\valid\hetero_ready_gcbert\shard_0003.pt
[demo] saved to out/ckg_demo/demo.json
[metrics] CFAM=0.0337 CCS=0.0000


CKG Mining

In [14]:
# ===================== CKG re-mining + UTF-8 report + checkpoints + CKG-guided demo =====================
import os, json, math, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ---------- CONFIG ----------
CKPT       = r"out/cvul/1760989836/model.pt"

DATA_MINE  = r"Dataset/train/hetero_ready_gcbert"     # prefer JSON graphs w/ gcbert/gcb_x
DATA_DEMO  = r"Dataset/demo/hetero_ready_gcbert"      # optional; will fallback
FALLBACK_1 = r"Dataset/valid/hetero_ready_gcbert"
FALLBACK_2 = r"Dataset/train/hetero_ready_gcbert"

CKG_PATH       = r"ckg/ckg.json"                      # mined CKG
CKG_PATH_TMP   = r"ckg/ckg.json.tmp"                  # periodic checkpoint
CKG_RPT_JSON   = r"ckg/ckg_report.json"               # JSON report
CKG_RPT_MD     = r"ckg/ckg_report.md"                 # Markdown report (UTF-8)
DEMO_OUT       = r"out/ckg_demo/demo.json"            # demo beams + metrics

# Beam & prior (match training; slightly reduced for speed)
SEED_K        = 8
BEAM_WIDTH    = 16     # <= reduce vs 24 to speed up mining
BEAM_MAX_HOPS = 4      # <= reduce vs 5 to speed up mining
ALPHA_NODE    = 0.7
LAMBDA_PRIOR  = 0.15

# Mining budget
MINE_LIMIT    = None   # None = ALL files; or set an int cap
TOP_PATHS     = 2      # take top-2 beams per graph for speed

# Periodic checkpointing
CKPT_EVERY    = 200

# Optional demo causal checks (not used for mining)
DO_EVAL       = True

# ---------- device ----------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"
DEVICE = pick_device()
torch.set_grad_enabled(False)
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ---------- IO ----------
def safe_load(p):
    p = Path(p)
    if p.suffix.lower() == ".json":
        return json.loads(p.read_text(encoding="utf-8"))
    return torch.load(p, map_location="cpu")

def write_json_utf8(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def list_files(root_dir, patterns=("*.json","*.pt")):
    root = Path(root_dir)
    files=[]
    for pat in patterns:
        files.extend(sorted(root.glob(pat)))
    return files

# ---------- normalization helpers ----------
def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim == 1: t = t.view(-1, 1)
    return t

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0:
        return torch.zeros((2,0), dtype=torch.long)
    s,d = ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any():
        return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def _get_store_feat(store):
    cand = ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]
    for nm in cand:
        if hasattr(store, nm):
            val = getattr(store, nm)
            if val is not None:
                t = _first_2d_float(val)
                if t.numel() > 0:
                    return t
    if hasattr(store, "__dict__"):
        for nm in cand:
            if nm in store.__dict__ and store.__dict__[nm] is not None:
                t = _first_2d_float(store.__dict__[nm])
                if t.numel() > 0:
                    return t
    return None

def normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    node_stores = getattr(obj, "node_stores", None)
    edge_stores = getattr(obj, "edge_stores", None)
    if node_stores is None or edge_stores is None:
        return None

    candidates=[]
    for st in node_stores:
        key = getattr(st, "_key", None) or getattr(st, "type", None) or "code"
        feat = _get_store_feat(st)
        if feat is not None and feat.numel() > 0:
            candidates.append((key, feat))
    if not candidates:
        return None

    nt, x = max(candidates, key=lambda kv: kv[1].size(0))
    N = x.size(0)

    E = {r: torch.zeros((2,0), dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei = getattr(es, "edge_index", None)
        if ei is None: continue
        src_t = getattr(es, "src_type", None)
        dst_t = getattr(es, "dst_type", None)
        if (src_t is not None and dst_t is not None) and not (src_t == nt and dst_t == nt):
            continue
        rel = getattr(es, "edge_type", None) or getattr(es, "_key", None)
        if isinstance(rel, (tuple, list)) and len(rel) == 3:
            rel = str(rel[1])
        rel = str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E:
            E[rel] = sanitize_edges(N, to_long_2(ei))

    if ADD_SUMMARY_EDGES:
        base = E.get("DFG", torch.zeros((2,0), dtype=torch.long))
        for r in ("ARG2PARAM", "RET2CALL", "RET2LHS"):
            if r in E and E[r].numel() > 0:
                base = torch.cat([base, E[r]], dim=1)
        E["DFG_THIN"] = base

    return {"x": x, "edges": E, "y": None, "source": "default"}

def coerce_x_any(obj):
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj and obj[k] is not None:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES:
            base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in RELATIONS:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if isinstance(obj, dict) and k in obj:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj = obj["graph"]
    if "torch_geometric" in str(type(obj)) and not isinstance(obj, dict):
        g_het = normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g_het is not None:
            return g_het
        if hasattr(obj, "x") and obj.x is not None:
            x = _first_2d_float(obj.x); N=x.size(0)
            E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
            if hasattr(obj,"edge_index"):
                E["DFG"]=sanitize_edges(N, to_long_2(obj.edge_index))
            if ADD_SUMMARY_EDGES:
                E["DFG_THIN"]=E["DFG"]
            return {"x":x,"edges":E,"y":None,"source":"default"}
        return None
    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is None or x.numel()==0:
            return None
        N = x.size(0)
        E = build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES)
        return {"x":x, "edges":E, "y":None, "source":"default"}
    return None

def iter_graphs_with_progress(root_dir, RELATIONS, ADD_SUMMARY_EDGES=True, limit=None, patterns=("*.json","*.pt")):
    files = list_files(root_dir, patterns)
    if limit is None: limit = len(files)
    usable = 0
    for fp in tqdm(files[:limit], total=min(limit, len(files)), desc=f"[mine] {root_dir}"):
        try:
            obj = safe_load(fp)
            g = normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
            if g is None or g["x"] is None or g["x"].numel()==0:
                continue
            g["path"] = str(fp)
            usable += 1
            yield g
        except Exception:
            continue
    if usable == 0:
        print(f"[warn] 0 usable graphs from {root_dir}")

# ---------- model ----------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations = relations
        self.lin_rel = nn.ModuleDict({r: nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self= nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H = h
        for r in self.relations:
            ei = E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d = ei
            msg = self.lin_rel[r](H)
            agg = torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H = H + agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None):
        super().__init__()
        if relations is None:
            relations = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"]
        self.relations = list(relations)
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden    = hidden
        self.layers    = layers

    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]

    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h

    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out

    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path:str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"])
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    base_relations = [r for r in relations if r != "DFG_THIN"]
    add_summary = ("DFG_THIN" in relations)
    return model, base_relations, add_summary

# ---------- beam search ----------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc, E, N:int):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

def run_beam_plain(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                sc = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

def run_beam_with_ckg(p, edge_sc, E, seeds, ckg, width=24, max_hops=5, alpha_node=0.7, lambda_prior=0.15):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)
    ep = ckg.get("edge_prior_prob", {})   # P(r)
    bp = ckg.get("bigram_prob", {})       # P(r_t | r_{t-1})
    def logp_edge(r):       return math.log(max(float(ep.get(r, 1e-6)), 1e-9))
    def logp_bigram(a, b):  return math.log(max(float(bp.get(a, {}).get(b, 1e-6)), 1e-9))
    def clog(x):            return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            prev_r = b.rels[-1] if b.rels else None
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                base = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                base += lambda_prior * logp_edge(r)
                if prev_r is not None:
                    base += lambda_prior * logp_bigram(prev_r, r)
                nxt.append(BeamPath(base, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# ---------- helpers ----------
def to_device_graph(g):
    return {"x": g["x"].to(DEVICE),
            "edges": {r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": None, "source": g.get("source","default")}

def pick_demo_graph_or_fallback(demo_dir, relations, add_summary=True):
    for root in [demo_dir, FALLBACK_1, FALLBACK_2]:
        files = list_files(root, ("*.json","*.pt"))
        for g in iter_graphs_with_progress(root, relations, add_summary, limit=len(files)):
            return g
    raise RuntimeError("No usable graph found for demo.")

# ---------- optional demo causal checks ----------
def compute_CFAM(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _, nl, _, _ = model.forward_full(x, {r: e.to(DEVICE) for r, e in g_cpu["edges"].items()})
        s = torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE)
    with torch.no_grad():
        _, nl, _, _ = model.forward_full(x, {r: e.to(DEVICE) for r, e in g_cpu["edges"].items()})
        p0 = torch.sigmoid(nl).mean().item()
    cf = x.clone()
    causal = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal:
        cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _, nl2, _, _ = model.forward_full(cf, {r: e.to(DEVICE) for r, e in g_cpu["edges"].items()})
        p1 = torch.sigmoid(nl2).mean().item()
    return (p0 - p1) ** 2

# ---------- CKG assembly helper ----------
def _build_ckg_obj(data_dir, rels, edge_count, bigram_count, start_count, end_count, hop_hist, trigram_count, used):
    total_edges = sum(edge_count.values()) or 1
    edge_prior_prob = {r: edge_count[r]/total_edges for r in edge_count}
    bigram_prob = {}
    for r in bigram_count:
        s = sum(bigram_count[r].values()) or 1
        bigram_prob[r] = {q: bigram_count[r][q]/s for q in bigram_count[r]}

    motifs_topk = []
    if trigram_count:
        for (a,b,c), cnt in trigram_count.most_common(20):
            motifs_topk.append({"count": int(cnt), "rels": [a,b,c]})
    if not motifs_topk:
        flat = [((a,b), cnt) for a,row in bigram_count.items() for b,cnt in row.items() if cnt>0]
        flat.sort(key=lambda x:x[1], reverse=True)
        for (a,b), cnt_ab in flat[:20]:
            row_b = bigram_count.get(b, {})
            if row_b:
                c = max(row_b, key=lambda q: row_b[q])
                motifs_topk.append({"count": int(min(cnt_ab, row_b[c])), "rels": [a,b,c]})
    if not motifs_topk:
        top_rels = [r for r,_ in sorted(edge_count.items(), key=lambda kv: kv[1], reverse=True)[:3]]
        if len(top_rels)>=3: motifs_topk=[{"count":1,"rels":top_rels[:3]}]
        elif len(top_rels)==2: motifs_topk=[{"count":1,"rels":[top_rels[0],top_rels[1],top_rels[1]]}]
        elif len(top_rels)==1: motifs_topk=[{"count":1,"rels":[top_rels[0]]*3}]

    return {
        "meta": {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "graphs_used": used,
            "mined_from": str(Path(data_dir).resolve())
        },
        "relations": list(rels),
        "edge_prior_count": edge_count,
        "edge_prior_prob": edge_prior_prob,
        "bigram_count": bigram_count,
        "bigram_prob": bigram_prob,
        "start_count": start_count,
        "end_count": end_count,
        "hop_histogram": dict(hop_hist),
        "motifs_topk": motifs_topk
    }

# ---------- CKG mining (with checkpoints, non-empty motifs) ----------
def mine_ckg(model, data_dir, base_relations, add_summary=True,
             limit=None, top_paths=2, width=16, hops=4, alpha=0.7):
    rels = list(base_relations) + (["DFG_THIN"] if add_summary and "DFG_THIN" not in base_relations else [])
    edge_count   = {r:0 for r in rels}
    bigram_count = {r:{q:0 for q in rels} for r in rels}
    start_count  = {r:0 for r in rels}
    end_count    = {r:0 for r in rels}
    hop_hist     = Counter()
    trigram_count= Counter()
    used = 0

    files = list_files(data_dir, ("*.json","*.pt"))
    if limit is None: limit = len(files)
    pbar  = tqdm(files[:limit], total=min(limit, len(files)), desc=f"[mine] {data_dir} → CKG")

    with torch.inference_mode():
        for fp in pbar:
            try:
                obj = safe_load(fp)
                g_cpu = normalize_graph(obj, base_relations, add_summary)
                if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0:
                    continue
                g = to_device_graph(g_cpu)
                sd, nl, h, es = model.forward_full(g["x"], g["edges"])
                seeds = pick_seeds(sd, g["edges"], SEED_K)
                p  = torch.sigmoid(nl)
                beams = run_beam_plain(p, es, g["edges"], seeds, width=width, max_hops=hops, alpha_node=alpha)
                if not beams:
                    continue
                used += 1
                for bp in beams[:top_paths]:
                    hop_hist[len(bp.nodes)-1] += 1
                    if bp.rels:
                        start_count[bp.rels[0]] += 1
                        end_count  [bp.rels[-1]]+= 1
                    prev = None
                    for j, r in enumerate(bp.rels):
                        edge_count[r] += 1
                        if prev is not None:
                            bigram_count[prev][r] += 1
                        if j >= 2:
                            tr = (bp.rels[j-2], bp.rels[j-1], bp.rels[j])
                            trigram_count[tr] += 1
                        prev = r
                # periodic checkpoint
                if used and (used % CKPT_EVERY == 0):
                    ckg_partial = _build_ckg_obj(data_dir, rels, edge_count, bigram_count,
                                                 start_count, end_count, hop_hist, trigram_count, used)
                    write_json_utf8(CKG_PATH_TMP, ckg_partial)
                    pbar.set_postfix_str(f"ckpt@{used}")
            except Exception:
                continue

    return _build_ckg_obj(data_dir, rels, edge_count, bigram_count,
                          start_count, end_count, hop_hist, trigram_count, used)

# ---------- Reporting (UTF-8; ASCII arrows fallback) ----------
def save_ckg_reports(ckg:Dict, json_path:str, md_path:str):
    write_json_utf8(json_path, ckg)

    rels = ckg["relations"]
    ec   = ckg["edge_prior_count"]
    ep   = ckg["edge_prior_prob"]
    sc   = ckg.get("start_count", {})
    ec_end = ckg.get("end_count", {})
    hop = ckg.get("hop_histogram", {})
    motifs = ckg.get("motifs_topk", [])

    lines=[]
    lines.append(f"# CKG Mining Report\n")
    lines.append(f"- **Mined from:** `{ckg['meta']['mined_from']}`")
    lines.append(f"- **Graphs used:** {ckg['meta']['graphs_used']}")
    lines.append(f"- **Created at:** {ckg['meta']['created_at']}\n")

    # Edge priors
    lines.append("## Edge Priors (counts & probabilities)\n")
    lines.append("| Relation | Count | Prob |")
    lines.append("|---|---:|---:|")
    for r in sorted(rels, key=lambda r: ec.get(r,0), reverse=True):
        lines.append(f"| {r} | {ec.get(r,0)} | {ep.get(r,0.0):.4f} |")
    lines.append("")

    # Start/End stats
    if sc and ec_end:
        lines.append("## Start/End Relation Frequencies\n")
        lines.append("| Relation | Start Count | End Count |")
        lines.append("|---|---:|---:|")
        for r in rels:
            lines.append(f"| {r} | {sc.get(r,0)} | {ec_end.get(r,0)} |")
        lines.append("")

    # Hop histogram
    if hop:
        lines.append("## Hop Histogram (path length in edges)\n")
        lines.append("| Hops | Count |")
        lines.append("|---:|---:|")
        for h in sorted(hop, key=lambda x: int(x) if isinstance(x,str) else x):
            v = hop[h] if not isinstance(h, str) else hop[h]
            lines.append(f"| {h} | {v} |")
        lines.append("")

    # Motifs
    lines.append("## Top Motifs\n")
    if motifs:
        lines.append("| # | Count | Relations (tri-gram) |")
        lines.append("|---:|---:|---|")
        for i,m in enumerate(motifs[:20], 1):
            lines.append(f"| {i} | {m['count']} | {' -> '.join(m['rels'])} |")
    else:
        lines.append("_No motifs mined._")
    lines.append("")

    Path(md_path).parent.mkdir(parents=True, exist_ok=True)
    Path(md_path).write_text("\n".join(lines), encoding="utf-8")

# ---------- MAIN ----------
for p in [CKG_PATH, CKG_PATH_TMP, CKG_RPT_JSON, CKG_RPT_MD, DEMO_OUT]:
    Path(p).parent.mkdir(parents=True, exist_ok=True)

# Load model
assert os.path.exists(CKPT), f"Checkpoint not found: {CKPT}"
model, BASE_REL, ADD_SUMMARY = load_model(CKPT)
print(f"[model] hidden={model.hidden} layers={model.layers} relations={model.relations}")

# Mine CKG (all graphs by default) + report
ckg = mine_ckg(model, DATA_MINE, BASE_REL, add_summary=ADD_SUMMARY,
               limit=MINE_LIMIT, top_paths=TOP_PATHS,
               width=BEAM_WIDTH, hops=BEAM_MAX_HOPS, alpha=ALPHA_NODE)
write_json_utf8(CKG_PATH, ckg)
save_ckg_reports(ckg, CKG_RPT_JSON, CKG_RPT_MD)
print(f"[CKG] saved: {CKG_PATH}")
print(f"[REPORT] saved: {CKG_RPT_JSON}, {CKG_RPT_MD}")
print(f"  graphs_used = {ckg['meta']['graphs_used']}")
print(f"  motifs_topk = {len(ckg['motifs_topk'])} (should be > 0)\n")

# Demo graph (demo → valid → train)
g_demo = pick_demo_graph_or_fallback(DATA_DEMO, BASE_REL, ADD_SUMMARY)
print("[demo] graph:", g_demo.get("path","<in-memory>"))

# Inference: plain vs CKG-guided
with torch.inference_mode():
    g = {"x": g_demo["x"].to(DEVICE), "edges": {r:e.to(DEVICE) for r,e in g_demo["edges"].items()}}
    sd, nl, h, es = model.forward_full(g["x"], g["edges"])
    seeds = pick_seeds(sd, g["edges"], SEED_K)
    p  = torch.sigmoid(nl)
beams_plain = run_beam_plain(p, es, g["edges"], seeds, width=BEAM_WIDTH, max_hops=BEAM_MAX_HOPS, alpha_node=ALPHA_NODE)
beams_ckg   = run_beam_with_ckg(p, es, g["edges"], seeds, ckg, width=BEAM_WIDTH, max_hops=BEAM_MAX_HOPS,
                                alpha_node=ALPHA_NODE, lambda_prior=LAMBDA_PRIOR)

# Optional demo metrics (for sanity only; not part of mining)
metrics = {}
if DO_EVAL:
    cfam = compute_CFAM(model, g_demo, beams_ckg)
    ccs  = compute_CCS(model, g_demo,  beams_ckg)
    metrics = {"CFAM": cfam, "CCS": ccs}

# Save demo outputs
demo = {
    "meta": {
        "graph_path": g_demo.get("path"),
        "ckpt": str(Path(CKPT).resolve()),
        "ckg":  str(Path(CKG_PATH).resolve())
    },
    "plain_beams": [{"score":bp.score, "nodes":bp.nodes, "rels":bp.rels} for bp in beams_plain],
    "ckg_beams":   [{"score":bp.score, "nodes":bp.nodes, "rels":bp.rels} for bp in beams_ckg],
    "metrics": metrics
}
write_json_utf8(DEMO_OUT, demo)
print(f"[demo] saved: {DEMO_OUT}")
print("[done]")


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29944\3919640640.py:311: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


[env] torch=2.4.1+cu121 device=cuda
[model] hidden=64 layers=3 relations=['DFG', 'CFG', 'CALL', 'ARG2PARAM', 'RET2CALL', 'RET2LHS', 'DFG_THIN']


[mine] Dataset/train/hetero_ready_gcbert → CKG:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29944\3919640640.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related 

[CKG] saved: ckg/ckg.json
[REPORT] saved: ckg/ckg_report.json, ckg/ckg_report.md
  graphs_used = 3438
  motifs_topk = 20 (should be > 0)



[mine] Dataset/demo/hetero_ready_gcbert: 0it [00:00, ?it/s]


[warn] 0 usable graphs from Dataset/demo/hetero_ready_gcbert


[mine] Dataset/valid/hetero_ready_gcbert:   0%|          | 0/2906 [00:00<?, ?it/s]


[demo] graph: Dataset\valid\hetero_ready_gcbert\shard_0003.pt


RuntimeError: Inference tensors cannot be saved for backward. To work around you can make a clone to get a normal tensor and use it in autograd.